# Anxiety, Model-Based Revaluation & Overthinking — V4 (Merged Analysis)

*Merged from `Modelling_AllChoices_V4.ipynb` + `RT_Analyses.ipynb` (2026-06-11). Restructured **poster-first**: model-agnostic headline analyses up front, heavy modeling + validations after.*

**Structure**

1. **Prep for main analyses** — imports, load, derived data, the canonical **159/277** model-agnostic MB classifier, RT/OT prep, and a lean per-subject `subj` table (no CBM).
2. **Main / Poster analyses (model-agnostic)** — overthinking, **H1** (somatic×OT), **H2** (worry×OT), **H3** (log-delib RT), rank-moderation interaction, main correlations, somatic–worry dissociation. All partition on the **159** classifier; no CBM needed.
3. **Modeling (heavy) + CBM confirmation** — CBM/HBI, BMS/PXP, PPC, group collapse, then **H1/H2 re-run under the CBM classification** to confirm the agnostic findings, κ convergent validity, recovery, Bayesian RT-LMM, DDMs.
4. **Validations & Sensitivity (heavy)** — sample descriptives, paradigm-validation LMMs, questionnaire structure, assumption checks, power, partials, robustness, OT-half decompositions.

---

### ⚠ Notes
**Two dataframes:** Modelling uses `df`/`trial_df`; the RT pipeline is namespaced to `rt_df`/`rt_trial`.

**Poster needs no CBM.** §2 H1/H2/H3 partition on the model-agnostic **159** split. The CBM-group overlay panels stay off in §2 (a gate checks whether CBM labels exist) and turn on in §3 when the same cells re-run after `subj` is rebuilt with CBM labels — that re-run is the confirmation step.

**Canonical MB split = 159/277.** Some preserved §3–§4 outputs predate the 2026-06-11 reconciliation (older Ns); re-run to refresh.

---

# Section 1 · Prep for Main Analyses

Only what the model-agnostic poster results need before they can run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import linregress
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11

# ══════════════════════════════════════════════════════════════════════
# LOAD EXPORT  ── single source of truth for the input dataset.
# To switch datasets (e.g. the ~400-participant export), change
# DATA_FILE and DATASET_TAG below; nothing else needs editing.
# Older test-run exports live in prereg_3_testdrives/.
# ══════════════════════════════════════════════════════════════════════
DATA_FILE   = 'experiment_results_v4_1_export.csv'  # input CSV
DATASET_TAG = 'v4_1'                                # used in OUT_PREFIX (cell 2)

df = pd.read_csv(DATA_FILE)
df['subject_id'] = df['prolific_id'].fillna(df['firestore_uid'])
df = df[df['status'].str.upper() == 'COMPLETED'].reset_index(drop=True)
N_LOADED = len(df)
print(f'Loaded N = {N_LOADED} COMPLETED subjects from {DATA_FILE}  (DATASET_TAG={DATASET_TAG!r})')

# ══════════════════════════════════════════════════════════════════════
# SUBJECT FILTERING (folded in from V3 Generate_All_Files Steps 2-3)
# ══════════════════════════════════════════════════════════════════════
# Two filters, applied directly here so this notebook is the single
# source of truth for the filtering logic. After applying both filters,
# this cell ALSO writes filtered_data_<DATASET_TAG>.csv as a derived
# artifact for the Paul/RT_Analyses.ipynb companion notebook (see the
# to_csv call at the end of this cell). Paul never re-derives the
# exclusions; it just consumes what we write here.
#
# 1. Zig-zag inconsistency (P(chose fixed) curve should be monotonic
#    in offer value within each query x state cell). Sum the "drops"
#    across the 4 cells; exclude if total >= ZIGZAG_THRESHOLD.
# 2. Knowledge_Check_Accuracy > KNOWLEDGE_CHECK_MIN.
#
# NOT applying the V3 dropna() step: in V4 some columns are
# expected-NaN (e.g. Teleprompter_Silence_Triggered is NaN for
# participants who never triggered the silence warning) and
# total_time_minutes is unpopulated in the V4 export. A blanket
# dropna() would remove every subject. Per-column NaN handling
# happens downstream where it matters.
# ══════════════════════════════════════════════════════════════════════

ZIGZAG_THRESHOLD = 2.0
KNOWLEDGE_CHECK_MIN = 0.25

# Build long-format choice table for the zig-zag check
_long_rows = []
for _, row in df.iterrows():
    sid = row['subject_id']
    for q in [1, 2]:
        for t in range(1, 21):
            state = row.get(f'Query_{q}_T{t}_State')
            offer = row.get(f'Query_{q}_T{t}_FixedOffer')
            choice = row.get(f'Query_{q}_T{t}_Choice')
            if pd.isna(state) or pd.isna(offer) or pd.isna(choice):
                continue
            _long_rows.append({
                'subject_id': sid, 'Query': q, 'State': state,
                'FixedOffer': offer, 'ChoseFixed': 1 if choice == 0 else 0,
            })
_long = pd.DataFrame(_long_rows)

def _zigzag_score(sub_df):
    total = 0.0
    for _, g in sub_df.groupby(['Query', 'State']):
        curve = g.groupby('FixedOffer')['ChoseFixed'].mean().sort_index().values
        for i in range(len(curve) - 1):
            if curve[i] > curve[i+1]:
                total += curve[i] - curve[i+1]
    return total

zz = _long.groupby('subject_id', group_keys=False).apply(_zigzag_score)
zz_excluded = set(zz[zz >= ZIGZAG_THRESHOLD].index)

# Knowledge check filter
kc_excluded = set(df.loc[df['Knowledge_Check_Accuracy'] <= KNOWLEDGE_CHECK_MIN, 'subject_id'])

# Combine
all_excluded = zz_excluded | kc_excluded
filtering_log = pd.DataFrame({
    'subject_id': df['subject_id'],
    'zigzag_score': df['subject_id'].map(zz),
    'zigzag_excluded': df['subject_id'].isin(zz_excluded),
    'knowledge_check': df['Knowledge_Check_Accuracy'],
    'kc_excluded': df['subject_id'].isin(kc_excluded),
    'excluded': df['subject_id'].isin(all_excluded),
})

df = df[~df['subject_id'].isin(all_excluded)].reset_index(drop=True)
N_SUBJ = len(df)
print(f'Excluded zig-zag (>= {ZIGZAG_THRESHOLD}): {len(zz_excluded)}')
print(f'Excluded knowledge check (<= {KNOWLEDGE_CHECK_MIN}): {len(kc_excluded)}')
print(f'Final N after filtering: {N_SUBJ} (kept {N_SUBJ}/{N_LOADED})')

# ═════════════════════════════════════════════════════════════════════
# PREREG ALIGNMENT (2026-05-27): write the post-exclusion filtered df
# to CSV so the Paul/RT_Analyses.ipynb companion notebook can consume
# it without duplicating the exclusion logic. V4 remains the single
# source of truth for filtering; the CSV is a derived artifact.
# ═════════════════════════════════════════════════════════════════════
_filtered_csv = f'filtered_data_{DATASET_TAG}.csv'
df.to_csv(_filtered_csv, index=False)
print(f"Wrote {_filtered_csv} ({len(df)} rows) for RT_Analyses companion notebook.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIG (V4)
# ══════════════════════════════════════════════════════════════════════
#   FIT_ALL_TRIALS = True   → use every query trial
#   FIT_ALL_TRIALS = False  → use only anti-MF trials (the MB-vs-MF
#                             discriminative subset — for V4 this is
#                             roughly 18/24 trials by design)
#
# Output paths get a <DATASET_TAG>_<suffix> prefix (e.g. v4_1_all) so
# outputs from different exports don't collide. Old v4 test-run outputs
# are archived under prereg_3_testdrives/ and cbm_output_v4_all/.
# ══════════════════════════════════════════════════════════════════════
FIT_ALL_TRIALS = True

SEED = 42
np.random.seed(SEED)

FILTER_SUFFIX = "all" if FIT_ALL_TRIALS else "antimf"
FILTER_LABEL  = "ALL trials" if FIT_ALL_TRIALS else "ANTI-MF trials only"
OUT_PREFIX    = f"{DATASET_TAG}_{FILTER_SUFFIX}"   # e.g. v4_1_all
print(f"[CONFIG] FIT_ALL_TRIALS={FIT_ALL_TRIALS}  ({FILTER_LABEL})")
print(f"[CONFIG] SEED={SEED}")
print(f"[CONFIG] FILTER_SUFFIX='{FILTER_SUFFIX}'  OUT_PREFIX='{OUT_PREFIX}'")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# COMPUTE DERIVED VARIABLES & BUILD TRIAL-LEVEL DATA (V4)
# ══════════════════════════════════════════════════════════════════════
# Differences from V3 pipeline:
#   - 12 trials per query (not 8): range(1, 13)
#   - EV_MAP[(2,'Feather')] = -10 (was 0 in V3, because afterReval2 T1
#     changed from -20 to -60). All other EV_MAP entries unchanged.
#   - id_vars now carries V4 threat-induction columns so downstream
#     analyses can use Final_Earned_Speaking_Time_Sec etc.

# ── Survey subscales ──
erq_reappraisal_items = ['Survey_ERQ_Q1_Raw','Survey_ERQ_Q3_Raw','Survey_ERQ_Q5_Raw',
                         'Survey_ERQ_Q7_Raw','Survey_ERQ_Q8_Raw','Survey_ERQ_Q10_Raw']
erq_suppression_items = ['Survey_ERQ_Q2_Raw','Survey_ERQ_Q4_Raw',
                         'Survey_ERQ_Q6_Raw','Survey_ERQ_Q9_Raw']

df = df.copy()
df['ERQ_Reappraisal'] = df[erq_reappraisal_items].sum(axis=1)
df['ERQ_Suppression']  = df[erq_suppression_items].sum(axis=1)
df['somatic_anxiety']  = df['Survey_ANXIETY_2_Total_Score']
df['worry']            = df['Survey_ANXIETY_1_Total_Score']

# ── Ratings ──
for stim in ['Feather', 'Triangle']:
    for t in [1, 2, 3]:
        df[f'{stim}_R{t}'] = df[f'Rating_{t}_{stim}_Val']

# ── EV map (V4: only (2,Feather) differs from V3) ──
EV_MAP = {
    (1, 'Feather'): -15, (1, 'Triangle'): 15,
    (2, 'Feather'): -10, (2, 'Triangle'): 10,
}
MF_MAP = {'Feather': 5, 'Triangle': -5}

# ── Build trial-level dataframe ──
N_QUERY_TRIALS = 12   # V4: 6 Feather + 6 Triangle offers per query
rows = []
for _, subj in df.iterrows():
    sid = subj['subject_id']
    for q in [1, 2]:
        for t in range(1, N_QUERY_TRIALS + 1):
            pfx = f'Query_{q}_T{t}'
            state  = subj.get(f'{pfx}_State')
            choice = subj.get(f'{pfx}_Choice')
            offer  = subj.get(f'{pfx}_FixedOffer')
            rt     = subj.get(f'{pfx}_RT')
            if pd.isna(state) or pd.isna(choice) or pd.isna(offer):
                continue
            ev = EV_MAP.get((q, state), np.nan)
            mf = MF_MAP[state]
            ev_diff = ev - offer
            abs_ev_diff = np.abs(ev_diff)
            optimal = 1 if ev > offer else (0 if ev < offer else 1)
            correct = int(choice == optimal) if not np.isnan(optimal) else np.nan

            # PREREG: correct_canon scoring ∈ {0, 0.5, 1}
            #   1.0 if choice matches sign(EV − offer); 0.0 if wrong sign;
            #   0.5 if EV == offer (value-neutral, neither right nor wrong).
            if ev > offer:
                correct_canon = 1.0 if choice == 1 else 0.0
            elif ev < offer:
                correct_canon = 1.0 if choice == 0 else 0.0
            else:
                correct_canon = 0.5

            mf_says_gamble = 1 if mf > offer else (0 if mf < offer else np.nan)
            mb_says_gamble = 1 if ev > offer else (0 if ev < offer else np.nan)

            if pd.notna(mb_says_gamble):
                if pd.notna(mf_says_gamble) and mf_says_gamble != mb_says_gamble:
                    is_anti_mf = 1; chose_mb = int(choice == mb_says_gamble)
                elif pd.isna(mf_says_gamble):
                    is_anti_mf = 1; chose_mb = int(choice == mb_says_gamble)
                else:
                    is_anti_mf = 0; chose_mb = np.nan
            else:
                is_anti_mf = np.nan; chose_mb = np.nan

            log_rt = np.log(rt) if pd.notna(rt) and rt > 0 else np.nan

            rows.append({
                'subject_id': sid, 'query_num': q, 'trial': t,
                'trial_total': t + (N_QUERY_TRIALS if q == 2 else 0),
                'state': state, 'is_triangle': 1 if state == 'Triangle' else 0,
                'choice': int(choice), 'offer': offer, 'ev': ev, 'mf_value': mf,
                'ev_diff': ev_diff, 'abs_ev_diff': abs_ev_diff, 'correct': correct,
                'is_anti_mf': is_anti_mf, 'chose_mb': chose_mb,
                'correct_canon': correct_canon,
                'rt': rt, 'log_rt': log_rt,
                'somatic_anxiety': subj['somatic_anxiety'],
                'erq_reappraisal': subj['ERQ_Reappraisal'],
                'erq_suppression': subj['ERQ_Suppression'],
            })

trial_df = pd.DataFrame(rows)
trial_df = trial_df.dropna(subset=['correct'])

trial_df['within_query_ev_diff_z'] = trial_df.groupby(['query_num', 'state'])['ev_diff'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0)

anti_mf_df = trial_df[trial_df['is_anti_mf'] == 1].copy()

# ── Subject-level id_vars (now carries V4 threat-induction columns) ──
id_vars = df[['subject_id', 'somatic_anxiety', 'worry',
              'ERQ_Reappraisal', 'ERQ_Suppression',
              'Final_Earned_Speaking_Time_Sec', 'Teleprompter_Attempts',
              'Teleprompter_Silence_Triggered', 'Threat_Induction_RT']].copy()
id_vars = id_vars.rename(columns={'Final_Earned_Speaking_Time_Sec': 'earned_time'})

print(f'Trial-level data: {len(trial_df)} obs from {trial_df.subject_id.nunique()} subjects')
print(f'Anti-MF trials:   {trial_df.is_anti_mf.sum():.0f} / {trial_df.is_anti_mf.notna().sum()}')
print(f'RT available:     {trial_df.log_rt.notna().sum()} trials with valid log(RT)')
print(f'N subjects:       {trial_df.subject_id.nunique()}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MODEL-AGNOSTIC mb_score CLASSIFIER  (preregistered primary partition)
# ══════════════════════════════════════════════════════════════════════
# Per the prereg (Variables / Model-agnostic model-based classification):
#
#   mb_score_s  =  P(chose MB | anti-MF query trial in stim s)
#
# Subject is MB-classified if  mb_score_F > 0.5  OR  mb_score_T > 0.5
# (lenient, either-stim, above-chance rule).
#
# This cell builds `mb_score_df`, a per-subject DataFrame with columns
# {mb_score_F, mb_score_T, n_anti_F, n_anti_T, mb_classif_agnostic}.
# H1, H2, H3 confirmatory tests partition on `mb_classif_agnostic`.
# The CBM/HBI classification (built later in section 8) is used as the
# convergent-validity check, not the primary partition.
# ══════════════════════════════════════════════════════════════════════

# Restrict to anti-MF trials with a valid chose_mb
_anti = trial_df.dropna(subset=['chose_mb'])
_anti = _anti[_anti['is_anti_mf'] == 1].copy()

# Per-subject × per-stim mean of chose_mb
_mb = (_anti
       .groupby(['subject_id', 'state'])['chose_mb']
       .agg(['mean', 'count'])
       .rename(columns={'mean': 'mb_score', 'count': 'n_anti'})
       .reset_index())

# Pivot to one row per subject with F and T columns
_pivot = _mb.pivot(index='subject_id', columns='state',
                   values=['mb_score', 'n_anti'])
_pivot.columns = [f'{a}_{b[0]}' for a, b in _pivot.columns]   # e.g. mb_score_F
_pivot = _pivot.reset_index()

# Make sure both columns exist even when a stim has no anti-MF data
for col in ['mb_score_F', 'mb_score_T', 'n_anti_F', 'n_anti_T']:
    if col not in _pivot.columns:
        _pivot[col] = np.nan

# Apply the lenient OR-rule: MB if either stim above chance
_pivot['mb_classif_agnostic'] = (
    (_pivot['mb_score_F'] > 0.5) | (_pivot['mb_score_T'] > 0.5)
)

mb_score_df = _pivot[['subject_id',
                      'mb_score_F', 'mb_score_T',
                      'n_anti_F', 'n_anti_T',
                      'mb_classif_agnostic']].copy()

# Summary print
_n_mb  = int(mb_score_df['mb_classif_agnostic'].sum())
_n_tot = len(mb_score_df)
_pct   = 100 * _n_mb / _n_tot if _n_tot else float('nan')
print(f'Model-agnostic classifier: {_n_mb}/{_n_tot} subjects MB ({_pct:.1f}%)')
print(f'  mb_score_F: mean={mb_score_df["mb_score_F"].mean():.3f}, '
      f'median={mb_score_df["mb_score_F"].median():.3f}, '
      f'n with data = {mb_score_df["mb_score_F"].notna().sum()}')
print(f'  mb_score_T: mean={mb_score_df["mb_score_T"].mean():.3f}, '
      f'median={mb_score_df["mb_score_T"].median():.3f}, '
      f'n with data = {mb_score_df["mb_score_T"].notna().sum()}')
print(f'  Subjects classified MB on Feather only:  '
      f'{int(((mb_score_df["mb_score_F"]>0.5) & ~(mb_score_df["mb_score_T"]>0.5)).sum())}')
print(f'  Subjects classified MB on Triangle only: '
      f'{int((~(mb_score_df["mb_score_F"]>0.5) & (mb_score_df["mb_score_T"]>0.5)).sum())}')
print(f'  Subjects classified MB on both stims:    '
      f'{int(((mb_score_df["mb_score_F"]>0.5) & (mb_score_df["mb_score_T"]>0.5)).sum())}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 0.  IMPORTS & DATA LOAD
# ══════════════════════════════════════════════════════════════════════
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr, ttest_ind, mannwhitneyu, wilcoxon
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 130

# Fixed task constants (canonical EV per (query, state) and MF prior values)
# PATCH 2026-05-27 (v4_1): (2,'Feather') was 0 in V3, is -10 in V4
# because afterReval2 T1 changed from -20 to -60 (see V4 preregistration).
EV_MAP = {
    (1, 'Feather'): -15, (1, 'Triangle'): 15,
    (2, 'Feather'): -10, (2, 'Triangle'): 10,
}
MF_MAP = {'Feather': 5, 'Triangle': -5}

# PATCH 2026-05-27 (v4_1): read the post-exclusion CSV written by
# Modelling_AllChoices_V4.ipynb cell 1 (folds in COMPLETED + zig-zag
# + Knowledge_Check exclusions). The V3 'filtered_data_v4_1.csv' is not
# regenerated by the V4 pipeline.
# Resilient path resolution: V4 writes the CSV with a relative path,
# so it lands at V4's cwd (project root). If Paul's kernel was started
# from this Paul/ subdir, the file is one level up. Search both.
def _find_filtered_csv(name='filtered_data_v4_1.csv'):
    here   = os.getcwd()
    parent = os.path.dirname(here)
    candidates = [name, os.path.join('..', name), os.path.join(parent, name)]
    for c in candidates:
        if os.path.isfile(c):
            return c
    raise FileNotFoundError(
        f"Could not find {name!r}. Searched:\n  " + "\n  ".join(candidates) +
        f"\nRun cell 1 of Modelling_AllChoices_V4.ipynb first "
        f"(it writes this file after the exclusion filters). "
        f"Current cwd: {here!r}.")

_FILTERED_CSV = _find_filtered_csv('filtered_data_v4_1.csv')
rt_df = pd.read_csv(_FILTERED_CSV)
rt_df['subject_id'] = rt_df['prolific_id'].astype(str)
N_SUBJ = len(rt_df)
print(f"N = {N_SUBJ} subjects loaded from {_FILTERED_CSV}")

# Two anxiety scales are present in the questionnaire data:
#   Survey_ANXIETY_1_* (8 items, PSWQ-A)  = WORRY            → variable `worry`
#   Survey_ANXIETY_2_* (10 items, MASQ-AA) = SOMATIC AROUSAL  → variable `somatic_anxiety`
#
# >>> Per user (2026-04-28): NEVER use a bare `anxiety` variable name.   <<<
# >>> Always use `somatic_anxiety` or `worry` explicitly, and run every  <<<
# >>> analysis on BOTH. Earlier code conflated the two by treating       <<<
# >>> `anxiety` as somatic; that conflation is now removed.              <<<
_worry_items   = [c for c in rt_df.columns
                  if c.startswith('Survey_ANXIETY_1') and 'Total' not in c]
_somatic_items = [c for c in rt_df.columns
                  if c.startswith('Survey_ANXIETY_2') and 'Total' not in c]
print(f"WORRY (PSWQ-A)   items: n={len(_worry_items)}    -> variable `worry`")
print(f"SOMATIC (MASQ-AA) items: n={len(_somatic_items)}   -> variable `somatic_anxiety`")
rt_df['worry']           = rt_df[_worry_items].sum(axis=1, skipna=False)
rt_df['somatic_anxiety'] = rt_df[_somatic_items].sum(axis=1, skipna=False)
# Canonical list of anxiety predictors used throughout the notebook.
# Every downstream analysis loops over BOTH and reports each separately.
ANX_VARS = ['somatic_anxiety', 'worry']
print(f"worry:           M={rt_df['worry'].mean():.2f}  SD={rt_df['worry'].std():.2f}  "
      f"range=[{rt_df['worry'].min():.0f},{rt_df['worry'].max():.0f}]")
print(f"somatic_anxiety: M={rt_df['somatic_anxiety'].mean():.2f}  SD={rt_df['somatic_anxiety'].std():.2f}  "
      f"range=[{rt_df['somatic_anxiety'].min():.0f},{rt_df['somatic_anxiety'].max():.0f}]")
print(f"\nDownstream loops will iterate ANX_VARS = {ANX_VARS}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1.  TRIAL-LEVEL FRAME with subjective EV + canonical/subj accuracy
# ══════════════════════════════════════════════════════════════════════
rows = []
for _, sj in rt_df.iterrows():
    sid = sj['subject_id']
    # Per-stim subject ratings: Rating_2 = post-rev1 (used for Q1 trials),
    # Rating_3 = post-rev2 (used for Q2 trials)
    r2_F, r2_T = sj.get('Rating_2_Feather_Val', np.nan), sj.get('Rating_2_Triangle_Val', np.nan)
    r3_F, r3_T = sj.get('Rating_3_Feather_Val', np.nan), sj.get('Rating_3_Triangle_Val', np.nan)

    for q in [1, 2]:
        for t in range(1, 13):   # PATCH v4_1: V4 has 12 trials/query (was 8 in V3)
            pfx = f'Query_{q}_T{t}'
            state  = sj.get(f'{pfx}_State')
            choice = sj.get(f'{pfx}_Choice')
            offer  = sj.get(f'{pfx}_FixedOffer')
            rt_ms  = sj.get(f'{pfx}_RT')
            if pd.isna(state) or pd.isna(choice) or pd.isna(offer):
                continue

            ev = EV_MAP.get((q, state), np.nan)        # canonical EV
            mf = MF_MAP[state]                          # MF value
            # Subjective rating for this (q, stim)
            if q == 1: subj_rating = r2_F if state == 'Feather' else r2_T
            else:      subj_rating = r3_F if state == 'Feather' else r3_T

            ev_diff      = ev          - offer            # canonical
            ev_diff_subj = subj_rating - offer            # subjective
            ev_diff_mf   = mf          - offer

            # Canonical optimal: gamble if EV > offer.
            # Per user (2026-04-28): when EV == offer, the trial is value-neutral
            # — either choice is equally rational — so accuracy is 0.5, not 0/1.
            if ev > offer:
                optimal       = 1
                correct_canon = 1.0 if choice == 1 else 0.0
            elif ev < offer:
                optimal       = 0
                correct_canon = 1.0 if choice == 0 else 0.0
            else:  # ev == offer
                optimal       = 0.5
                correct_canon = 0.5
            # Same rule for subjective accuracy: rating == offer ⇒ 0.5.
            if pd.isna(subj_rating):
                optimal_subj = np.nan
                correct_subj = np.nan
            elif subj_rating > offer:
                optimal_subj = 1
                correct_subj = 1.0 if choice == 1 else 0.0
            elif subj_rating < offer:
                optimal_subj = 0
                correct_subj = 1.0 if choice == 0 else 0.0
            else:
                optimal_subj = 0.5
                correct_subj = 0.5
            optimal_mf    = 1 if mf >= offer else 0  # MF strict; only used for anti-MF detection

            # Anti-MF detection still uses the historical strict ≥ rule for MB
            # so anti-MF labels are unaffected by the 0.5-on-tie accuracy change.
            mb_strict      = 1 if ev >= offer else 0
            mf_says_gamble = optimal_mf
            is_anti_mf = int(mb_strict != mf_says_gamble)
            chose_mb   = int(choice == mb_strict) if is_anti_mf == 1 else np.nan

            log_rt_ms = np.log(rt_ms) if pd.notna(rt_ms) and rt_ms > 0 else np.nan
            rows.append({
                'subject_id':    sid,
                'query_num':     q,
                'trial':         t,
                'state':         state,
                'is_triangle':   int(state == 'Triangle'),
                'is_q2':         int(q == 2),
                'choice':        int(choice),
                'offer':         float(offer),
                'ev':            ev,
                'mf_value':      mf,
                'subj_rating':   subj_rating,
                'ev_diff':       ev_diff,           # canonical
                'ev_diff_subj':  ev_diff_subj,      # subjective
                'abs_ev_diff':       abs(ev_diff),
                'abs_ev_diff_subj':  abs(ev_diff_subj) if not np.isnan(ev_diff_subj) else np.nan,
                'ev_pos':       int(ev_diff > 0),       # canonical sign(EV − offer)
                'ev_pos_subj':  int(ev_diff_subj > 0) if not np.isnan(ev_diff_subj) else np.nan,
                'correct_canon': correct_canon,
                'correct_subj':  correct_subj,
                'is_anti_mf':    is_anti_mf,
                'chose_mb':      chose_mb,
                'rt_ms':         rt_ms,
                'log_rt_ms':     log_rt_ms,
                # Two separate anxiety predictors. NEVER use a bare 'anxiety'
                # column downstream; always specify which scale.
                'somatic_anxiety': sj['somatic_anxiety'],
                'worry':           sj['worry'],
            })

rt_trial = pd.DataFrame(rows)
rt_trial = rt_trial.dropna(subset=['choice'])
print(f"Trial-level frame: {len(rt_trial)} rows from {rt_trial['subject_id'].nunique()} subjects")
print(f"  Canonical correct:   M={rt_trial['correct_canon'].mean():.3f}")
print(f"  Subjective correct:  M={rt_trial['correct_subj'].mean():.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2.  MOTOR BASELINE = first-trial Step0 RT, averaged over SL and VL
# ══════════════════════════════════════════════════════════════════════
# Per user (2026-04-28): use the FIRST TRIAL ONLY of SL and VL together.
# Rationale: T1 captures the cleanest motor/encoding speed before any
# task-specific learning has happened in the phase. Combining the two
# phases (SL_T1 + VL_T1) gives a 2-observation per-subject estimate.
sl_cols = sorted([c for c in rt_df.columns
                  if c.startswith('SL_T1_') and 'Step0_RT' in c])
vl_cols = sorted([c for c in rt_df.columns
                  if c.startswith('VL_T1_') and 'Step0_RT' in c])
motor_cols = sl_cols + vl_cols
print(f"Motor-baseline source: {len(motor_cols)} columns "
      f"(SL_T1 + VL_T1 Step0_RT, first trial of each phase)")
print(f"  cols: {motor_cols}")

# Per-subject mean across the T1 step RTs (in seconds) — the motor baseline
motor_s = (rt_df[motor_cols].mean(axis=1) / 1000.0).rename('motor_s')
rt_motor_map = pd.Series(motor_s.values, index=rt_df['subject_id'].values).to_dict()

# Per-trial bookkeeping (kept for downstream LMMs that use trial-level data)
rt_trial['rt_s']      = (rt_trial['rt_ms'] / 1000.0).clip(lower=0.01)
rt_trial['log_rt']    = np.log(rt_trial['rt_s'])
rt_trial['motor_s']   = rt_trial['subject_id'].map(rt_motor_map)
rt_trial['delib_s']   = (rt_trial['rt_s'] - rt_trial['motor_s'].fillna(0)).clip(lower=0.01)
rt_trial['log_delib'] = np.log(rt_trial['delib_s'])

print(f"Per-subject motor baseline (mean of SL_T1 + VL_T1 Step0_RT, s): "
      f"M={pd.Series(rt_motor_map).mean():.3f}, "
      f"median={pd.Series(rt_motor_map).median():.3f}, "
      f"range [{pd.Series(rt_motor_map).min():.3f}, {pd.Series(rt_motor_map).max():.3f}]")
print(f"Per-trial query RT (s): median={rt_trial['rt_s'].median():.2f}")

> **⚠ Stale output:** code is canonical (159/277); the displayed output is the old 217 run. Re-run to refresh.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.  MODEL-AGNOSTIC MODEL-BASEDNESS  (per-stim)  +  ABOVE-CHANCE GROUP
# ══════════════════════════════════════════════════════════════════════
# FIX 2026-06-11: CANONICAL scoring is authoritative — it matches the
# model-agnostic classifier in Modelling_AllChoices_V4 (159/277 MB).
#   canonical (PRIMARY): a choice counts as "MB" iff it matches the canonical
#       (probability-weighted) MB choice; a trial is anti-MF iff MF disagrees
#       with the canonical MB choice.   -> mb_score, mb_score_feather/triangle
#   lenient  (reference only): ALSO credits the deterministic "best-leaf" MB
#       choice (canon OR det). That over-counts MB (217/277): on the Q2 trials
#       where canonical and best-leaf MB disagree it credits best-leaf choices,
#       promoting 58 subjects. Kept only as *_lenient diagnostic columns.
#
# Classification: subject is "MB" if CANONICAL mb_score > 0.5 (above chance)
# on EITHER Feather OR Triangle. Otherwise "non-MB".

# ── EV maps ──
EV_CANON_MAP = EV_MAP                  # from cell 1  (PRIMARY)
EV_DET_MAP   = {
    (1, 'Feather'): -15, (1, 'Triangle'): 15,
    (2, 'Feather'):   0, (2, 'Triangle'): 20,   # best leaf (reference only)
}

# ── Per-trial canonical + deterministic MB choice / anti-MF flags ──
_tt = rt_trial.copy()
_tt['ev_canon']  = _tt.apply(lambda r: EV_CANON_MAP[(int(r['query_num']), r['state'])], axis=1)
_tt['ev_det']    = _tt.apply(lambda r: EV_DET_MAP  [(int(r['query_num']), r['state'])], axis=1)
_tt['mb_canon']  = (_tt['ev_canon']  >= _tt['offer']).astype(int)
_tt['mb_det']    = (_tt['ev_det']    >= _tt['offer']).astype(int)
_tt['mf_say']    = (_tt['mf_value']  >= _tt['offer']).astype(int)
# Lenient (reference): MF disagrees with EITHER canonical OR deterministic MB.
_tt['is_anti_mf_lenient'] = (
    (_tt['mb_canon'] != _tt['mf_say']) | (_tt['mb_det'] != _tt['mf_say'])
).astype(int)
_tt['chose_mb_lenient']   = (
    (_tt['choice'] == _tt['mb_canon']) | (_tt['choice'] == _tt['mb_det'])
).astype(int)

# rt_trial already carries the CANONICAL flags is_anti_mf / chose_mb (cell 8 / §1).
n_anti_canon   = int((_tt['is_anti_mf'] == 1).sum())
n_anti_lenient = int((_tt['is_anti_mf_lenient'] == 1).sum())
print(f"Anti-MF trials  —  canonical (PRIMARY): {n_anti_canon},  "
      f"lenient (canon OR det, reference): {n_anti_lenient}")

# ── Overall + per-stim scores: CANONICAL primary ('') and lenient ('_lenient') ──
def _overall_score(df_trials, flag_col, score_col, suffix):
    return (df_trials[df_trials[flag_col] == 1]
            .groupby('subject_id')[score_col]
            .agg(['mean','count'])
            .rename(columns={'mean':f'mb_score{suffix}',
                             'count':f'n_anti_mf{suffix}'})
            .reset_index())

def _per_stim_score(df_trials, flag_col, score_col, suffix):
    p = (df_trials[df_trials[flag_col] == 1]
         .groupby(['subject_id','state'])[score_col]
         .agg(['mean','count'])
         .rename(columns={'mean':f'mb_score{suffix}',
                          'count':f'n_anti_mf{suffix}'})
         .reset_index())
    w = p.pivot(index='subject_id', columns='state',
                values=[f'mb_score{suffix}', f'n_anti_mf{suffix}'])
    w.columns = [f'{m}_{s.lower()}' for m, s in w.columns]
    return w.reset_index()

_overall_C = _overall_score(_tt, 'is_anti_mf',         'chose_mb',         '')          # CANONICAL primary
_overall_L = _overall_score(_tt, 'is_anti_mf_lenient', 'chose_mb_lenient', '_lenient')  # reference
_perstim_C = _per_stim_score(_tt, 'is_anti_mf',         'chose_mb',         '')
_perstim_L = _per_stim_score(_tt, 'is_anti_mf_lenient', 'chose_mb_lenient', '_lenient')

mb_per_subj = (_overall_C
               .merge(_perstim_C, on='subject_id', how='left')
               .merge(_overall_L, on='subject_id', how='left')
               .merge(_perstim_L, on='subject_id', how='left'))

print(f"\nMB-ness scored on {n_anti_canon} canonical anti-MF trials (PRIMARY).")
print(f"  Overall mb_score (canonical):   M={mb_per_subj['mb_score'].mean():.3f},  "
      f"median={mb_per_subj['mb_score'].median():.3f}")
print(f"  Overall mb_score (lenient ref): M={mb_per_subj['mb_score_lenient'].mean():.3f},  "
      f"median={mb_per_subj['mb_score_lenient'].median():.3f}")
for stim in ['feather','triangle']:
    sC = mb_per_subj[f'mb_score_{stim}'].dropna()
    print(f"  mb_score_{stim:8s} (canonical): M={sC.mean():.3f},  median={sC.median():.3f},  "
          f"above-chance: {int((sC > 0.5).sum())}/{len(sC)}")

# ── Above-chance classification: MB if CANONICAL mb_score > 0.5 on EITHER stim ──
mb_per_subj['above_feather']  = (mb_per_subj['mb_score_feather']  > 0.5).fillna(False)
mb_per_subj['above_triangle'] = (mb_per_subj['mb_score_triangle'] > 0.5).fillna(False)
mb_per_subj['mb_group'] = np.where(
    mb_per_subj['above_feather'] | mb_per_subj['above_triangle'],
    'MB', 'non-MB')
print(f"\nAbove-chance MB classification (CANONICAL mb_score > 0.5 on EITHER stim):")
print(f"  MB     (above chance on at least one stim): n={(mb_per_subj['mb_group']=='MB').sum()}")
print(f"  non-MB (not above chance on either stim):   n={(mb_per_subj['mb_group']=='non-MB').sum()}")

# Reference only: what the lenient (canon-OR-deterministic) rule would have given.
_above_F_l = (mb_per_subj['mb_score_lenient_feather']  > 0.5).fillna(False)
_above_T_l = (mb_per_subj['mb_score_lenient_triangle'] > 0.5).fillna(False)
mb_group_lenient = np.where(_above_F_l | _above_T_l, 'MB', 'non-MB')
n_extra = int(((mb_group_lenient=='MB') & (mb_per_subj['mb_group']=='non-MB')).sum())
print(f"  (lenient rule would give n_MB={int((mb_group_lenient=='MB').sum())}; "
      f"over-counts by {n_extra} subjects — NOT used.)")

print(f"\nDecomposition (canonical):")
both    = int(( (mb_per_subj['above_feather']) & ( mb_per_subj['above_triangle'])).sum())
only_F  = int(( (mb_per_subj['above_feather']) & (~mb_per_subj['above_triangle'])).sum())
only_T  = int(((~mb_per_subj['above_feather']) & ( mb_per_subj['above_triangle'])).sum())
neither = int(((~mb_per_subj['above_feather']) & (~mb_per_subj['above_triangle'])).sum())
print(f"  Above on both stims:       n={both}")
print(f"  Above only on Feather:     n={only_F}")
print(f"  Above only on Triangle:    n={only_T}")
print(f"  Above on neither (non-MB): n={neither}")

# Per-subject anxiety lookup — pull BOTH somatic_anxiety and worry
anx_per_subj = (rt_trial.groupby('subject_id')[['somatic_anxiety','worry']]
                       .first().reset_index())
subj_meta = mb_per_subj.merge(anx_per_subj, on='subject_id', how='left')
subj_meta = subj_meta.dropna(subset=['somatic_anxiety','worry']).copy()
print(f"\nSubjects with both anxiety scales + MB scores: {len(subj_meta)}")

# Sanity: each MB score vs each anxiety scale (both reported)
for anx in ['somatic_anxiety','worry']:
    print(f"\n  {anx}:")
    for col in ['mb_score','mb_score_feather','mb_score_triangle',
                'mb_score_lenient','mb_score_lenient_feather','mb_score_lenient_triangle']:
        v = subj_meta[col].fillna(subj_meta[col].median())
        rs, ps = spearmanr(subj_meta[anx], v)
        tag = ' (canon)  ' if 'lenient' not in col else ' (lenient)'
        print(f"    {anx} ~ {col:30s}{tag}:  rho={rs:+.3f}, p={ps:.3g}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A.  PER-SUBJECT × CONDITION SUMMARY TABLE  (medians, not means)
# ══════════════════════════════════════════════════════════════════════
# Per-subject scalars per condition (all / feather / triangle):
#   median_rt_s   = median of per-trial RT (seconds) on that condition
#   delib_s       = max(median_rt_s − motor_baseline_s,  0.01)
#                   (motor_baseline = per-subject median SL Step RT)
#   log_rt        = log(median_rt_s)
#   log_delib     = log(delib_s)
#   acc_canon, acc_subj  = mean accuracy on that condition
def _agg_subj(d, motor_baseline=None):
    # Per-subject scalars used downstream by the single OT definition:
    #   OT = z(delib_log) - z(logit(acc_canon))
    # where delib_log = mean over trials of log(per-trial delib_s).
    # Per user (2026-04-28): all delib measures are on the log scale.
    # `motor_baseline` is unused — per-trial delib_s and log_delib already
    # exist on rt_trial (computed once in the motor-baseline cell).
    return pd.Series({
        'median_rt_s': d['rt_s'].median(),       # raw seconds (descriptive)
        'delib_log':   d['log_delib'].mean(),    # PRIMARY: mean(log(per-trial delib_s))
        'log_rt_med':  np.log(max(d['rt_s'].median(), 0.01)),  # diagnostic only
        'acc_canon':   d['correct_canon'].mean(),
        'n_trials':    len(d),
    })

panels = {
    'all':      rt_trial,
    'feather':  rt_trial[rt_trial['state'] == 'Feather'],
    'triangle': rt_trial[rt_trial['state'] == 'Triangle'],
}
rt_per_subj = None
for cond, df_cond in panels.items():
    s_rows = []
    for sid, g in df_cond.groupby('subject_id'):
        mb = rt_motor_map.get(sid, np.nan)
        if not np.isfinite(mb): continue
        row = _agg_subj(g, mb)
        row['subject_id'] = sid
        s_rows.append(row)
    s = pd.DataFrame(s_rows)
    s = s.rename(columns={c: f'{c}_{cond}' for c in s.columns if c != 'subject_id'})
    rt_per_subj = s if rt_per_subj is None else rt_per_subj.merge(s, on='subject_id', how='outer')

# Attach MB group + BOTH anxiety scales (somatic_anxiety + worry).
# Per user (2026-04-28): never store a bare `anxiety` column on rt_per_subj;
# always carry both scales explicitly.
rt_per_subj = rt_per_subj.merge(
    subj_meta[['subject_id','mb_score','mb_group','somatic_anxiety','worry']],
    on='subject_id', how='left')
rt_per_subj = rt_per_subj.dropna(subset=['somatic_anxiety','worry']).copy()
print(f"Per-subject table: {len(rt_per_subj)} subjects, {rt_per_subj.shape[1]} columns")
print(f"\n  Per-subject median query RT (s):  M={rt_per_subj['median_rt_s_all'].mean():.2f}  "
      f"med={rt_per_subj['median_rt_s_all'].median():.2f}")
print(f"  Per-subject delib_log = mean(log(per-trial delib_s)):"
      f"  M={rt_per_subj['delib_log_all'].mean():+.3f}  "
      f"med={rt_per_subj['delib_log_all'].median():+.3f}")

### 1A-pre — Distribution check before z-scoring

Before pool-scaling the per-subject RT and deliberation values for
the OT computation, we check whether their distributions are
heavy-tailed. If they are, **standard z-scoring (mean / SD)** will
be unduly influenced by extreme subjects and will compress the
middle of the scale; **robust scaling** ((x − median) / (IQR /
1.349)) gives a similar normalised metric that's z-equivalent
under normality but resistant to outliers.

We plot the per-subject distributions of:

* `median_rt_s` (raw seconds — the median of query RTs)
* `delib_s`     (median query RT − motor baseline)
* `log_rt`, `log_delib` (their log-transforms)

Skewness > 1 or excess kurtosis > 1 ⇒ flag as heavy-tailed.
The diagnostic chooses **standard z** for normal-ish variables
and **robust scaling** for heavy-tailed ones, then scales each
column individually.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A-pre.  RT/Delib DISTRIBUTION CHECK + ROBUST-SCALING DECISION
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import skew, kurtosis

DIAG_COLS = []
for cond in ['all','feather','triangle']:
    DIAG_COLS += [f'median_rt_s_{cond}', f'delib_log_{cond}', f'log_rt_med_{cond}']

print(f"\nDistribution diagnostics (per-subject scalars):")
print(f"{'column':>22s}  {'M':>7s} {'med':>7s} {'SD':>7s} {'IQR':>7s} "
      f"{'skew':>7s} {'kurt':>7s}  {'heavy?':>7s}")
diag_results = {}
for col in DIAG_COLS:
    if col not in rt_per_subj.columns: continue
    x = rt_per_subj[col].dropna().values
    if len(x) < 10: continue
    sk = float(skew(x)); kt = float(kurtosis(x))   # Fisher (excess) kurt
    iqr = float(np.percentile(x, 75) - np.percentile(x, 25))
    heavy = (abs(sk) > 1.0) or (abs(kt) > 1.0)
    diag_results[col] = dict(skew=sk, kurt=kt, iqr=iqr, heavy=bool(heavy),
                              med=float(np.median(x)), sd=float(np.std(x, ddof=1)))
    flag = 'YES' if heavy else 'no'
    print(f"{col:>22s}  {x.mean():>7.3f} {np.median(x):>7.3f} "
          f"{x.std(ddof=1):>7.3f} {iqr:>7.3f} {sk:>+7.2f} {kt:>+7.2f}  {flag:>7s}")

# ── Plot: 3 columns × 2 rows, raw RT + log RT + delib_log distributions ──
BG = "#FAFBFC"
fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor=BG)
COND = 'all'
COL_ROW = [
    [(f'median_rt_s_{COND}', 'median query RT (s)', 'raw'),
     (f'log_rt_med_{COND}',  'log(median query RT)', 'log'),
     (f'delib_log_{COND}',   'delib_log = mean(log(delib_t))', 'log')],
    [(f'median_rt_s_{COND}', 'median query RT (s)', 'raw'),
     (f'log_rt_med_{COND}',  'log(median RT)', 'log'),
     (f'delib_log_{COND}',   'delib_log', 'log')],
]
# Row 0: histograms
for j, (col, lab, kind) in enumerate(COL_ROW[0]):
    ax = axes[0, j]; ax.set_facecolor(BG)
    x = rt_per_subj[col].dropna().values
    n_bins = max(10, min(30, int(np.sqrt(len(x)))))
    ax.hist(x, bins=n_bins, color='#fdae6b', edgecolor='black', alpha=.85)
    d = diag_results.get(col, {})
    ax.axvline(d.get('med', np.nan), color='black', ls='--', alpha=.7,
                label=f"median = {d.get('med', float('nan')):.2f}")
    ax.set_xlabel(lab); ax.set_ylabel('# subjects')
    ax.set_title(f"{lab}\nskew={d.get('skew',float('nan')):+.2f}, "
                 f"kurt={d.get('kurt',float('nan')):+.2f}  "
                 f"{'⚠ heavy-tailed' if d.get('heavy') else 'OK'}",
                  fontweight='bold' if d.get('heavy') else 'normal', fontsize=9)
    ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Row 1: Q-Q plot vs normal
from scipy.stats import probplot
for j, (col, lab, kind) in enumerate(COL_ROW[1]):
    ax = axes[1, j]; ax.set_facecolor(BG)
    x = rt_per_subj[col].dropna().values
    probplot(x, dist='norm', plot=ax)
    ax.get_lines()[0].set_marker('o'); ax.get_lines()[0].set_color('#08519c')
    ax.get_lines()[0].set_markersize(4); ax.get_lines()[0].set_alpha(0.7)
    ax.get_lines()[1].set_color('red'); ax.get_lines()[1].set_lw(1.5)
    ax.set_xlabel("Theoretical normal quantile")
    ax.set_ylabel(f"observed {lab.split(' ')[0]}")
    ax.set_title(f"Q-Q  ({lab})", fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Section 1A-pre — distribution check before scaling  (RT and delib, all-trials condition)",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section1A_distribution_check.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section1A_distribution_check.png")

# ── Verdict: print which variables will be robust-scaled ──
print("\nScaling decision per variable:")
for col, d in diag_results.items():
    method = 'robust  (median / IQR-1.349)' if d['heavy'] else 'standard z (mean / SD)'
    print(f"  {col:>22s} → {method}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A.  SINGLE OT DEFINITION
#       OT = z(delib_log) - z( logit(canonical_accuracy) )
# ══════════════════════════════════════════════════════════════════════
# Per user (2026-04-28) every cell adheres to ONE OT definition with all
# delib-time measures on the log scale:
#
#     OT = z( delib_log )  -  z( logit(acc_canon) )
#
# delib_log  = mean over trials of log( max(rt_t - motor_baseline, 0.01) )
#              i.e. per-subject mean(log(per-trial delib_s)).
# acc_canon  = mean of correct_canon, where correct_canon ∈ {0, 0.5, 1};
#              ties (EV == offer) score 0.5 regardless of choice.
# logit      = log(p / (1-p)) with p clipped to [0.025, 0.975].
# z(.)       = pool z-score (sample mean, sample SD) across subjects.
#
# Trial subsets retained: {all, feather, triangle}. (anti-MF and
# non-anti-MF are computed in §2.12 with the same formula.)

def _zscale(s):
    '''Standard pool z, NaN-safe.'''
    s_clean = s.dropna()
    out = pd.Series(np.nan, index=s.index)
    if len(s_clean) < 3 or s_clean.std() == 0:
        return out
    out.loc[s.notna()] = (s_clean - s_clean.mean()) / s_clean.std()
    return out

def _logit(p, eps=0.025):
    p = pd.Series(p).clip(lower=eps, upper=1-eps)
    return np.log(p / (1 - p))

for cond in ['all','feather','triangle']:
    rt_per_subj[f'z_delib_log_{cond}']     = _zscale(rt_per_subj[f'delib_log_{cond}'])
    rt_per_subj[f'logit_acc_canon_{cond}'] = _logit(rt_per_subj[f'acc_canon_{cond}'])
    rt_per_subj[f'z_acc_canon_{cond}']     = _zscale(rt_per_subj[f'logit_acc_canon_{cond}'])
    rt_per_subj[f'OT_{cond}']              = (rt_per_subj[f'z_delib_log_{cond}']
                                           - rt_per_subj[f'z_acc_canon_{cond}'])

# Diagnostics
from scipy.stats import skew as _skew, kurtosis as _kurt
print("\ndelib_log = mean(log(per-trial delib_s)) distributions:")
for cond in ['all','feather','triangle']:
    x = rt_per_subj[f'delib_log_{cond}'].dropna().values
    print(f"  delib_log_{cond:8s}: M={x.mean():+.2f} med={np.median(x):+.2f} "
          f"SD={x.std(ddof=1):.2f} skew={float(_skew(x)):+.2f} kurt={float(_kurt(x)):+.2f}")

print("\nLogit canonical-accuracy distributions (acc tied=0.5 rule):")
for cond in ['all','feather','triangle']:
    x = rt_per_subj[f'logit_acc_canon_{cond}'].dropna().values
    print(f"  logit_acc_canon_{cond:8s}: M={x.mean():+.2f} med={np.median(x):+.2f} "
          f"SD={x.std(ddof=1):.2f} skew={float(_skew(x)):+.2f} kurt={float(_kurt(x)):+.2f}")

print("\nOT = z(delib_log) - z(logit(acc_canon)) summary:")
for cond in ['all','feather','triangle']:
    x = rt_per_subj[f'OT_{cond}'].dropna().values
    print(f"  OT_{cond:8s}: M={x.mean():+.2f} med={np.median(x):+.2f} "
          f"SD={x.std(ddof=1):.2f} range=[{x.min():+.2f},{x.max():+.2f}] n={len(x)}")

rt_per_subj.to_csv("rt_analyses_per_subject.csv", index=False)
print("\nSaved rt_analyses_per_subject.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RT / OT PREP (lean) — per-subject `subj` table WITHOUT CBM labels
# ══════════════════════════════════════════════════════════════════════
# Builds the per-subject deliberation + canonical-accuracy table the poster
# H1/H2/H3 + OT cells consume, using ONLY model-agnostic inputs so the
# headline analyses run BEFORE the heavy CBM block. `model` is left NaN here;
# the full RT-descriptives cell in the Modeling section rebuilds `subj` with
# real CBM labels, at which point the CBM-group panels populate (confirmation).
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, rankdata, mannwhitneyu, kruskal
BG_COLOR="#FAFBFC"; ANX_COLORS={"Low":"#3498DB","High":"#E74C3C"}
MODEL_COLORS={"MF":"#3498DB","MB":"#2ECC71"}
MODEL_LABELS={"MF":"MF-only (M2)","MB":"Model-Based (M3 + M7 + M9)"}
MODEL_SHORT={"MF":"MF","MB":"MB"}; MODEL_GROUPS=["MF","MB"]

sl_cols=sorted([c for c in df.columns if c.startswith("SL_T") and ("Step0_RT" in c or "Step1_RT" in c)])
_m=df[["subject_id"]+sl_cols].copy(); _m["subject_id"]=_m["subject_id"].astype(str)
_m["motor_s"]=_m[sl_cols].median(axis=1)/1000
motor_map=_m.set_index("subject_id")["motor_s"].to_dict()
rt=trial_df[trial_df["log_rt"].notna()].copy(); rt["subject_id"]=rt["subject_id"].astype(str)
rt["rt_s"]=np.exp(rt["log_rt"])/1000; rt["motor_s"]=rt["subject_id"].map(motor_map)
rt["delib_s"]=(rt["rt_s"]-rt["motor_s"].fillna(0)).clip(lower=0.05); rt["log_delib"]=np.log(rt["delib_s"])
model_map_s=({str(s):i["model"] for s,i in winning_params.items()} if "winning_params" in dir() else {})
rt["model"]=rt["subject_id"].map(model_map_s)
anx_map=rt.groupby("subject_id")["somatic_anxiety"].first(); med_anx=anx_map.median()
rt["anx_group"]=rt["subject_id"].map(anx_map.apply(lambda a:"High" if a>=med_anx else "Low"))
subj=rt.groupby("subject_id").agg(
    mean_delib=("delib_s","mean"), median_delib=("delib_s","median"),
    mean_log_delib=("log_delib","mean"), motor_s=("motor_s","first"),
    mean_rt_s=("rt_s","mean"), somatic_anxiety=("somatic_anxiety","first"),
    model=("model","first"), anx_group=("anx_group","first"),
    accuracy=("correct","mean"), accuracy_canon=("correct_canon","mean"),
).reset_index()
print(f"[lean prep] subj: {len(subj)} subjects | CBM labels present: {bool(subj['model'].notna().any())}")


---

# Section 2 · Main / Poster Analyses (model-agnostic)

Headline results on the 159 classifier — run straight after prep, no CBM.

---
## 6 · Overthinking Analysis (Condensed)

**Model-agnostic approach**: Split trials by |EV diff| (median → Hard/Easy) and subjects by anxiety (median → High/Low).

1. **Log RT** by difficulty × anxiety
2. **P(Chose MB)** by difficulty × anxiety
3. **Overthinking metrics**: original index + alternate definition

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 6A  MODEL-AGNOSTIC 2×2 PLOTS: Difficulty × Somatic Anxiety
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = '#FAFBFC'

# ── Setup: difficulty split & anxiety split on anti-MF trials ──
anti_ot = anti_mf_df[anti_mf_df['log_rt'].notna()].copy()
median_diff = anti_ot['abs_ev_diff'].median()
anti_ot['difficulty'] = np.where(anti_ot['abs_ev_diff'] <= median_diff, 'Hard', 'Easy')

median_anx = anti_ot.groupby('subject_id')['somatic_anxiety'].first().median()
anti_ot['anx_group'] = anti_ot['subject_id'].map(
    anti_ot.groupby('subject_id')['somatic_anxiety'].first().apply(
        lambda a: 'High' if a >= median_anx else 'Low'))

n_subj = anti_ot['subject_id'].nunique()
print(f'Overthinking analysis: N = {n_subj}, {len(anti_ot)} anti-MF trials with RT')

# ── Cell means ──
cell_means = anti_ot.groupby(['anx_group', 'difficulty']).agg(
    log_rt_mean=('log_rt', 'mean'), log_rt_sem=('log_rt', 'sem'),
    mb_mean=('chose_mb', 'mean'), mb_sem=('chose_mb', 'sem'),
    n=('chose_mb', 'count'),
).reset_index()

print('\nCell means (Somatic Anxiety × Difficulty):')
for _, row in cell_means.iterrows():
    print(f'  {row["anx_group"]:>4s} ANX, {row["difficulty"]:>4s}: '
          f'logRT={row["log_rt_mean"]:.3f}±{row["log_rt_sem"]:.3f}  '
          f'P(MB)={row["mb_mean"]:.3f}±{row["mb_sem"]:.3f}  n={int(row["n"])}')

# ══════════════════════════════════════════════════════════════════════
# PLOT 1: Log RT by Difficulty × Somatic Anxiety (model-agnostic)
# ══════════════════════════════════════════════════════════════════════
ANX_COLORS = {'Low': '#3498DB', 'High': '#E74C3C'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), facecolor=BG_COLOR)

for ax, metric, sem_col, ylabel, title in [
    (ax1, 'log_rt_mean', 'log_rt_sem', 'Log RT', 'Deliberation Time (Log RT)'),
    (ax2, 'mb_mean',     'mb_sem',     'P(Chose MB)', 'MB Choice Rate')]:
    ax.set_facecolor(BG_COLOR)
    x_order = {'Hard': 0, 'Easy': 1}
    for anx in ['Low', 'High']:
        sub = cell_means[cell_means['anx_group'] == anx].copy()
        sub['x'] = sub['difficulty'].map(x_order)
        sub = sub.sort_values('x')
        offset = -0.05 if anx == 'Low' else 0.05
        ax.errorbar(sub['x'] + offset, sub[metric], yerr=1.96 * sub[sem_col],
                    color=ANX_COLORS[anx], marker='o', lw=2.5, capsize=5,
                    ms=9, label=f'{anx} Somatic Anxiety', markeredgecolor='white', markeredgewidth=1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Hard\n(low |ΔEV|)', 'Easy\n(high |ΔEV|)'])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if metric == 'mb_mean': ax.axhline(0.5, color='gray', ls=':', lw=0.8)

fig.suptitle('Model-Agnostic: Difficulty × Somatic Anxiety on Anti-MF Trials', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"fig_overthinking_condensed_{OUT_PREFIX}.png", dpi=200, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()

# ── Interaction tests ──
from scipy.stats import mannwhitneyu
for metric_name, metric_col in [('Log RT', 'log_rt'), ('P(MB)', 'chose_mb')]:
    hard_high = anti_ot[(anti_ot['difficulty']=='Hard') & (anti_ot['anx_group']=='High')][metric_col]
    hard_low  = anti_ot[(anti_ot['difficulty']=='Hard') & (anti_ot['anx_group']=='Low')][metric_col]
    easy_high = anti_ot[(anti_ot['difficulty']=='Easy') & (anti_ot['anx_group']=='High')][metric_col]
    easy_low  = anti_ot[(anti_ot['difficulty']=='Easy') & (anti_ot['anx_group']=='Low')][metric_col]
    # Difficulty effect by anxiety group
    t_h, p_h = stats.ttest_ind(hard_high, easy_high, equal_var=False)
    t_l, p_l = stats.ttest_ind(hard_low, easy_low, equal_var=False)
    print(f'\n{metric_name} — Difficulty effect:')
    print(f'  High ANX: Hard vs Easy t={t_h:.2f}, p={p_h:.4f}')
    print(f'  Low  ANX: Hard vs Easy t={t_l:.2f}, p={p_l:.4f}')
    # Anxiety effect on hard trials
    t_a, p_a = stats.ttest_ind(hard_high, hard_low, equal_var=False)
    print(f'  Hard trials: High vs Low ANX t={t_a:.2f}, p={p_a:.4f}')

In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# OVERTHINKING (preregistered formula)
# ══════════════════════════════════════════════════════════════════════
# Per-subject means only. Z-score ONCE across the full MF+MB sample
# (between-subject standardization), per prereg.
#
#   OT_i = z(delib_log_i) − z(logit(acc_canon_i))
#
# where logit(p) = log(p/(1-p)) with p clipped to [0.025, 0.975].
# Positive OT = OVERTHINKING (slow AND inaccurate).
# Negative OT = efficient (fast AND accurate).
# (This is the sign convention named in the prereg footnote — the prior
#  pilot's OT had the opposite sign; the current convention's predicted
#  H1/H2 correlations are POSITIVE.)

from scipy.stats import spearmanr, pearsonr, rankdata

BG_COLOR = "#FAFBFC"

# PREREG-FIDELITY: pool z-score across the FULL sample (no CBM filter),
# and carry the model-agnostic MB classifier on ot so H1/H2 inherit it.
ot = subj.copy()
ot = ot.merge(mb_score_df[['subject_id', 'mb_classif_agnostic']],
              on='subject_id', how='left')
ot = ot.dropna(subset=["somatic_anxiety", "mean_log_delib", "accuracy_canon"])

# Pooled z-scoring across ALL subjects in MF + MB
# Prereg formula — z(delib_log) − z(logit(acc_canon)), with the logit
# clipped to [0.025, 0.975] to avoid infinities at the extremes.
def _logit_clipped(p, lo=0.025, hi=0.975):
    p_c = np.clip(p, lo, hi)
    return np.log(p_c / (1.0 - p_c))

ot["z_delib_log"] = (ot["mean_log_delib"] - ot["mean_log_delib"].mean()) / ot["mean_log_delib"].std()
ot["logit_acc"]   = _logit_clipped(ot["accuracy_canon"])
ot["z_logit_acc"] = (ot["logit_acc"] - ot["logit_acc"].mean()) / ot["logit_acc"].std()
ot["ot"]          = ot["z_delib_log"] - ot["z_logit_acc"]    # POSITIVE = overthinking
# Keep legacy names available so downstream plot code that references
# z_rt / z_acc still works (z_rt aliases z_delib_log; z_acc aliases the
# new z_logit_acc — both are now under the prereg's transform).
ot["z_rt"]  = ot["z_delib_log"]
ot["z_acc"] = ot["z_logit_acc"]

print(f"Pooled z-scoring across {len(ot)} subjects (MF + MB)")
print(f"  mean_log_delib:   M={ot['mean_log_delib'].mean():+.3f},  SD={ot['mean_log_delib'].std():.3f}")
print(f"  accuracy:         M={ot['accuracy'].mean():+.3f},   SD={ot['accuracy'].std():.3f}")
print(f"  OT (pooled z):    M={ot['ot'].mean():+.3f},   SD={ot['ot'].std():.3f}")

# ── CBM-group OT panels/figure: only after the Modeling section builds CBM labels ──
if bool(subj['model'].notna().any()):

    print("\n" + "="*70)
    print("OT ~ Anxiety  (per group, pooled z-scoring)")
    print("="*70)

    rows = []
    for grp in MODEL_GROUPS:
        g = ot[ot["model"] == grp].dropna(subset=["somatic_anxiety","ot","z_rt","z_acc"])
        if len(g) < 2:
            rows.append(dict(grp=grp, n=len(g), rho_ot=np.nan, p_ot=np.nan,
                             r_ot=np.nan, p_ot_p=np.nan,
                             rho_rt=np.nan, p_rt=np.nan, rho_acc=np.nan, p_acc=np.nan))
            print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)}) — too few subjects for correlations, skipping")
            continue
        rho_ot,  p_ot_s  = spearmanr(g["somatic_anxiety"], g["ot"])
        r_ot,    p_ot_p  = pearsonr(g["somatic_anxiety"], g["ot"])
        rho_rt,  p_rt    = spearmanr(g["somatic_anxiety"], g["z_rt"])
        rho_acc, p_acc   = spearmanr(g["somatic_anxiety"], g["z_acc"])
        rows.append(dict(grp=grp, n=len(g), rho_ot=rho_ot, p_ot=p_ot_s,
                         r_ot=r_ot, p_ot_p=p_ot_p,
                         rho_rt=rho_rt, p_rt=p_rt, rho_acc=rho_acc, p_acc=p_acc))

        s_ot  = "***" if p_ot_s<.001  else "**" if p_ot_s<.01  else "*" if p_ot_s<.05  else ""
        s_rt  = "***" if p_rt<.001    else "**" if p_rt<.01    else "*" if p_rt<.05    else ""
        s_acc = "***" if p_acc<.001   else "**" if p_acc<.01   else "*" if p_acc<.05   else ""

        print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)})")
        print(f"    Anxiety ~ OT:              rho={rho_ot:+.3f}, p={p_ot_s:.4f} {s_ot:<3s}  "
              f"|  Pearson r={r_ot:+.3f}, p={p_ot_p:.4f}")
        print(f"    Anxiety ~ z(log_delib):    rho={rho_rt:+.3f},  p={p_rt:.4f} {s_rt}")
        print(f"    Anxiety ~ z(accuracy):     rho={rho_acc:+.3f},  p={p_acc:.4f} {s_acc}")
        if p_rt < .05 and p_acc > .1:
            print(f"    → OVERTHINKING signature: anxiety increases RT but NOT accuracy.")
        elif p_rt < .05 and p_acc < .05 and rho_acc > 0:
            print(f"    → Productive deliberation: anxiety increases both RT and accuracy.")
        elif p_rt > .1:
            print(f"    → No anxiety → RT effect in this group.")

    # Also a between-groups test: does OT differ between MF and MB?
    from scipy.stats import mannwhitneyu
    g_mf = ot[ot["model"] == "MF"]["ot"]
    g_mb = ot[ot["model"] == "MB"]["ot"]
    u, p_u = mannwhitneyu(g_mf, g_mb, alternative="two-sided")
    print(f"\n  Between-group OT: MF median={g_mf.median():+.3f}, MB median={g_mb.median():+.3f}")
    print(f"    Mann-Whitney U={u:.0f}, p={p_u:.4f}")

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 1: OT ~ Anxiety (rank-rank), one panel per group + RT/Acc scatter
    # ══════════════════════════════════════════════════════════════════════

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG_COLOR)

    for j, grp in enumerate(MODEL_GROUPS):
        ax = axes[j]; ax.set_facecolor(BG_COLOR)
        g = ot[ot["model"] == grp]
        xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
        ax.scatter(xr, yr, alpha=0.65, s=50,
                   color=MODEL_COLORS[grp], edgecolors="white", linewidths=0.6)
        z = np.polyfit(xr, yr, 1)
        xs = np.linspace(xr.min(), xr.max(), 100)
        ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
        rho, p = spearmanr(g["somatic_anxiety"], g["ot"])
        star = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_xlabel("Anxiety (rank)", fontsize=11)
        if j == 0:
            ax.set_ylabel("OT = z(acc) − z(log Delib) (rank)", fontsize=11)
        ax.set_title(f"{MODEL_LABELS[grp]}\nn={len(g)},  rho={rho:.3f},  p={p:.3f} {star}",
                     fontweight="bold", fontsize=11)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # Third panel: z(RT) vs z(acc), colored by anxiety, shaped by group
    ax = axes[2]; ax.set_facecolor(BG_COLOR)
    markers = {"MF": "o", "MB": "s"}
    for grp in MODEL_GROUPS:
        g = ot[ot["model"] == grp]
        sc = ax.scatter(g["z_rt"], g["z_acc"], c=g["somatic_anxiety"], cmap="RdYlBu_r",
                        alpha=0.8, s=65, marker=markers[grp],
                        edgecolors=MODEL_COLORS[grp], linewidths=1.8,
                        label=f"{MODEL_SHORT[grp]} (n={len(g)})")
    plt.colorbar(sc, ax=ax, label="Anxiety", shrink=0.8)
    ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
    ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
    ax.set_xlabel("z(log Delib RT)   ← faster | slower →", fontsize=10)
    ax.set_ylabel("z(accuracy)   ← less | more →", fontsize=10)
    ax.set_title("RT × Accuracy (pooled z)\nupper-left = efficient  |  lower-right = overthinking",
                 fontweight="bold", fontsize=10)
    ax.legend(fontsize=9, loc="best")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("Simple OT = z(accuracy) − z(log Delib RT)   —   pooled z, per-group correlation",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_simple_pooled_z_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H1 (preregistered) — Somatic Anxiety × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Predictor : Survey_ANXIETY_2_Total_Score (10-item Physiological Anxiety
#             Checklist = somatic anxiety), assigned to df['somatic_anxiety']
#             in cell 3 of the V4 notebook (post-rename 2026-05-27).
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across the
#             full MF+MB sample, computed in the simple-OT cell above and
#             held in `ot`. Positive OT = overthinking (prereg sign).
# Test      : Spearman ρ, one-sided positive p (per prereg, α = .05).
# Partition : mb_classif_agnostic — model-agnostic MB classifier per prereg.
# Panels    : All, mb_score-MB (PRIMARY, H1), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#
# Additive cell: parallels the H2 cell but substitutes the somatic-anxiety
# predictor. Does not modify any earlier analysis.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError("`ot` not defined. Run the simple OT × Anxiety cell first.")
if "somatic_anxiety" not in df.columns:
    raise RuntimeError("df['somatic_anxiety'] missing — re-run cell 3 (it sets "
                       "Anxiety = Survey_ANXIETY_2_Total_Score, the somatic scale).")
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score classifier cell.")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join somatic anxiety into the OT table
somatic_map = df.set_index("subject_id")["somatic_anxiety"]
ot_h1 = ot.copy()
ot_h1["somatic_anxiety"] = ot_h1["subject_id"].map(somatic_map)
ot_h1 = ot_h1.dropna(subset=["somatic_anxiety", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).

print("=" * 70)
print("H1 — Somatic Anxiety × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN somatic/OT: N = {len(ot_h1)}")

analysis_groups_h1 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
]
if bool(subj['model'].notna().any()):   # CBM panels only after the Modeling section
    analysis_groups_h1 += [
        ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory,
        ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory,
    ]
rows_h1 = []
for label, kind, val, is_primary in analysis_groups_h1:
    if kind == "all":
        g = ot_h1
    elif kind == "mb_agn":
        g = ot_h1[ot_h1["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h1[ot_h1["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h1.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["somatic_anxiety"], g["ot"])
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["somatic_anxiety"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h1_tag = "   ← H1 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h1_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h1.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot ──
_BG_H1 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H1 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#E74C3C",    # red — primary
    "nonMB (mb_score)": "#F1948A",
    "MF  (CBM)":        "#FAD7A0",    # exploratory
    "MB  (CBM)":        "#F5CBA7",    # exploratory
}

n_panels = len(rows_h1)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H1)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h1):
    ax = axes[j]; ax.set_facecolor(_BG_H1)
    ax.set_xlabel("Somatic Anxiety (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h1_tag = "  ← H1" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h1_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H1.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h1_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H1 — Somatic Anxiety × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h1_somatic_ot_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=_BG_H1)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H2 (preregistered) — Worry × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Additive cell: does not modify any earlier analysis. Parallels the
# simple OT × Anxiety cell above but substitutes the Worry predictor.
#
# Predictor : Survey_ANXIETY_1_Total_Score (8-item PSWQ trait Worry),
#             already assigned to df["worry"] in cell 3.
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across
#             MF+MB, computed in the simple-OT cell above and held in `ot`.
#             Positive OT = overthinking (prereg sign convention), so the
#             predicted one-sided positive Spearman ρ tests the H2 claim
#             that higher worry → more overthinking.
# Test      : Spearman rho with a one-sided positive p (per prereg, α=.05).
# Panels    : All, mb_score-MB (PRIMARY, H2), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#             Per prereg, H2 is the mb_score-MB panel.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError(
        "`ot` not defined. Run the simple OT × Anxiety cell first "
        "(it builds the pooled-z OT table this analysis reuses).")
if "worry" not in df.columns:
    raise RuntimeError(
        "df['worry'] missing — re-run cell 3 (it sets "
        "worry = sum of PSWQ-8 items = Survey_ANXIETY_1_Total_Score).")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join Worry into the OT table
worry_map = df.set_index("subject_id")["worry"]
ot_h2 = ot.copy()
ot_h2["worry"] = ot_h2["subject_id"].map(worry_map)
ot_h2 = ot_h2.dropna(subset=["worry", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score "
                       "classifier cell near the top of the notebook.")

print("=" * 70)
print("H2 — Worry (PSWQ trait) × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN Worry/OT: N = {len(ot_h2)}")

# Panel definitions: (label, partition_kind, partition_value, is_h2_primary)
#   partition_kind ∈ {'all', 'mb_agn', 'cbm'}
analysis_groups_h2 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
]
if bool(subj['model'].notna().any()):   # CBM panels only after the Modeling section
    analysis_groups_h2 += [
        ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory,
        ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory,
    ]
rows_h2 = []
for label, kind, val, is_primary in analysis_groups_h2:
    if kind == "all":
        g = ot_h2
    elif kind == "mb_agn":
        g = ot_h2[ot_h2["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h2[ot_h2["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h2.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["worry"], g["ot"])
    # one-sided positive: halve two-sided when rho>0, else 1 - half
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["worry"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h2_tag = "   ← H2 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h2_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h2.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot: rank-rank scatter, one panel per analysis group ──
_BG_H2 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H2 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#2ECC71",    # green — primary
    "nonMB (mb_score)": "#3498DB",
    "MF  (CBM)":        "#85C1E9",    # light blue — exploratory
    "MB  (CBM)":        "#A9DFBF",    # light green — exploratory
}

n_panels = len(rows_h2)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H2)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h2):
    ax = axes[j]; ax.set_facecolor(_BG_H2)
    ax.set_xlabel("Worry / PSWQ (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h2_tag = "  ← H2" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h2_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["worry"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H2.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h2_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H2 — Worry (PSWQ trait) × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h2_worry_ot_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=_BG_H2)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H3 (preregistered) — log-deliberation RT, MB vs non-MB
# ══════════════════════════════════════════════════════════════════════
# Prereg H3:
#   Per-subject mean log-transformed deliberation RT (delib_log_i) is larger
#   in MODEL-BASED-classified subjects than in NON-MODEL-BASED subjects.
#
# Test (per prereg Analysis Plan):
#   One-sided Mann-Whitney U on delib_log_i, alternative='greater' (MB > non-MB).
#   Rank-biserial effect size:  r_rb = 1 − 2·U / (n_MB · n_nonMB).
#   Linear-regression sensitivity (Welch-corrected SE) reported alongside.
#
# Partition (per prereg Variables): MODEL-AGNOSTIC mb_score classifier,
# computed in the mb_score classifier cell near the top of the notebook
# (column `mb_classif_agnostic` in `mb_score_df`). The CBM/HBI partition
# (`subj.model`) is NOT used here — that one feeds the convergent-validity
# κ check separately.
#
# Additive cell: does not modify any earlier analysis.
# ══════════════════════════════════════════════════════════════════════
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` not defined. Run the CBM/HBI cells, the PPC + group-collapse "
        "cells, then the RT DESCRIPTIVES cell before this one.")
if 'mb_score_df' not in dir():
    raise RuntimeError(
        "`mb_score_df` not defined. Run the model-agnostic mb_score "
        "classifier cell (near the top of the notebook, right after the "
        "trial-level dataframe build) before this one.")

from scipy.stats import mannwhitneyu
import statsmodels.api as sm

# Build H3 frame: subject × {mean_log_delib, mb_classif_agnostic}
h3 = (subj[['subject_id', 'mean_log_delib']]
      .merge(mb_score_df[['subject_id', 'mb_classif_agnostic',
                          'mb_score_F', 'mb_score_T']],
             on='subject_id', how='inner')
      .dropna(subset=['mean_log_delib', 'mb_classif_agnostic']))

mb_grp    = h3.loc[h3['mb_classif_agnostic'],  'mean_log_delib'].values
nonmb_grp = h3.loc[~h3['mb_classif_agnostic'], 'mean_log_delib'].values
n_mb, n_nonmb = len(mb_grp), len(nonmb_grp)

print("=" * 70)
print("H3 (preregistered) — log-deliberation RT, MB vs non-MB (model-agnostic)")
print("=" * 70)
print(f"  N total      : {n_mb + n_nonmb}")
print(f"  N MB         : {n_mb}")
print(f"  N non-MB     : {n_nonmb}")
if n_mb >= 1 and n_nonmb >= 1:
    print(f"  mean log_delib  MB     : {mb_grp.mean():+.3f}  (SD={mb_grp.std(ddof=1):.3f})")
    print(f"  mean log_delib  non-MB : {nonmb_grp.mean():+.3f}  (SD={nonmb_grp.std(ddof=1):.3f})")

# ── One-sided Mann-Whitney (MB greater than non-MB) ──
if n_mb >= 2 and n_nonmb >= 2:
    u_stat, p_one = mannwhitneyu(mb_grp, nonmb_grp, alternative='greater')
    _, p_two      = mannwhitneyu(mb_grp, nonmb_grp, alternative='two-sided')
    r_rb = 1.0 - (2.0 * u_stat) / (n_mb * n_nonmb)
    star = '***' if p_one < .001 else '**' if p_one < .01 else '*' if p_one < .05 else ''
    print()
    print(f"  Mann-Whitney U (MB > non-MB) : U = {u_stat:.1f}")
    print(f"  one-sided p                  : {p_one:.4f} {star}")
    print(f"  two-sided p (for reference)  : {p_two:.4f}")
    print(f"  rank-biserial r_rb           : {r_rb:+.3f}")
else:
    print()
    print("  Mann-Whitney : insufficient subjects in one group (need ≥ 2 each).")

# ── Welch-corrected OLS sensitivity ──
if n_mb >= 2 and n_nonmb >= 2:
    h3_ols = h3.copy()
    h3_ols['mb'] = h3_ols['mb_classif_agnostic'].astype(int)
    X = sm.add_constant(h3_ols['mb'])
    model_ols = sm.OLS(h3_ols['mean_log_delib'], X).fit(cov_type='HC3')
    print()
    print("  OLS sensitivity (Welch-style HC3 SE):  mean_log_delib ~ mb_group")
    print("  " + str(model_ols.summary2().tables[1]).replace('\n', '\n  '))

# ── Quick visual ──
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.5))
ax.set_facecolor('#FAFBFC')
positions = [0, 1]
bp = ax.boxplot([nonmb_grp, mb_grp], positions=positions, widths=0.55,
                patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='white',
                               markeredgecolor='black', markersize=7))
colors = ['#3498DB', '#2ECC71']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.55)
# Scatter
rng = np.random.default_rng(0)
for i, g in enumerate([nonmb_grp, mb_grp]):
    x_jit = positions[i] + rng.uniform(-0.13, 0.13, size=len(g))
    ax.scatter(x_jit, g, color=colors[i], edgecolors='white',
               s=42, alpha=0.85, linewidths=0.7)
ax.set_xticks(positions)
ax.set_xticklabels([f'non-MB\n(n={n_nonmb})', f'MB\n(n={n_mb})'])
ax.set_ylabel('mean log(deliberation RT)')
title = 'H3 — MB vs non-MB log-deliberation RT (model-agnostic partition)'
if n_mb >= 2 and n_nonmb >= 2:
    title += f'\nU={u_stat:.0f}, one-sided p={p_one:.4f}, r_rb={r_rb:+.3f}'
ax.set_title(title, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"fig_h3_log_delib_mb_vs_nonmb_{OUT_PREFIX}.png", dpi=200,
            bbox_inches='tight', facecolor='#FAFBFC')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1B.  CORRELATIONS — every metric × {somatic_anxiety, worry}
#       Spearman ρ PRIMARY; Pearson r reported alongside
# ══════════════════════════════════════════════════════════════════════

def _spearman_table(rt_df, metric_cols, anx_col, group_col='mb_group'):
    '''One row per (metric × group) with Spearman ρ + Pearson r vs `anx_col`.'''
    rows = []
    for col in metric_cols:
        for grp in [None, 'non-MB', 'MB']:
            g = rt_df if grp is None else rt_df[rt_df[group_col] == grp]
            g = g.dropna(subset=[anx_col, col])
            grp_label = 'All' if grp is None else grp
            if len(g) < 5: continue
            rs, ps = spearmanr (g[anx_col], g[col])
            rp, pp = pearsonr  (g[anx_col], g[col])
            rows.append({'anx': anx_col, 'metric': col, 'group': grp_label,
                         'n': len(g),
                         'spearman_rho': rs, 'spearman_p': ps,    # PRIMARY
                         'pearson_r':    rp, 'pearson_p':    pp})
    return pd.DataFrame(rows)

# Single OT definition; delib_s and acc_canon only
RT_COLS  = [f'delib_log_{c}' for c in ['all','feather','triangle']]
ACC_COLS = [f'acc_canon_{c}' for c in ['all','feather','triangle']]
OT_COLS  = [f'OT_{c}'        for c in ['all','feather','triangle']]

rt_corr_all  = []
acc_corr_all = []
ot_corr_all  = []
for anx_col in ['somatic_anxiety','worry']:
    print(f"\n\n========================  {anx_col.upper()}  ========================")
    print(f"\n=== Delib RT × {anx_col}, per condition × subgroup ===")
    rt_corr  = _spearman_table(rt_per_subj, RT_COLS,  anx_col)
    print(rt_corr.round(3).to_string(index=False))
    rt_corr_all.append(rt_corr)

    print(f"\n=== Canonical accuracy × {anx_col}, per condition × subgroup ===")
    acc_corr = _spearman_table(rt_per_subj, ACC_COLS, anx_col)
    print(acc_corr.round(3).to_string(index=False))
    acc_corr_all.append(acc_corr)

    print(f"\n=== OT × {anx_col}, per condition × subgroup ===")
    ot_corr  = _spearman_table(rt_per_subj, OT_COLS,  anx_col)
    print(ot_corr.round(3).to_string(index=False))
    ot_corr_all.append(ot_corr)

pd.concat(rt_corr_all,  ignore_index=True).to_csv("rt_analyses_corr_rt.csv",  index=False)
pd.concat(acc_corr_all, ignore_index=True).to_csv("rt_analyses_corr_acc.csv", index=False)
pd.concat(ot_corr_all,  ignore_index=True).to_csv("rt_analyses_corr_ot.csv", index=False)
print("\nSaved rt_analyses_corr_{rt,acc,ot}.csv (each table contains BOTH somatic_anxiety and worry rows)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1D.  PLOT GRIDS — single OT, run for both somatic_anxiety AND worry.
#       3 × 3 grid: rows = delib RT / accuracy / OT,  cols = all / F / T.
# ══════════════════════════════════════════════════════════════════════
BG_COLOR = "#FAFBFC"
MB_COLORS = {"non-MB": "#3498DB", "MB": "#2ECC71"}

def _scatter(ax, rt_df, anx_col, ycol, label, axis_label, draw_zero=False):
    ax.set_facecolor(BG_COLOR)
    g = rt_df.dropna(subset=[anx_col, ycol])
    if len(g) < 5:
        ax.set_title(f"{label}: n={len(g)}"); return
    for grp, color in MB_COLORS.items():
        gg = g[g['mb_group']==grp]
        if len(gg) < 3: continue
        ax.scatter(gg[anx_col], gg[ycol], alpha=.6, s=28,
                   color=color, edgecolors='white', linewidths=0.4)
        m_g, b_g = np.polyfit(gg[anx_col], gg[ycol], 1)
        xr_g = np.linspace(gg[anx_col].min(), gg[anx_col].max(), 50)
        rs_g, ps_g = spearmanr(gg[anx_col], gg[ycol])
        star_g = '***' if ps_g<.001 else '**' if ps_g<.01 else '*' if ps_g<.05 else ''
        ax.plot(xr_g, m_g*xr_g+b_g, color=color, lw=1.6,
                label=f"{grp} rho={rs_g:+.2f}{star_g}")
    m, b = np.polyfit(g[anx_col], g[ycol], 1)
    xr = np.linspace(g[anx_col].min(), g[anx_col].max(), 50)
    ax.plot(xr, m*xr+b, color='black', lw=1.2, ls=':', label='pooled')
    rs, ps = spearmanr(g[anx_col], g[ycol])
    star = '***' if ps<.001 else '**' if ps<.01 else '*' if ps<.05 else ''
    if draw_zero:
        ax.axhline(0, color='k', lw=.4)
    ax.set_xlabel(anx_col); ax.set_ylabel(axis_label)
    ax.set_title(f"{label}\nAll: rho={rs:+.3f}, p={ps:.3g} {star}, n={len(g)}",
                 fontweight='bold', fontsize=9)
    ax.legend(fontsize=7); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

CONDS = [('all','all trials'), ('feather','Feather only'), ('triangle','Triangle only')]
ROWS = [('delib_log',   'delib_log = mean(log(delib_t))', False),
        ('acc_canon',   'canonical accuracy (ties=0.5)',  False),
        ('OT',          'OT = z(delib_log) - z(logit acc)', True)]

for anx_col in ['somatic_anxiety','worry']:
    fig, axes = plt.subplots(3, 3, figsize=(15, 13), facecolor=BG_COLOR)
    for r, (prefix, axis_label, draw0) in enumerate(ROWS):
        for c, (cond_key, cond_lab) in enumerate(CONDS):
            col = f'{prefix}_{cond_key}'
            _scatter(axes[r,c], rt_per_subj, anx_col, col,
                     f'{axis_label} ({cond_lab})', axis_label,
                     draw_zero=draw0)
    fig.suptitle(f"{anx_col} × {{delib RT, canonical acc, OT}}   "
                 f"(rows=metric, cols=trial subset; colour=MB-group)",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = f"fig_section1_grid_{anx_col}.png"
    plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print(f"  Saved {out}")

### 5.1 — Somatic vs worry: direct questionnaire correlation

The whole dissociation argument assumes the two anxiety scales are related but not redundant.
Report it directly: Spearman + Pearson with a 95% CI, and the shared variance.

In [ ]:
# ═══ 5.1  Somatic–worry correlation (direct readout) ═══════════════════════
from scipy.stats import spearmanr, pearsonr
_g = rt_per_subj[['somatic_anxiety','worry']].dropna()
_rs,_ps = spearmanr(_g['somatic_anxiety'], _g['worry'])
_rp,_pp = pearsonr(_g['somatic_anxiety'], _g['worry'])
_z = np.arctanh(_rp); _se = 1.0/np.sqrt(len(_g)-3)
_lo,_hi = np.tanh(_z-1.96*_se), np.tanh(_z+1.96*_se)
print(f"Somatic (MASQ-AA) vs Worry (PSWQ-A)   N={len(_g)}")
print(f"  Spearman rho = {_rs:+.3f}   (p={_ps:.2e})")
print(f"  Pearson  r   = {_rp:+.3f}   95% CI [{_lo:+.2f}, {_hi:+.2f}]   (p={_pp:.2e})")
print(f"  shared variance  r^2 = {_rp**2:.2f}   "
      f"({100*(1-_rp**2):.0f}% of variance is NOT shared)")
print("  => the two scales are correlated but largely distinct, so a somatic-vs-worry")
print("     dissociation in downstream effects is interpretable (not the same construct).")

### 5.3 — The dissociation, explicit: time rises, accuracy flat (item 3)

`OT = z(delib_log) - z(logit acc)`. Split it back into its two halves and show, **within MB**,
that anxiety tracks the **deliberation-time** half but **not** the **accuracy** half — the
"slower but no more accurate" signature of overthinking. Two readouts:
1. Spearman of each anxiety with the time half and the accuracy half, MB vs non-MB.
2. A median-split contrast inside MB: high- vs low-anxiety subjects' median RT and mean accuracy.

If the story holds: time-half rho **> 0** and significant, accuracy-half rho **≈ 0**; and high-anxiety
MB subjects are **slower** (one-sided) with **statistically indistinguishable accuracy**.

In [ ]:
# ═══ 5.3  Decompose OT: deliberation TIME vs ACCURACY, within MB ════════════
from scipy.stats import spearmanr, mannwhitneyu
mb_  = rt_per_subj[rt_per_subj['mb_group']=='MB'].copy()
non_ = rt_per_subj[rt_per_subj['mb_group']=='non-MB'].copy()

print("Spearman of anxiety with each OT half (delib-time half vs accuracy half):\n")
_rows53=[]
for grp,dd in [('MB',mb_),('non-MB',non_)]:
    for anx in ['worry','somatic_anxiety']:
        g = dd.dropna(subset=[anx,'delib_log_all','logit_acc_canon_all'])
        rt,pt = spearmanr(g[anx], g['delib_log_all'])        # TIME half
        ra,pa = spearmanr(g[anx], g['logit_acc_canon_all'])  # ACCURACY half
        pt1 = pt/2 if rt>0 else 1-pt/2
        print(f"  [{grp:6s}] {anx:16s}: delib-time rho={rt:+.3f} (1s p={pt1:.3f})   "
              f"accuracy rho={ra:+.3f} (2s p={pa:.3f})   n={len(g)}")
        _rows53.append(dict(group=grp,anx=anx,rho_time=rt,p_time_1s=pt1,
                            rho_acc=ra,p_acc_2s=pa,n=len(g)))
pd.DataFrame(_rows53).to_csv("rt_analyses_OT_decomposition.csv",index=False)

print("\nMB only — high vs low anxiety (median split): RT rises, accuracy holds?")
for anx,lab in [('worry','worry'),('somatic_anxiety','somatic')]:
    g  = mb_.dropna(subset=[anx,'median_rt_s_all','acc_canon_all']).copy()
    hi = g[anx] > g[anx].median()
    rt_hi,rt_lo = g.loc[hi,'median_rt_s_all'], g.loc[~hi,'median_rt_s_all']
    ac_hi,ac_lo = g.loc[hi,'acc_canon_all'],   g.loc[~hi,'acc_canon_all']
    _,p_rt = mannwhitneyu(rt_hi,rt_lo,alternative='greater')      # slower?
    _,p_ac = mannwhitneyu(ac_hi,ac_lo,alternative='two-sided')    # accuracy differs?
    print(f"  high {lab:8s}: median RT {rt_hi.median():.2f}s vs {rt_lo.median():.2f}s "
          f"(slower 1s p={p_rt:.3f}) | acc {ac_hi.mean():.3f} vs {ac_lo.mean():.3f} "
          f"(differ 2s p={p_ac:.3f})")

# Plot: MB — worry vs delib-time (left) and vs accuracy (right)
BG="#FAFBFC"; fig,axes=plt.subplots(1,2,figsize=(12,5),facecolor=BG)
for ax,(half,lab,ylab) in zip(
        axes,[('delib_log_all','deliberation time','delib_log (motor-corrected)'),
               ('logit_acc_canon_all','accuracy','logit(canonical accuracy)')]):
    ax.set_facecolor(BG)
    g = mb_.dropna(subset=['worry',half])
    ax.scatter(g['worry'],g[half],alpha=.5,s=30,color='#2ECC71',
               edgecolors='white',linewidths=.5)
    _m,_b = np.polyfit(g['worry'],g[half],1)
    xr=np.linspace(g['worry'].min(),g['worry'].max(),30)
    ax.plot(xr,_m*xr+_b,color='#157a3a',lw=2)
    r,p = spearmanr(g['worry'],g[half])
    ax.set_title(f"MB: worry vs {lab}\nrho={r:+.2f}, p={p:.3f}",
                 fontweight='bold',fontsize=10)
    ax.set_xlabel("worry (PSWQ-A)"); ax.set_ylabel(ylab)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.suptitle("Dissociation in MB subjects: deliberation time rises with worry, accuracy does not",
             fontsize=12,fontweight='bold')
plt.savefig("fig_section5_OT_dissociation.png",dpi=180,bbox_inches='tight',facecolor=BG)
plt.show()

---

# Section 3 · Modeling (heavy) + CBM confirmation

CBM/HBI fitting, then H1/H2 re-run under the CBM split to confirm, κ, recovery, RT-LMM, DDMs. **Re-running needs the `cbm` package / PyMC.**

---
## 8 · RL Model Fitting (CBM: Hierarchical Bayesian)

Using [Piray et al. (2019)](https://github.com/payampiray/cbm_python) — hierarchical Bayesian inference with Laplace approximation for concurrent model fitting and comparison.

### Choice-only models (anti-MF trials, predict chose_mb)

| # | Model | Parameters |
|---|-------|-----------|
| 1 | MF-only (fixed MF values) | beta |
| 2 | MF + learning rate | beta, alpha |
| 3 | MB-only | beta |
| 4 | MB + MF (weighted) | beta, w |
| 5 | MB separate beta per stimulus | beta_f, beta_t |
| 6 | MB separate beta per stimulus + MF | beta_f, beta_t, w |
| 7 | MB separate beta per revaluation | beta_q1, beta_q2 |
| 8 | MB separate beta per reval + MF | beta_q1, beta_q2, w |
| 9 | Separate w per stimulus | beta, w_f, w_t |
| 10 | Separate w per revaluation | beta, w_q1, w_q2 |
| 11 | Separate w per stimulus × reval | beta, w_f1, w_f2, w_t1, w_t2 |

### Cost-of-time models (joint choice + RT as deliberation cost)
RT modeled as: `log(RT) = c0 + gamma * |EV - offer| + noise`
Choice: `P(MB) = sigmoid(beta * (V - offer) - kappa * log(RT))`

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8A  MODEL DEFINITIONS FOR CBM
# ══════════════════════════════════════════════════════════════════════
# CBM models: f(parameters, data) -> log_likelihood (positive)
# Parameters in UNCONSTRAINED space (apply sigmoid/exp inside)

import pickle, os
from cbm import individual_fit, bms, hbi_main, hbi_null

CBM_DIR = f"cbm_output_{OUT_PREFIX}"
os.makedirs(CBM_DIR, exist_ok=True)

# ── Prepare per-subject data as list of dicts ──
# FIX (a): raw gamble/safe target. Filter depends on FIT_ALL_TRIALS.
if FIT_ALL_TRIALS:
    fit_df = trial_df.dropna(subset=["choice"]).copy()
else:
    fit_df = trial_df[trial_df["is_anti_mf"] == 1].dropna(subset=["choice"]).copy()
fit_df["is_feather"] = (fit_df["state"] == "Feather").astype(int)
fit_df["is_q1"] = (fit_df["query_num"] == 1).astype(int)

subject_ids = fit_df["subject_id"].unique()
N_SUBJ = len(subject_ids)

data_list = []
for sid in subject_ids:
    sdf = fit_df[fit_df["subject_id"] == sid]
    data_list.append({
        "ev":         sdf["ev"].values.astype(float),
        "mf_value":   sdf["mf_value"].values.astype(float),
        "offer":      sdf["offer"].values.astype(float),
        "choice":     sdf["choice"].values.astype(float),  # FIX (a): raw gamble/safe action (0/1), not chose_mb
        "is_feather": sdf["is_feather"].values.astype(float),
        "is_q1":      sdf["is_q1"].values.astype(float),
        "log_rt":     sdf["log_rt"].values.astype(float),
    })

print(f"CBM data prepared: {N_SUBJ} subjects, {len(fit_df)} trials ({FILTER_LABEL}; target = raw choice; dir={CBM_DIR})")

# ── Helper: Bernoulli log-likelihood ──
def _ll(p_gamble, choice):
    p = np.clip(p_gamble, 1e-8, 1 - 1e-8)
    return np.sum(choice * np.log(p) + (1 - choice) * np.log(1 - p))

# ══════════════════════════════════════════════════════════════════════
# CHOICE-ONLY MODELS (return log-likelihood)
# ══════════════════════════════════════════════════════════════════════

def m1_mf_only(params, d):
    beta = params[0]
    p = expit(beta * (d["mf_value"] - d["offer"]))
    return _ll(p, d["choice"])

def m2_mf_alpha(params, d):
    beta, alpha_raw = params[0], params[1]
    alpha = expit(alpha_raw)
    p = expit(beta * (alpha * d["mf_value"] - d["offer"]))
    return _ll(p, d["choice"])

def m3_mb_only(params, d):
    beta = params[0]
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m4_mb_mf(params, d):
    beta, w_raw = params[0], params[1]
    w = expit(w_raw)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m5_mb_sep_stim(params, d):
    beta_f, beta_t = params[0], params[1]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m7_mb_sep_reval(params, d):
    beta_q1, beta_q2 = params[0], params[1]
    beta = np.where(d["is_q1"], beta_q1, beta_q2)
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m9_sep_w_stim(params, d):
    beta, wf_raw, wt_raw = params[0], params[1], params[2]
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])
def m12_mb_mf_sep_stim(params, d):
    beta_f, beta_t,w = params[0], params[1],params[0]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m16_sep_beta_w_stim(params, d):
    """M16 — per-stimulus beta AND per-stimulus MB/MF mixing weight.
    Superset of M9 (beta shared) and M15 (w shared).
    v = w_stim * ev + (1 - w_stim) * mf_value  (= r_mid + w_stim * T in this task)
    P(MB) = sigmoid(beta_stim * (v - offer))
    """
    beta_f, beta_t, wf_raw, wt_raw = params[0], params[1], params[2], params[3]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    w    = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v    = w * d["ev"] + (1 - w) * d["mf_value"]
    p    = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m10_sep_w_reval(params, d):
    beta, w1_raw, w2_raw = params[0], params[1], params[2]
    w = np.where(d["is_q1"], expit(w1_raw), expit(w2_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m11_sep_beta_w_reval(params, d):
    
    beta_f, beta_t, w = params[0], params[1], params[2]
    beta = np.where(d["is_q1"], beta_f, beta_t)
    v    = w * d["ev"] + (1 - w) * d["mf_value"]
    p    = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m11_sep_w_stim_reval(params, d):
    beta = params[0]
    wf1, wf2 = expit(params[1]), expit(params[2])
    wt1, wt2 = expit(params[3]), expit(params[4])
    w = np.where(d["is_feather"],
                 np.where(d["is_q1"], wf1, wf2),
                 np.where(d["is_q1"], wt1, wt2))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

# ══════════════════════════════════════════════════════════════════════
# COST-OF-TIME MODELS (joint choice + RT)
# ══════════════════════════════════════════════════════════════════════

def _rt_ll(log_rt, ev, offer, c0, gamma, log_sigma):
    """Log-likelihood of RT component."""
    valid = ~np.isnan(log_rt)
    if valid.sum() == 0:
        return 0.0
    sigma = max(np.exp(log_sigma), 0.01)
    mu = c0 + gamma * np.abs(ev[valid] - offer[valid])
    return (-0.5 * np.sum(((log_rt[valid] - mu) / sigma)**2)
            - valid.sum() * np.log(sigma)
            - 0.5 * valid.sum() * np.log(2 * np.pi))

def _get_rt(log_rt, ev, offer, c0, gamma):
    out = log_rt.copy()
    m = np.isnan(out)
    if m.any():
        out[m] = c0 + gamma * np.abs(ev[m] - offer[m])
    return out

def ct_m3(params, d):
    beta, c0, gamma, kappa, log_sigma = params
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (d["ev"] - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m4(params, d):
    beta, w_raw, c0, gamma, kappa, log_sigma = params
    w = expit(w_raw)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m9(params, d):
    beta, wf_raw, wt_raw, c0, gamma, kappa, log_sigma = params
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m11(params, d):
    beta = params[0]
    wf1, wf2, wt1, wt2 = expit(params[1]), expit(params[2]), expit(params[3]), expit(params[4])
    c0, gamma, kappa, log_sigma = params[5], params[6], params[7], params[8]
    w = np.where(d["is_feather"],
                 np.where(d["is_q1"], wf1, wf2),
                 np.where(d["is_q1"], wt1, wt2))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

# ══════════════════════════════════════════════════════════════════════
# MODEL REGISTRY
# ══════════════════════════════════════════════════════════════════════

MODELS = {
    "M1_MF":            {"func": m1_mf_only,        "k": 1, "names": ["beta"]},
    "M2_MF_alpha":      {"func": m2_mf_alpha,       "k": 2, "names": ["beta","alpha_raw"]},
    "M3_MB":            {"func": m3_mb_only,        "k": 1, "names": ["beta"]},
    "M4_MB_MF":         {"func": m4_mb_mf,          "k": 2, "names": ["beta","w_raw"]},
    "M5_MB_stim":       {"func": m5_mb_sep_stim,    "k": 2, "names": ["beta_f","beta_t"]},
    "M7_MB_reval":      {"func": m7_mb_sep_reval,   "k": 2, "names": ["beta_q1","beta_q2"]},
    "M9_w_stim":        {"func": m9_sep_w_stim,     "k": 3, "names": ["beta","w_f_raw","w_t_raw"]},
    "M10_w_reval":      {"func": m10_sep_w_reval,   "k": 3, "names": ["beta","w_q1_raw","w_q2_raw"]},
    "m12_mb_mf_sep_stim":{"func": m12_mb_mf_sep_stim,   "k": 3, "names":["beta_f","beta_t",'w']},
    "m11_sep_beta_w_reval":{"func": m11_sep_beta_w_reval,   "k": 3, "names":["beta_f","beta_t",'w']},
    "m16_sep_beta_w_stim":{"func":m16_sep_beta_w_stim,"k":4,"names":["beta_f", "beta_t", "wf_raw", "wt_raw"]}
}

CT_MODELS = {
    "CT_M3":            {"func": ct_m3,  "k": 5, "names": ["beta","c0","gamma","kappa","log_sigma"]},
    # "CT_M4":            {"func": ct_m4,  "k": 6, "names": ["beta","w_raw","c0","gamma","kappa","log_sigma"]}
    "CT_M9":            {"func": ct_m9,  "k": 7, "names": ["beta","w_f","w_t","c0","gamma","kappa","log_sigma"]},
    # "CT_M11":           {"func": ct_m11, "k": 9, "names": ["beta","wf1","wf2","wt1","wt2","c0","gamma","kappa","log_sigma"]},
}






In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8B  CBM INDIVIDUAL FITS (MAP via Laplace approximation)
# ══════════════════════════════════════════════════════════════════════

PRIOR_VARIANCE = 6.25  # default prior variance (sd=2.5)

cbm_fits = {}

# ── Choice-only models ──
print("=" * 70)
print("FITTING CHOICE-ONLY MODELS (CBM individual_fit)")
print("=" * 70)

for mname, mspec in MODELS.items():
    k = mspec["k"]
    prior_mean = np.zeros(k)
    fname = os.path.join(CBM_DIR, f"{mname}.pkl")
    print(f"\nFitting {mname} (k={k})...")
    cbm_result = individual_fit(
        data_list, mspec["func"], prior_mean, PRIOR_VARIANCE,
        fname=fname, config={"verbose": False}
    )
    cbm_fits[mname] = cbm_result
    lme = cbm_result.output.log_evidence
    params = cbm_result.output.parameters
    print(f"  Mean log-evidence = {lme.mean():.2f}")
    for pi, pn in enumerate(mspec["names"]):
        vals = params[:, pi]
        print(f"  {pn:15s}: M={vals.mean():.3f}, SD={vals.std():.3f}")

# ── Cost-of-time models: FAST MLE (replaces slow CBM hierarchical) ──
print("\n" + "=" * 70)
print("FITTING COST-OF-TIME MODELS (Fast MLE)")
print("=" * 70)

# Filter data_list to subjects with sufficient RT
ct_data_list = []
ct_subject_mask = []
for i, d in enumerate(data_list):
    valid_rt = np.sum(~np.isnan(d["log_rt"]))
    if valid_rt >= 5:
        ct_data_list.append(d)
        ct_subject_mask.append(i)
ct_subject_mask = np.array(ct_subject_mask)
print(f"Cost-of-time: {len(ct_data_list)} subjects with >= 5 valid RT trials")

import time as _time

def _fit_ct_mle(ct_func, data_dict, k, param_names, n_starts=10):
    """Fast MLE fit for a single subject's cost-of-time model."""
    best_nll, best_params = np.inf, None
    c0_idx = param_names.index("c0") if "c0" in param_names else None
    for _ in range(n_starts):
        x0 = np.random.randn(k) * 0.5
        if c0_idx is not None:
            x0[c0_idx] = 8.0 + np.random.randn() * 0.3
        try:
            res = minimize(lambda p, d: -ct_func(p, d), x0, args=(data_dict,),
                          method='L-BFGS-B', options={"maxiter": 200})
            if res.fun < best_nll:
                best_nll, best_params = res.fun, res.x
        except Exception:
            continue
    return best_params, best_nll

ct_mle_results = {}
for mname, mspec in CT_MODELS.items():
    k = mspec["k"]
    pnames = mspec["names"]
    print(f"\nFitting {mname} (k={k}) via MLE...")
    t0 = _time.time()
    subj_params = np.zeros((len(ct_data_list), k))
    subj_nll = np.zeros(len(ct_data_list))
    for si, d in enumerate(ct_data_list):
        params, nll = _fit_ct_mle(mspec["func"], d, k, pnames)
        if params is not None:
            subj_params[si] = params
            subj_nll[si] = nll
        else:
            subj_nll[si] = np.inf
    n_trials_per = np.array([len(d["choice"]) for d in ct_data_list])
    subj_bic = k * np.log(n_trials_per) + 2 * subj_nll
    subj_aic = 2 * k + 2 * subj_nll
    ct_mle_results[mname] = {
        "params": subj_params,
        "nll": subj_nll,
        "bic": subj_bic,
        "aic": subj_aic,
        "k": k,
        "names": pnames,
    }
    elapsed = _time.time() - t0
    print(f"  Mean NLL = {subj_nll[subj_nll < np.inf].mean():.2f}  ({elapsed:.1f}s)")

print("\nAll CT models fitted via MLE.")

In [ ]:
## ══════════════════════════════════════════════════════════════════════
# 8C  BAYESIAN MODEL SELECTION
# ══════════════════════════════════════════════════════════════════════

# ── Choice-only BMS ──
model_names_choice = list(MODELS.keys())
lme_matrix = np.column_stack([cbm_fits[m].output.log_evidence for m in model_names_choice])

bms_result = bms(lme_matrix)

print("=" * 70)
print("BAYESIAN MODEL SELECTION: Choice-Only Models")
print("=" * 70)
print(f"{"Model":25s} {"Freq":>8s} {"ExcP":>8s} {"PXP":>8s} {"Mean LME":>10s}")
print("-" * 63)
for i, mname in enumerate(model_names_choice):
    print(f"{mname:25s} {bms_result.model_frequency[i]:8.3f} "
          f"{bms_result.exceedance_prob[i]:8.3f} "
          f"{bms_result.protected_exceedance_prob[i]:8.3f} "
          f"{lme_matrix[:, i].mean():10.2f}")
print(f"\nBayes Omnibus Risk: {bms_result.bor:.4f}")

best_choice_model = model_names_choice[np.argmax(bms_result.protected_exceedance_prob)]
print(f"Best choice-only model (highest PXP): {best_choice_model}")

# ── Cost-of-time: BIC-based model comparison (from MLE fits) ──
ct_model_names = list(CT_MODELS.keys())
print("\n" + "=" * 70)
print("COST-OF-TIME MODEL COMPARISON (MLE + BIC)")
print("=" * 70)
print(f"{'Model':25s} {'Mean BIC':>10s} {'Mean AIC':>10s} {'Mean NLL':>10s}")
print("-" * 58)
ct_bic_matrix = np.column_stack([ct_mle_results[m]["bic"] for m in ct_model_names])
ct_aic_matrix = np.column_stack([ct_mle_results[m]["aic"] for m in ct_model_names])
for i, mname in enumerate(ct_model_names):
    r = ct_mle_results[mname]
    valid = r["nll"] < np.inf
    print(f"{mname:25s} {r['bic'][valid].mean():10.2f} {r['aic'][valid].mean():10.2f} {r['nll'][valid].mean():10.2f}")

# Per-subject best CT model (by BIC)
ct_best_idx = np.argmin(ct_bic_matrix, axis=1)
ct_best_names = [ct_model_names[i] for i in ct_best_idx]
best_ct_model = max(set(ct_best_names), key=ct_best_names.count)
print(f"\nBest CT model (most frequent BIC winner): {best_ct_model}")
from collections import Counter as _Counter
for mname, cnt in _Counter(ct_best_names).most_common():
    print(f"  {mname:25s}: {cnt} subjects ({100*cnt/len(ct_best_names):.1f}%)")

# ── Bar plot: PXP ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(range(len(model_names_choice)), bms_result.protected_exceedance_prob,
         color=["C2" if m == best_choice_model else "C0" for m in model_names_choice], alpha=0.8)
ax1.set_yticks(range(len(model_names_choice)))
ax1.set_yticklabels(model_names_choice, fontsize=9)
ax1.set_xlabel("Protected Exceedance Probability")
ax1.set_title("Choice-Only Models", fontweight="bold")
ax1.invert_yaxis()
ax1.axvline(1/len(model_names_choice), color="grey", ls="--", alpha=0.5, label="chance")
ax1.legend(fontsize=8)

# CT model comparison: show proportion of subjects where each model wins
ct_win_counts = np.array([ct_best_names.count(m) for m in ct_model_names])
ct_win_freq = ct_win_counts / ct_win_counts.sum()
ax2.barh(range(len(ct_model_names)), ct_win_freq,
         color=["C2" if m == best_ct_model else "C1" for m in ct_model_names], alpha=0.8)
ax2.set_yticks(range(len(ct_model_names)))
ax2.set_yticklabels(ct_model_names, fontsize=9)
ax2.set_xlabel("Proportion Best (BIC)")
ax2.set_title("Cost-of-Time Models (MLE)", fontweight="bold")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(f"fig_bms_pxp_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8D  HIERARCHICAL BAYESIAN INFERENCE (HBI)
# ══════════════════════════════════════════════════════════════════════
# HBI jointly refines parameters and model assignments across subjects

# ── Choice-only HBI ──
choice_model_funcs = [MODELS[m]["func"] for m in model_names_choice]
choice_cbm_paths   = [os.path.join(CBM_DIR, f"{m}.pkl") for m in model_names_choice]

print("Running HBI on choice-only models...")
hbi_result = hbi_main(
    data_list, choice_model_funcs, choice_cbm_paths,
    fname=os.path.join(CBM_DIR, "hbi_choice.pkl"),
    config={"verbose": 1, "save_prog": False}
)

# Protected exceedance
hbi_result = hbi_null(data_list, os.path.join(CBM_DIR, "hbi_choice.pkl"))

print("\n" + "=" * 70)
print("HBI RESULTS: Choice-Only Models")
print("=" * 70)
print(f"{"Model":25s} {"Freq":>8s} {"PXP":>8s}")
for i, mname in enumerate(model_names_choice):
    print(f"{mname:25s} {hbi_result.output.model_frequency[i]:8.3f} "
          f"{hbi_result.output.protected_exceedance_prob[i]:8.3f}")

# ── Per-subject winning model ──
responsibility = hbi_result.output.responsibility  # N x K
winning_model_idx = np.argmax(responsibility, axis=1)
winning_model_names = [model_names_choice[i] for i in winning_model_idx]

print(f"\nPer-subject winning model distribution:")
from collections import Counter
for mname, count in Counter(winning_model_names).most_common():
    print(f"  {mname:25s}: {count} subjects ({100*count/N_SUBJ:.1f}%)")

# ── Store per-subject winning params ──
# HBI returns refined parameters per model: output.parameters[k] is (N, d_k)
winning_params = {}
for i, sid in enumerate(subject_ids):
    k = winning_model_idx[i]
    mname = model_names_choice[k]
    params = hbi_result.output.parameters[k][i, :]
    winning_params[sid] = {
        "model": mname,
        "params": params,
        "param_names": MODELS[mname]["names"],
        "responsibility": responsibility[i, :],
    }

print(f"\nStored winning model + params for {len(winning_params)} subjects")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8E  POSTERIOR PREDICTIVE CHECKS
# ══════════════════════════════════════════════════════════════════════
# Simulate choices (and RT for CT models) from each subject's winning
# model + HBI-refined params, then compare model-agnostic patterns
# between real and simulated data.

N_SIM = 50  # simulations per subject
np.random.seed(42)

# ── Simulate from winning choice-only models ──
sim_rows = []
for subj_i, sid in enumerate(subject_ids):
    d = data_list[subj_i]
    info = winning_params[sid]
    mname = info["model"]
    params = info["params"]
    func = MODELS[mname]["func"]
    n_trials = len(d["choice"])

    # Get P(chose_MB) per trial by computing model predictions
    # For each trial, compute log-lik at choice=1 and choice=0
    p_mb = np.zeros(n_trials)
    for t in range(n_trials):
        d1 = {k: np.array([v[t]]) for k, v in d.items()}
        d1["choice"] = np.array([1.0])
        ll1 = func(params, d1)
        d1["choice"] = np.array([0.0])
        ll0 = func(params, d1)
        # ll1 = log(p), ll0 = log(1-p) for Bernoulli
        p_mb[t] = np.exp(ll1)

    for sim in range(N_SIM):
        sim_choices = (np.random.rand(n_trials) < p_mb).astype(float)
        for t in range(n_trials):
            sim_rows.append({
                "subject_id": sid,
                "sim": sim,
                "ev": d["ev"][t],
                "mf_value": d["mf_value"][t],
                "offer": d["offer"][t],
                "is_feather": d["is_feather"][t],
                "is_q1": d["is_q1"][t],
                "abs_ev_diff": np.abs(d["ev"][t] - d["offer"][t]),
                "chose_mb_sim": sim_choices[t],
                "chose_mb_real": d["choice"][t],
                "log_rt_real": d["log_rt"][t],
                "somatic_anxiety": fit_df[fit_df["subject_id"]==sid]["somatic_anxiety"].iloc[0],
            })

sim_df = pd.DataFrame(sim_rows)
median_diff = sim_df["abs_ev_diff"].median()
sim_df["difficulty"] = np.where(sim_df["abs_ev_diff"] <= median_diff, "Hard", "Easy")
median_anx = sim_df.groupby("subject_id")["somatic_anxiety"].first().median()
sim_df["anx_group"] = sim_df["subject_id"].map(
    sim_df.groupby("subject_id")["somatic_anxiety"].first().apply(
        lambda a: "High" if a >= median_anx else "Low"))

print(f"Posterior predictive simulations: {len(sim_df)} rows ({N_SIM} sims x {len(fit_df)} trials)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8F  POSTERIOR PREDICTIVE CHECK PLOTS
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# ── Helper: compute pattern for real vs simulated data ──
def pattern_by_group(df, groupcols, real_col, sim_col):
    """Compute mean real and mean simulated across group levels."""
    real = df.groupby(groupcols)[real_col].mean().reset_index()
    real.columns = list(groupcols) + ["real"]
    sim = df.groupby(groupcols + ["sim"])[sim_col].mean().reset_index()
    sim_agg = sim.groupby(groupcols)[sim_col].agg(["mean","std"]).reset_index()
    sim_agg.columns = list(groupcols) + ["sim_mean","sim_std"]
    return real.merge(sim_agg, on=list(groupcols))

# ══════════════════════════════════════════════════════════════════════
# PPC 1: P(chose MB) by Difficulty
# ══════════════════════════════════════════════════════════════════════
ppc1 = pattern_by_group(sim_df, ["difficulty"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 2: P(chose MB) by Stimulus (feather vs triangle)
# ══════════════════════════════════════════════════════════════════════
sim_df["stimulus"] = np.where(sim_df["is_feather"]==1, "Feather", "Triangle")
ppc2 = pattern_by_group(sim_df, ["stimulus"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 3: P(chose MB) by Query (reval 1 vs 2)
# ══════════════════════════════════════════════════════════════════════
sim_df["query"] = np.where(sim_df["is_q1"]==1, "Query 1", "Query 2")
ppc3 = pattern_by_group(sim_df, ["query"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 4: P(chose MB) by Difficulty × Somatic Anxiety (the key 2x2)
# ══════════════════════════════════════════════════════════════════════
ppc4 = pattern_by_group(sim_df, ["difficulty","anx_group"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 5: P(chose MB) by Difficulty × Stimulus
# ══════════════════════════════════════════════════════════════════════
ppc5 = pattern_by_group(sim_df, ["difficulty","stimulus"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PLOT: 2×3 grid of posterior predictive checks
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=BG_COLOR)

def ppc_bar(ax, ppc_df, group_col, title):
    """Bar plot: real (solid) vs simulated (hatched) for each group level."""
    levels = ppc_df[group_col].values
    x = np.arange(len(levels))
    w = 0.35
    ax.bar(x - w/2, ppc_df["real"], w, color="C0", alpha=0.8, label="Real data")
    ax.bar(x + w/2, ppc_df["sim_mean"], w, color="C3", alpha=0.6,
           yerr=ppc_df["sim_std"], capsize=4, label="Simulated (model)")
    ax.set_xticks(x)
    ax.set_xticklabels(levels)
    ax.set_ylabel("P(Chose MB)")
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
    ax.set_facecolor(BG_COLOR)

# Row 1: simple patterns
ppc_bar(axes[0,0], ppc1, "difficulty", "PPC 1: Difficulty")
ppc_bar(axes[0,1], ppc2, "stimulus", "PPC 2: Stimulus")
ppc_bar(axes[0,2], ppc3, "query", "PPC 3: Query")

# Row 2: interaction patterns
# PPC 4: Difficulty × Somatic Anxiety
ax = axes[1,0]; ax.set_facecolor(BG_COLOR)
for anx_i, anx in enumerate(["Low", "High"]):
    sub = ppc4[ppc4["anx_group"] == anx]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + anx_i * 0.2
    color_r = "C0" if anx == "Low" else "C3"
    color_s = "C4" if anx == "Low" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{anx} ANX (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{anx} ANX (sim)")
    ax.errorbar([x+offset for x in xs], sub["sim_mean"],
               yerr=sub["sim_std"], fmt="none", color=color_s, capsize=3)
    # Connect with lines
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_ylabel("P(Chose MB)")
ax.set_title("PPC 4: Difficulty x Somatic Anxiety", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)

# PPC 5: Difficulty × Stimulus
ax = axes[1,1]; ax.set_facecolor(BG_COLOR)
for stim_i, stim in enumerate(["Feather", "Triangle"]):
    sub = ppc5[ppc5["stimulus"] == stim]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + stim_i * 0.2
    color_r = "C0" if stim == "Feather" else "C3"
    color_s = "C4" if stim == "Feather" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{stim} (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{stim} (sim)")
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_title("PPC 5: Difficulty x Stimulus", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)

# PPC 6: Subject-level scatter — real vs simulated MB rate
ax = axes[1,2]; ax.set_facecolor(BG_COLOR)
subj_real = sim_df.groupby("subject_id")["chose_mb_real"].mean()
subj_sim  = sim_df.groupby(["subject_id","sim"])["chose_mb_sim"].mean().groupby("subject_id").mean()
merged = pd.DataFrame({"real": subj_real, "sim": subj_sim}).dropna()
ax.scatter(merged["real"], merged["sim"], alpha=0.5, s=25)
ax.plot([0,1],[0,1], "k--", alpha=0.5, lw=1)
r_ppc, p_ppc = stats.pearsonr(merged["real"], merged["sim"])
ax.set_xlabel("Real P(chose MB)")
ax.set_ylabel("Simulated P(chose MB)")
ax.set_title(f"PPC 6: Subject-level (r={r_ppc:.3f})", fontweight="bold", fontsize=10)

fig.suptitle("Posterior Predictive Checks: Real vs Model-Simulated Data",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_ppc_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# Print numerical summary
print("Posterior Predictive Check Summary:")
for name, ppc_data, gcol in [("Difficulty", ppc1, "difficulty"),
                               ("Stimulus", ppc2, "stimulus"),
                               ("Query", ppc3, "query")]:
    print(f"\n  {name}:")
    for _, row in ppc_data.iterrows():
        print(f"    {row[gcol]:>10s}: real={row['real']:.3f}  sim={row['sim_mean']:.3f} +/- {row['sim_std']:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8F-ii  RT POSTERIOR PREDICTIVE CHECK (from MLE cost-of-time models)
# ══════════════════════════════════════════════════════════════════════
# Simulate RTs from the best-fit CT model for each subject, then
# compare simulated RT patterns to real RT patterns.

BG_COLOR = "#FAFBFC"
N_RT_SIM = 100
np.random.seed(42)

# ── Get best CT model params per subject ──
rt_sim_rows = []
for si, d in enumerate(ct_data_list):
    subj_idx = ct_subject_mask[si]
    sid = subject_ids[subj_idx]
    best_mi = ct_best_idx[si]
    best_mname = ct_model_names[best_mi]
    best_params = ct_mle_results[best_mname]["params"][si]
    mspec = CT_MODELS[best_mname]
    pnames = mspec["names"]

    # Extract RT params
    c0 = best_params[pnames.index("c0")]
    gamma = best_params[pnames.index("gamma")]
    log_sigma = best_params[pnames.index("log_sigma")]
    sigma_rt = max(np.exp(log_sigma), 0.01)

    # Get per-trial EV (use MB EV or blended depending on model)
    if "w_raw" in pnames:
        w_idx = [i for i, n in enumerate(pnames) if n in ("w_raw", "w_f", "w_t")]
        if "w_f" in pnames and "w_t" in pnames:
            wf = expit(best_params[pnames.index("w_f")])
            wt = expit(best_params[pnames.index("w_t")])
            w_arr = np.where(d["is_feather"], wf, wt)
        elif "w_raw" in pnames:
            w_arr = expit(best_params[pnames.index("w_raw")])
        else:
            w_arr = 1.0
        ev_eff = w_arr * d["ev"] + (1 - w_arr) * d["mf_value"]
    else:
        ev_eff = d["ev"]

    n_trials = len(d["choice"])
    valid_rt = ~np.isnan(d["log_rt"])

    for sim in range(N_RT_SIM):
        # Simulate log(RT) from the model
        mu_rt = c0 + gamma * np.abs(ev_eff - d["offer"])
        sim_log_rt = mu_rt + sigma_rt * np.random.randn(n_trials)

        for t in range(n_trials):
            if not valid_rt[t]:
                continue
            rt_sim_rows.append({
                "subject_id": sid,
                "sim": sim,
                "ev": d["ev"][t],
                "mf_value": d["mf_value"][t],
                "offer": d["offer"][t],
                "is_feather": d["is_feather"][t],
                "is_q1": d["is_q1"][t],
                "abs_ev_diff": np.abs(d["ev"][t] - d["offer"][t]),
                "log_rt_real": d["log_rt"][t],
                "log_rt_sim": sim_log_rt[t],
                "somatic_anxiety": fit_df[fit_df["subject_id"]==sid]["somatic_anxiety"].iloc[0],
            })

rt_sim_df = pd.DataFrame(rt_sim_rows)
median_diff = rt_sim_df["abs_ev_diff"].median()
rt_sim_df["difficulty"] = np.where(rt_sim_df["abs_ev_diff"] <= median_diff, "Hard", "Easy")
median_anx = rt_sim_df.groupby("subject_id")["somatic_anxiety"].first().median()
rt_sim_df["anx_group"] = rt_sim_df["subject_id"].map(
    rt_sim_df.groupby("subject_id")["somatic_anxiety"].first().apply(
        lambda a: "High" if a >= median_anx else "Low"))
rt_sim_df["stimulus"] = np.where(rt_sim_df["is_feather"]==1, "Feather", "Triangle")
rt_sim_df["query"] = np.where(rt_sim_df["is_q1"]==1, "Query 1", "Query 2")

# Convert log RT to RT in ms for interpretability
rt_sim_df["rt_real_ms"] = np.exp(rt_sim_df["log_rt_real"])
rt_sim_df["rt_sim_ms"] = np.exp(rt_sim_df["log_rt_sim"])

print(f"RT PPC: {len(rt_sim_df)} rows ({N_RT_SIM} sims x trials with valid RT)")

# ══════════════════════════════════════════════════════════════════════
# RT PPC PLOTS
# ══════════════════════════════════════════════════════════════════════

def rt_pattern(df, groupcols):
    """Compute mean real and simulated RT across group levels."""
    real = df.groupby(groupcols)["rt_real_ms"].mean().reset_index()
    real.columns = list(groupcols) + ["real"]
    sim = df.groupby(groupcols + ["sim"])["rt_sim_ms"].mean().reset_index()
    sim_agg = sim.groupby(groupcols)["rt_sim_ms"].agg(["mean","std"]).reset_index()
    sim_agg.columns = list(groupcols) + ["sim_mean","sim_std"]
    return real.merge(sim_agg, on=list(groupcols))

def rt_ppc_bar(ax, ppc_df, group_col, title):
    """Bar plot: real vs simulated RT."""
    levels = ppc_df[group_col].values
    x = np.arange(len(levels))
    w = 0.35
    ax.bar(x - w/2, ppc_df["real"], w, color="C0", alpha=0.8, label="Real RT")
    ax.bar(x + w/2, ppc_df["sim_mean"], w, color="C3", alpha=0.6,
           yerr=ppc_df["sim_std"], capsize=4, label="Simulated RT")
    ax.set_xticks(x)
    ax.set_xticklabels(levels)
    ax.set_ylabel("Mean RT (ms)")
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_facecolor(BG_COLOR)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=BG_COLOR)

# PPC-RT1: RT by difficulty
ppc_rt1 = rt_pattern(rt_sim_df, ["difficulty"])
rt_ppc_bar(axes[0,0], ppc_rt1, "difficulty", "RT-PPC 1: Difficulty")

# PPC-RT2: RT by stimulus
ppc_rt2 = rt_pattern(rt_sim_df, ["stimulus"])
rt_ppc_bar(axes[0,1], ppc_rt2, "stimulus", "RT-PPC 2: Stimulus")

# PPC-RT3: RT by query
ppc_rt3 = rt_pattern(rt_sim_df, ["query"])
rt_ppc_bar(axes[0,2], ppc_rt3, "query", "RT-PPC 3: Query")

# PPC-RT4: Difficulty x Somatic Anxiety (key 2x2)
ax = axes[1,0]; ax.set_facecolor(BG_COLOR)
ppc_rt4 = rt_pattern(rt_sim_df, ["difficulty", "anx_group"])
for anx_i, anx in enumerate(["Low", "High"]):
    sub = ppc_rt4[ppc_rt4["anx_group"] == anx]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + anx_i * 0.2
    color_r = "C0" if anx == "Low" else "C3"
    color_s = "C4" if anx == "Low" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{anx} ANX (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{anx} ANX (sim)")
    ax.errorbar([x+offset for x in xs], sub["sim_mean"],
               yerr=sub["sim_std"], fmt="none", color=color_s, capsize=3)
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_ylabel("Mean RT (ms)")
ax.set_title("RT-PPC 4: Difficulty x Somatic Anxiety", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)

# PPC-RT5: Difficulty x Stimulus
ax = axes[1,1]; ax.set_facecolor(BG_COLOR)
ppc_rt5 = rt_pattern(rt_sim_df, ["difficulty", "stimulus"])
for stim_i, stim in enumerate(["Feather", "Triangle"]):
    sub = ppc_rt5[ppc_rt5["stimulus"] == stim]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + stim_i * 0.2
    color_r = "C0" if stim == "Feather" else "C3"
    color_s = "C4" if stim == "Feather" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{stim} (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{stim} (sim)")
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_title("RT-PPC 5: Difficulty x Stimulus", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)

# PPC-RT6: Subject-level scatter (mean log RT)
ax = axes[1,2]; ax.set_facecolor(BG_COLOR)
subj_rt_real = rt_sim_df.groupby("subject_id")["log_rt_real"].mean()
subj_rt_sim = rt_sim_df.groupby(["subject_id","sim"])["log_rt_sim"].mean().groupby("subject_id").mean()
merged_rt = pd.DataFrame({"real": subj_rt_real, "sim": subj_rt_sim}).dropna()
ax.scatter(merged_rt["real"], merged_rt["sim"], alpha=0.5, s=25)
lims_rt = [min(merged_rt.min()), max(merged_rt.max())]
ax.plot(lims_rt, lims_rt, "k--", alpha=0.5, lw=1)
r_rt, p_rt = stats.pearsonr(merged_rt["real"], merged_rt["sim"])
ax.set_xlabel("Real mean log(RT)")
ax.set_ylabel("Simulated mean log(RT)")
ax.set_title(f"RT-PPC 6: Subject-level (r={r_rt:.3f})", fontweight="bold", fontsize=10)

fig.suptitle("RT Posterior Predictive Checks: Real vs Model-Simulated Reaction Times",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_ppc_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ── Numerical summary ──
print("RT Posterior Predictive Check Summary:")
for name, ppc_data, gcol in [("Difficulty", ppc_rt1, "difficulty"),
                               ("Stimulus", ppc_rt2, "stimulus"),
                               ("Query", ppc_rt3, "query")]:
    print(f"\n  {name}:")
    for _, row in ppc_data.iterrows():
        print(f"    {row[gcol]:>10s}: real={row['real']:.0f}ms  sim={row['sim_mean']:.0f}ms +/- {row['sim_std']:.0f}ms")

print(f"\n  Subject-level log(RT) correlation: r = {r_rt:.3f}, p = {p_rt:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8D-ii  MODEL RESPONSIBILITY & FREQUENCY PLOT
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# responsibility: N_subjects x K_models (from HBI)
# Each row sums to 1 — posterior P(model k | subject i)

fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=BG_COLOR,
                          gridspec_kw={"width_ratios": [1.2, 2, 1]})

# ── Panel A: Winning model frequency (bar chart) ──
ax = axes[0]; ax.set_facecolor(BG_COLOR)
from collections import Counter
win_counts = Counter(winning_model_names)
# Sort by model number
sorted_models = sorted(win_counts.keys(), key=lambda m: list(MODELS.keys()).index(m))
counts = [win_counts[m] for m in sorted_models]
colors = ["C2" if m == best_choice_model else "C0" for m in sorted_models]
ax.barh(range(len(sorted_models)), counts, color=colors, alpha=0.85)
ax.set_yticks(range(len(sorted_models)))
ax.set_yticklabels(sorted_models, fontsize=9)
ax.set_xlabel("N subjects (best model)")
ax.set_title("Winning Model Frequency", fontweight="bold", fontsize=11)
ax.invert_yaxis()
for i, v in enumerate(counts):
    if v > 0:
        ax.text(v + 0.5, i, str(v), va="center", fontsize=9)

# ── Panel B: Full responsibility matrix (heatmap) ──
ax = axes[1]; ax.set_facecolor(BG_COLOR)
# Sort subjects by winning model, then by confidence
sort_idx = np.lexsort((responsibility.max(axis=1),  # secondary: confidence
                        winning_model_idx))          # primary: winning model
resp_sorted = responsibility[sort_idx, :]
im = ax.imshow(resp_sorted.T, aspect="auto", cmap="Blues", vmin=0, vmax=1,
               interpolation="nearest")
ax.set_yticks(range(len(model_names_choice)))
ax.set_yticklabels(model_names_choice, fontsize=8)
ax.set_xlabel(f"Subjects (n={N_SUBJ}, sorted by winning model)")
ax.set_title("Model Responsibility Matrix\nP(model | subject)", fontweight="bold", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8, label="Responsibility")

# ── Panel C: Confidence distribution ──
ax = axes[2]; ax.set_facecolor(BG_COLOR)
max_resp = responsibility.max(axis=1)
ax.hist(max_resp, bins=20, color="C0", alpha=0.7, edgecolor="white")
ax.axvline(1/len(model_names_choice), color="grey", ls="--", alpha=0.6,
           label=f"Chance ({1/len(model_names_choice):.2f})")
ax.axvline(max_resp.mean(), color="C3", ls="-", lw=2,
           label=f"Mean ({max_resp.mean():.2f})")
ax.set_xlabel("Max responsibility (confidence)")
ax.set_ylabel("N subjects")
ax.set_title("Assignment Confidence", fontweight="bold", fontsize=11)
ax.legend(fontsize=8)

fig.suptitle("HBI Model Assignment: Per-Subject Responsibility",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_model_responsibility_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

print(f"Mean max responsibility: {max_resp.mean():.3f} (SD={max_resp.std():.3f})")
print(f"Subjects with confidence > 0.8: {(max_resp > 0.8).sum()} / {N_SUBJ}")
print(f"Subjects with confidence > 0.5: {(max_resp > 0.5).sum()} / {N_SUBJ}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GROUP COLLAPSE: pure model-free  vs  model-based + hybrids
# ══════════════════════════════════════════════════════════════════════
# Prereg (Convergent validity, §Variables): the CBM/HBI binary split is
# "pure-model-free models  vs  model-based and hybrid variants". So ALL
# 11 winning models collapse into exactly two bins (nothing is dropped):
#   MF — pure model-free ......... M1_MF, M2_MF_alpha
#   MB — model-based + hybrids ... M3, M4, M5, M7, M9, M10, m11, m12, m16
# Each subject's winning_params[sid]["model"] is overwritten to "MF" or
# "MB"; the original name is preserved in winning_params[sid]["original_model"].
# NOTE (2026-07-02): an earlier narrow map collapsed only M2->MF and
# M3/M7/M9->MB, leaving hybrids unmapped; that dropped 159 subjects at the
# isin(['MF','MB']) filter and yielded the stale N=118 convergent-validity
# figure. The map below is the prereg-correct full collapse (N=277).

GROUP_MAP = {
    # --- Model-free: choice driven only by cached stage value (no EV/tree term) ---
    "M1_MF":                "MF",
    "M2_MF_alpha":          "MF",
    # --- Model-based: EV (tree) term enters the choice ---
    "M3_MB":                "MB",
    "M4_MB_MF":             "MB",
    "M5_MB_stim":           "MB",
    "M7_MB_reval":          "MB",
    "M9_w_stim":            "MB",
    "M10_w_reval":          "MB",
    "m11_sep_beta_w_reval": "MB",
    "m12_mb_mf_sep_stim":   "MB",
    "m16_sep_beta_w_stim":  "MB",
}

for sid, info in winning_params.items():
    info.setdefault("original_model", info["model"])
    info["model"] = GROUP_MAP.get(info["original_model"], info["original_model"])

from collections import Counter
print("Post-collapse group assignments:")
for grp, cnt in Counter(v["model"] for v in winning_params.values()).most_common():
    print(f"  {grp:25s}: {cnt} subjects")

# Canonical constants for all downstream cells
MODEL_GROUPS = ["MF", "MB"]
MODEL_COLORS = {"MF": "#3498DB", "MB": "#2ECC71"}
MODEL_LABELS = {"MF": "Pure MF (M1, M2)",
                "MB": "Model-Based + hybrids"}
MODEL_SHORT  = {"MF": "MF", "MB": "MB"}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8K  P(Chose MB) BY STIMULUS: MF vs MB Groups
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# Two-group palette + labels (MF vs MB collapsed groups)
MODEL_GROUPS_329 = ["MF", "MB"]
MODEL_COLORS_329 = {"MF": "#3498DB", "MB": "#2ECC71"}  # blue vs green
MODEL_LABELS_329 = {
    "MF": "MF-only\n(M2)",
    "MB": "Model-Based\n(M3 + M7 + M9)",
}

# Tag each trial with the subject's winning model
model_map = {sid: info["model"] for sid, info in winning_params.items()}
plot_df = fit_df.copy()
plot_df["winning_model"] = plot_df["subject_id"].map(model_map)

# Filter to the three groups
m3 = plot_df[plot_df["winning_model"].isin(MODEL_GROUPS_329)].copy()
m3["stimulus"] = np.where(m3["state"] == "Feather", "Feather", "Triangle")
m3["model_label"] = m3["winning_model"].map(MODEL_LABELS_329)

n_by_model = {m: m3[m3["winning_model"] == m]["subject_id"].nunique() for m in MODEL_GROUPS_329}
for m in MODEL_GROUPS_329:
    print(f"{MODEL_LABELS_329[m].split(chr(10))[0]}: {n_by_model[m]} subjects")

# ── Compute cell means ──
cell_means = m3.groupby(["winning_model", "stimulus"]).agg(
    mb_mean=("chose_mb", "mean"),
    mb_sem=("chose_mb", "sem"),
    n=("chose_mb", "count"),
).reset_index()

cell_means_q = m3.groupby(["winning_model", "stimulus", "query_num"]).agg(
    mb_mean=("chose_mb", "mean"),
    mb_sem=("chose_mb", "sem"),
    n=("chose_mb", "count"),
).reset_index()

# ── Plot: 2 panels ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG_COLOR)

# Panel A: P(chose MB) by stimulus, MF / MB
ax = ax1; ax.set_facecolor(BG_COLOR)
stimuli = ["Feather", "Triangle"]
x = np.arange(len(stimuli))
n_models = len(MODEL_GROUPS_329)
w = 0.8 / n_models  # total group width 0.8

for i, mname in enumerate(MODEL_GROUPS_329):
    sub = cell_means[cell_means["winning_model"] == mname].set_index("stimulus").reindex(stimuli)
    offs = (i - (n_models - 1) / 2) * w
    ax.bar(x + offs, sub["mb_mean"], w, yerr=1.96 * sub["mb_sem"],
           label=MODEL_LABELS_329[mname], color=MODEL_COLORS_329[mname],
           capsize=5, alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(["Feather\n(pos->neg reval)", "Triangle\n(neg->pos reval)"])
ax.set_ylabel("P(Chose MB)", fontsize=11)
ax.set_title("MB Choice Rate by Stimulus", fontweight="bold", fontsize=12)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
ax.legend(fontsize=7, loc="lower right")
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Panel B: Split by query (reval 1 vs 2)
ax = ax2; ax.set_facecolor(BG_COLOR)
cells_b = [("Feather", 1), ("Feather", 2), ("Triangle", 1), ("Triangle", 2)]
x = np.arange(len(cells_b))
xlabels = ["Feather\nReval 1", "Feather\nReval 2", "Triangle\nReval 1", "Triangle\nReval 2"]

for i, mname in enumerate(MODEL_GROUPS_329):
    sub = cell_means_q[cell_means_q["winning_model"] == mname].copy()
    sub["key"] = list(zip(sub["stimulus"], sub["query_num"]))
    sub = sub.set_index("key").reindex(cells_b)
    offs = (i - (n_models - 1) / 2) * w
    ax.bar(x + offs, sub["mb_mean"], w, yerr=1.96 * sub["mb_sem"],
           label=MODEL_LABELS_329[mname], color=MODEL_COLORS_329[mname],
           capsize=4, alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel("P(Chose MB)", fontsize=11)
ax.set_title("MB Choice Rate by Stimulus x Query", fontweight="bold", fontsize=12)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
ax.legend(fontsize=7, loc="lower right")
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("MF vs MB: Choice Rates on Anti-MF Trials",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_mb_choice_mf_vs_mb_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ── Print stats ──
print("\nCell means:")
for _, row in cell_means.iterrows():
    mlbl = MODEL_LABELS_329[row["winning_model"]].split(chr(10))[0]
    print(f"  {mlbl:>30s} x {row['stimulus']:>10s}: "
          f"P(MB)={row['mb_mean']:.3f} +/- {row['mb_sem']:.3f}  (n={int(row['n'])})")

# ── Pairwise tests per stimulus ──
from itertools import combinations
print("\nPairwise Mann-Whitney (per stimulus, subject-mean P(MB)):")
for stim in stimuli:
    per_subj = {}
    for mname in MODEL_GROUPS_329:
        sub = m3[(m3["winning_model"] == mname) & (m3["stimulus"] == stim)]
        per_subj[mname] = sub.groupby("subject_id")["chose_mb"].mean()
    print(f"  [{stim}]")
    for a, b in combinations(MODEL_GROUPS_329, 2):
        u_stat, u_p = stats.mannwhitneyu(per_subj[a], per_subj[b], alternative="two-sided")
        la = MODEL_LABELS_329[a].split(chr(10))[0]
        lb = MODEL_LABELS_329[b].split(chr(10))[0]
        print(f"    {la} (M={per_subj[a].mean():.3f}) vs {lb} (M={per_subj[b].mean():.3f}): "
              f"U={u_stat:.0f}, p={u_p:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RANK-BASED MEDIATION HELPER
# ══════════════════════════════════════════════════════════════════════
# All mediations use rank-transformed variables (Conover & Iman, 1982)
# to avoid normality/linearity assumptions. Equivalent to running
# path analysis on Spearman ranks. Bootstrap CI on indirect effect.

import statsmodels.formula.api as smf
from scipy.stats import rankdata

def rank_mediation(df, x_col, m_col, y_col, n_boot=5000, seed=42, label=""):
    """Run rank-based mediation: X -> M -> Y with bootstrap CI."""
    work = df[[x_col, m_col, y_col]].dropna().copy()
    n = len(work)
    if n < 15:
        print(f"  WARNING: n={n} too small for mediation. Skipping.")
        return None

    # Rank-transform all variables
    work["X_r"] = rankdata(work[x_col])
    work["M_r"] = rankdata(work[m_col])
    work["Y_r"] = rankdata(work[y_col])

    # Standardize ranks for interpretable betas
    for col in ["X_r", "M_r", "Y_r"]:
        work[col] = (work[col] - work[col].mean()) / work[col].std()

    # Path c: X -> Y (total)
    mod_c = smf.ols("Y_r ~ X_r", data=work).fit()
    c_total = mod_c.params["X_r"]
    c_p = mod_c.pvalues["X_r"]

    # Path a: X -> M
    mod_a = smf.ols("M_r ~ X_r", data=work).fit()
    a = mod_a.params["X_r"]
    a_p = mod_a.pvalues["X_r"]

    # Path b + c': Y ~ X + M
    mod_bc = smf.ols("Y_r ~ X_r + M_r", data=work).fit()
    b = mod_bc.params["M_r"]
    b_p = mod_bc.pvalues["M_r"]
    c_prime = mod_bc.params["X_r"]
    c_prime_p = mod_bc.pvalues["X_r"]

    # Indirect effect + bootstrap
    indirect = a * b
    np.random.seed(seed)
    boot_ab = np.zeros(n_boot)
    for bi in range(n_boot):
        idx = np.random.choice(n, size=n, replace=True)
        bdf = work.iloc[idx]
        # Re-rank within bootstrap sample
        bdf = bdf.copy()
        bdf["X_r"] = rankdata(bdf[x_col])
        bdf["M_r"] = rankdata(bdf[m_col])
        bdf["Y_r"] = rankdata(bdf[y_col])
        for col in ["X_r", "M_r", "Y_r"]:
            bdf[col] = (bdf[col] - bdf[col].mean()) / bdf[col].std()
        try:
            ba = smf.ols("M_r ~ X_r", data=bdf).fit().params["X_r"]
            bb = smf.ols("Y_r ~ X_r + M_r", data=bdf).fit().params["M_r"]
            boot_ab[bi] = ba * bb
        except:
            boot_ab[bi] = np.nan

    boot_ab = boot_ab[~np.isnan(boot_ab)]
    ci_lo, ci_hi = np.percentile(boot_ab, [2.5, 97.5])
    sig = (ci_lo > 0) or (ci_hi < 0)

    # Print results
    print(f"\n  Path c (total): {x_col} -> {y_col}")
    print(f"    beta = {c_total:.4f}, p = {c_p:.4f}")
    print(f"\n  Path a: {x_col} -> {m_col}")
    print(f"    beta = {a:.4f}, p = {a_p:.4f}")
    print(f"\n  Path b: {m_col} -> {y_col} (controlling for {x_col})")
    print(f"    beta = {b:.4f}, p = {b_p:.4f}")
    print(f"\n  Path c' (direct): {x_col} -> {y_col} (controlling for {m_col})")
    print(f"    beta = {c_prime:.4f}, p = {c_prime_p:.4f}")
    print(f"\n  Indirect effect (a * b) = {indirect:.4f}")
    print(f"  Bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  Mediation significant: {sig}")
    if abs(c_total) > 0.001:
        print(f"  Proportion mediated: {indirect / c_total:.3f}")

    # Path diagram
    print(f"\n  Path diagram:")
    print(f"                   a = {a:.3f} (p={a_p:.3f})")
    print(f"    {x_col} ────> {m_col}")
    print(f"       |                       |")
    print(f"       |                       | b = {b:.3f} (p={b_p:.3f})")
    print(f"       |                       v")
    print(f"       +──────────────> {y_col}")
    print(f"         c' = {c_prime:.3f} (p={c_prime_p:.3f})")
    print(f"         c  = {c_total:.3f} (p={c_p:.3f}) [total]")

    return {"a": a, "a_p": a_p, "b": b, "b_p": b_p,
            "c": c_total, "c_p": c_p, "c_prime": c_prime, "c_prime_p": c_prime_p,
            "indirect": indirect, "ci": (ci_lo, ci_hi), "sig": sig, "n": n,
            "work": work, "x_col": x_col, "m_col": m_col, "y_col": y_col}


def plot_mediation(res, title_str):
    """3-panel scatter for a rank-mediation result."""
    BG_COLOR = "#FAFBFC"
    work = res["work"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor=BG_COLOR)
    paths = [
        ("X_r", "M_r", f"Path a: {res["x_col"]} -> {res["m_col"]}", res["a"], res["a_p"]),
        ("M_r", "Y_r", f"Path b: {res["m_col"]} -> {res["y_col"]}", res["b"], res["b_p"]),
        ("X_r", "Y_r", f"Path c: {res["x_col"]} -> {res["y_col"]}", res["c"], res["c_p"]),
    ]
    for j, (xc, yc, ptitle, beta, pval) in enumerate(paths):
        ax = axes[j]; ax.set_facecolor(BG_COLOR)
        x, y = work[xc], work[yc]
        ax.scatter(x, y, alpha=0.4, s=25, color="C0")
        z = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 100)
        ax.plot(xl, np.polyval(z, xl), "C3", lw=2.5)
        sig = "***" if pval<.001 else "**" if pval<.01 else "*" if pval<.05 else ""
        ax.set_title(f"{ptitle}\nbeta={beta:.3f}, p={pval:.3f} {sig}", fontsize=10)
        ax.set_xlabel(xc.replace("_r"," (rank)"))
        ax.set_ylabel(yc.replace("_r"," (rank)"))
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(f"{title_str} (n={res["n"]})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    return fig

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RT DESCRIPTIVES
# ══════════════════════════════════════════════════════════════════════

import seaborn as sns
from scipy.stats import spearmanr, pearsonr, rankdata, mannwhitneyu, kruskal

BG_COLOR = "#FAFBFC"
ANX_COLORS = {"Low": "#3498DB", "High": "#E74C3C"}
MODEL_COLORS = {"MF": "#3498DB", "MB": "#2ECC71"}  # blue vs green
MODEL_LABELS = {"MF": "MF-only (M2)",
                "MB": "Model-Based (M3 + M7 + M9)"}
MODEL_SHORT  = {"MF": "MF", "MB": "MB"}
MODEL_GROUPS = ["MF", "MB"]

# ── Motor baseline (SL trials, no planning) ──
# Use the in-memory df (already filtered). No separate filtered_data.csv read.
sl_cols = sorted([c for c in df.columns if c.startswith("SL_T") and ("Step0_RT" in c or "Step1_RT" in c)])
_m = df[["subject_id"] + sl_cols].copy()
_m["subject_id"] = _m["subject_id"].astype(str)
_m["motor_s"] = _m[sl_cols].median(axis=1) / 1000
motor_map = _m.set_index("subject_id")["motor_s"].to_dict()

# ── Prepare all trials ──
rt = trial_df[trial_df["log_rt"].notna()].copy()
rt["subject_id"] = rt["subject_id"].astype(str)
rt["rt_s"] = np.exp(rt["log_rt"]) / 1000
rt["motor_s"] = rt["subject_id"].map(motor_map)
rt["delib_s"] = (rt["rt_s"] - rt["motor_s"].fillna(0)).clip(lower=0.05)
rt["log_delib"] = np.log(rt["delib_s"])

model_map_s = {str(sid): info["model"] for sid, info in winning_params.items()}
rt["model"] = rt["subject_id"].map(model_map_s)
rt["model_label"] = rt["model"].map(MODEL_LABELS)

anx_map = rt.groupby("subject_id")["somatic_anxiety"].first()
med_anx = anx_map.median()
rt["anx_group"] = rt["subject_id"].map(anx_map.apply(lambda a: "High" if a >= med_anx else "Low"))

# Per-subject summaries
subj = rt.groupby("subject_id").agg(
    mean_delib=("delib_s", "mean"), median_delib=("delib_s", "median"),
    mean_log_delib=("log_delib", "mean"), motor_s=("motor_s", "first"),
    mean_rt_s=("rt_s", "mean"), somatic_anxiety=("somatic_anxiety", "first"),
    model=("model", "first"), anx_group=("anx_group", "first"),
    accuracy=("correct", "mean"),
    accuracy_canon=("correct_canon", "mean"),   # PREREG: 0/0.5/1 scoring
).reset_index()

N = rt["subject_id"].nunique()
print(f"RT Descriptives: {len(rt)} trials, {N} subjects")
print(f"  Motor RT: M={subj['motor_s'].mean():.2f}s")
print(f"  Query RT: M={subj['mean_rt_s'].mean():.2f}s")
print(f"  Deliberation: M={subj['mean_delib'].mean():.2f}s")
print("\n  Subjects per winning model:")
for m in MODEL_GROUPS:
    n_m = subj[subj["model"] == m]["subject_id"].nunique()
    print(f"    {MODEL_SHORT[m]:>4s}: n={n_m}")

# ══════════════════════════════════════════════════════════════════════
# PLOT 1: Motor RT distribution
# ══════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG_COLOR)

ax = ax1; ax.set_facecolor(BG_COLOR)
ax.hist(subj["motor_s"] * 1000, bins=25, color="C7", alpha=0.7, edgecolor="white")
ax.set_xlabel("Motor RT (ms)")
ax.set_ylabel("N subjects")
ax.set_title("Motor Baseline (SL trials)\nNo planning — pure motor speed", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ax = ax2; ax.set_facecolor(BG_COLOR)
ax.hist(subj["mean_delib"] * 1000, bins=25, color="C0", alpha=0.7, edgecolor="white")
ax.set_xlabel("Mean Deliberation RT (ms)")
ax.set_title("Query Deliberation (RT − motor)\nPlanning + evaluation time", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"fig_rt_distributions_basic_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 2: Deliberation RT distribution by MF / MB
# ══════════════════════════════════════════════════════════════════════

m3 = subj[subj["model"].isin(MODEL_GROUPS)].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# Hist
ax = ax1; ax.set_facecolor(BG_COLOR)
for mname in MODEL_GROUPS:
    vals = m3[m3["model"] == mname]["mean_log_delib"]
    ax.hist(vals, bins=15, alpha=0.45, color=MODEL_COLORS[mname],
            label=MODEL_LABELS[mname], edgecolor="white")
ax.set_xlabel("Mean log(Deliberation RT)", fontsize=11)
ax.set_ylabel("N subjects")
ax.set_title("RT Distribution: MF vs MB", fontweight="bold")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Box + strip
ax = ax2; ax.set_facecolor(BG_COLOR)
m3["model_short"] = m3["model"].map(MODEL_SHORT)
order = [MODEL_SHORT[m] for m in MODEL_GROUPS]
palette = {MODEL_SHORT[m]: MODEL_COLORS[m] for m in MODEL_GROUPS}
sns.boxplot(data=m3, x="model_short", y="mean_log_delib", order=order, palette=palette,
            ax=ax, width=0.5, fliersize=0)
sns.stripplot(data=m3, x="model_short", y="mean_log_delib", order=order, color="black",
              alpha=0.3, size=4, ax=ax, jitter=0.15)
ax.set_xlabel("Model Group")
ax.set_ylabel("Mean log(Deliberation RT)")
ax.set_title("MF vs MB: Deliberation Time", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Kruskal-Wallis across 3 groups
groups_vals = [m3[m3["model"]==m]["mean_log_delib"].values for m in MODEL_GROUPS]
H, p_H = kruskal(*groups_vals)
ax.text(0.5, 0.96, f"Kruskal-Wallis H={H:.2f}, p={p_H:.4f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=10)

# Pairwise Mann-Whitney
print("\nRT by model (pairwise Mann-Whitney on mean_log_delib):")
for a_i in range(len(MODEL_GROUPS)):
    for b_i in range(a_i+1, len(MODEL_GROUPS)):
        a, b = MODEL_GROUPS[a_i], MODEL_GROUPS[b_i]
        u, pu = mannwhitneyu(m3[m3["model"]==a]["mean_log_delib"],
                             m3[m3["model"]==b]["mean_log_delib"])
        print(f"  {MODEL_SHORT[a]} vs {MODEL_SHORT[b]}: U={u:.0f}, p={pu:.4f}")

plt.tight_layout()
plt.savefig(f"fig_rt_by_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 2b (NEW): Anxiety distribution by model group
# ══════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# KDE / hist
ax = ax1; ax.set_facecolor(BG_COLOR)
for mname in MODEL_GROUPS:
    vals = m3[m3["model"] == mname]["somatic_anxiety"].dropna()
    ax.hist(vals, bins=15, alpha=0.45, color=MODEL_COLORS[mname],
            label=f"{MODEL_SHORT[mname]} (n={len(vals)})", edgecolor="white")
ax.set_xlabel("Somatic Anxiety", fontsize=11)
ax.set_ylabel("N subjects")
ax.set_title("Somatic Anxiety Distribution by Model Group", fontweight="bold")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Box + strip with Kruskal-Wallis
ax = ax2; ax.set_facecolor(BG_COLOR)
sns.boxplot(data=m3.dropna(subset=["somatic_anxiety"]), x="model_short", y="somatic_anxiety",
            order=order, palette=palette, ax=ax, width=0.5, fliersize=0)
sns.stripplot(data=m3.dropna(subset=["somatic_anxiety"]), x="model_short", y="somatic_anxiety",
              order=order, color="black", alpha=0.3, size=4, ax=ax, jitter=0.15)
ax.set_xlabel("Model Group")
ax.set_ylabel("Somatic Anxiety")
ax.set_title("Somatic Anxiety by Model Group", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

anx_groups = [m3[m3["model"]==m]["somatic_anxiety"].dropna().values for m in MODEL_GROUPS]
H_a, p_Ha = kruskal(*anx_groups)
ax.text(0.5, 0.96, f"Kruskal-Wallis H={H_a:.2f}, p={p_Ha:.4f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=10)

print("\nSomatic Anxiety by model group:")
for m in MODEL_GROUPS:
    v = m3[m3["model"]==m]["somatic_anxiety"].dropna()
    print(f"  {MODEL_SHORT[m]:>4s} (n={len(v):3d}): M={v.mean():.3f}, SD={v.std():.3f}, "
          f"median={v.median():.3f}")
print(f"  Kruskal-Wallis across 3 groups: H={H_a:.3f}, p={p_Ha:.4f}")
print("  Pairwise Mann-Whitney:")
for a_i in range(len(MODEL_GROUPS)):
    for b_i in range(a_i+1, len(MODEL_GROUPS)):
        a, b = MODEL_GROUPS[a_i], MODEL_GROUPS[b_i]
        va = m3[m3["model"]==a]["somatic_anxiety"].dropna()
        vb = m3[m3["model"]==b]["somatic_anxiety"].dropna()
        u, pu = mannwhitneyu(va, vb)
        print(f"    {MODEL_SHORT[a]} vs {MODEL_SHORT[b]}: U={u:.0f}, p={pu:.4f}")

plt.tight_layout()
plt.savefig(f"fig_anxiety_by_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 3: RT by Anxiety within each model group
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG_COLOR)

for i, mname in enumerate(MODEL_GROUPS):
    ax = axes[i]; ax.set_facecolor(BG_COLOR)
    ms = m3[m3["model"] == mname]
    for anx in ["Low", "High"]:
        vals = ms[ms["anx_group"] == anx]["mean_log_delib"]
        ax.hist(vals, bins=12, alpha=0.5, color=ANX_COLORS[anx],
                label=f"{anx} Somatic Anxiety", edgecolor="white")
    ax.set_xlabel("Mean log(Deliberation RT)")
    if i == 0: ax.set_ylabel("N subjects")
    n_s = len(ms)
    v = ms.dropna(subset=["somatic_anxiety"])
    if len(v) >= 10:
        rho, p = spearmanr(v["somatic_anxiety"], v["mean_log_delib"])
        s = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_title(f"{MODEL_LABELS[mname]} (n={n_s})\nAnxiety~RT: rho={rho:.3f}, p={p:.3f} {s}",
                     fontweight="bold", fontsize=10)
    else:
        ax.set_title(f"{MODEL_LABELS[mname]} (n={n_s})", fontweight="bold", fontsize=10)
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("Deliberation RT by Somatic Anxiety × Model Group", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_by_anxiety_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 4: log(Delib RT) by |EV diff| × Anxiety, for each model group
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG_COLOR)

rt_m = rt[rt["model"].isin(MODEL_GROUPS)].copy()

for i, mname in enumerate(MODEL_GROUPS):
    ax = axes[i]; ax.set_facecolor(BG_COLOR)
    sub = rt_m[rt_m["model"] == mname]
    for anx in ["Low", "High"]:
        anx_sub = sub[sub["anx_group"] == anx]
        if len(anx_sub) == 0: continue
        means = anx_sub.groupby("abs_ev_diff").agg(
            y_mean=("log_delib", "mean"), y_sem=("log_delib", "sem")).reset_index()
        means = means.sort_values("abs_ev_diff")
        ax.errorbar(means["abs_ev_diff"], means["y_mean"], yerr=1.96 * means["y_sem"],
                    color=ANX_COLORS[anx], marker="o", lw=2.5, capsize=4, ms=8,
                    label=f"{anx} Somatic Anxiety", markeredgecolor="white", markeredgewidth=1)
    ax.set_xlabel("|EV − Offer|", fontsize=11)
    if i == 0: ax.set_ylabel("log(Deliberation RT)", fontsize=11)
    ax.set_title(f"{MODEL_LABELS[mname]}", fontweight="bold", fontsize=11)
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("Deliberation RT by Difficulty × Somatic Anxiety: MF vs MB\n(RT = query RT − motor baseline)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_by_evdiff_anxiety_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

print("\n" + "=" * 70)
print("ANXIETY ~ DELIBERATION RT CORRELATIONS")
print("=" * 70)
for mname in ["All"] + MODEL_GROUPS:
    s = subj if mname == "All" else subj[subj["model"] == mname]
    v = s.dropna(subset=["somatic_anxiety"])
    if len(v) < 10: continue
    rho, p = spearmanr(v["somatic_anxiety"], v["mean_log_delib"])
    stars = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
    mlbl = mname if mname == "All" else MODEL_SHORT[mname]
    print(f"  {mlbl:>4s} (n={len(v):3d}): rho={rho:+.3f}, p={p:.4f} {stars}")


### Re-run H1 / H2 under the CBM classification (confirmation)

*Same cells as §2; now that `subj` carries CBM model labels, the CBM-group panels populate — confirming whether the model-agnostic findings hold under the computational classification.*

In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# OVERTHINKING (preregistered formula)
# ══════════════════════════════════════════════════════════════════════
# Per-subject means only. Z-score ONCE across the full MF+MB sample
# (between-subject standardization), per prereg.
#
#   OT_i = z(delib_log_i) − z(logit(acc_canon_i))
#
# where logit(p) = log(p/(1-p)) with p clipped to [0.025, 0.975].
# Positive OT = OVERTHINKING (slow AND inaccurate).
# Negative OT = efficient (fast AND accurate).
# (This is the sign convention named in the prereg footnote — the prior
#  pilot's OT had the opposite sign; the current convention's predicted
#  H1/H2 correlations are POSITIVE.)

from scipy.stats import spearmanr, pearsonr, rankdata

BG_COLOR = "#FAFBFC"

# PREREG-FIDELITY: pool z-score across the FULL sample (no CBM filter),
# and carry the model-agnostic MB classifier on ot so H1/H2 inherit it.
ot = subj.copy()
ot = ot.merge(mb_score_df[['subject_id', 'mb_classif_agnostic']],
              on='subject_id', how='left')
ot = ot.dropna(subset=["somatic_anxiety", "mean_log_delib", "accuracy_canon"])

# Pooled z-scoring across ALL subjects in MF + MB
# Prereg formula — z(delib_log) − z(logit(acc_canon)), with the logit
# clipped to [0.025, 0.975] to avoid infinities at the extremes.
def _logit_clipped(p, lo=0.025, hi=0.975):
    p_c = np.clip(p, lo, hi)
    return np.log(p_c / (1.0 - p_c))

ot["z_delib_log"] = (ot["mean_log_delib"] - ot["mean_log_delib"].mean()) / ot["mean_log_delib"].std()
ot["logit_acc"]   = _logit_clipped(ot["accuracy_canon"])
ot["z_logit_acc"] = (ot["logit_acc"] - ot["logit_acc"].mean()) / ot["logit_acc"].std()
ot["ot"]          = ot["z_delib_log"] - ot["z_logit_acc"]    # POSITIVE = overthinking
# Keep legacy names available so downstream plot code that references
# z_rt / z_acc still works (z_rt aliases z_delib_log; z_acc aliases the
# new z_logit_acc — both are now under the prereg's transform).
ot["z_rt"]  = ot["z_delib_log"]
ot["z_acc"] = ot["z_logit_acc"]

print(f"Pooled z-scoring across {len(ot)} subjects (MF + MB)")
print(f"  mean_log_delib:   M={ot['mean_log_delib'].mean():+.3f},  SD={ot['mean_log_delib'].std():.3f}")
print(f"  accuracy:         M={ot['accuracy'].mean():+.3f},   SD={ot['accuracy'].std():.3f}")
print(f"  OT (pooled z):    M={ot['ot'].mean():+.3f},   SD={ot['ot'].std():.3f}")

# ── CBM-group OT panels/figure: only after the Modeling section builds CBM labels ──
if bool(subj['model'].notna().any()):

    print("\n" + "="*70)
    print("OT ~ Anxiety  (per group, pooled z-scoring)")
    print("="*70)

    rows = []
    for grp in MODEL_GROUPS:
        g = ot[ot["model"] == grp].dropna(subset=["somatic_anxiety","ot","z_rt","z_acc"])
        if len(g) < 2:
            rows.append(dict(grp=grp, n=len(g), rho_ot=np.nan, p_ot=np.nan,
                             r_ot=np.nan, p_ot_p=np.nan,
                             rho_rt=np.nan, p_rt=np.nan, rho_acc=np.nan, p_acc=np.nan))
            print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)}) — too few subjects for correlations, skipping")
            continue
        rho_ot,  p_ot_s  = spearmanr(g["somatic_anxiety"], g["ot"])
        r_ot,    p_ot_p  = pearsonr(g["somatic_anxiety"], g["ot"])
        rho_rt,  p_rt    = spearmanr(g["somatic_anxiety"], g["z_rt"])
        rho_acc, p_acc   = spearmanr(g["somatic_anxiety"], g["z_acc"])
        rows.append(dict(grp=grp, n=len(g), rho_ot=rho_ot, p_ot=p_ot_s,
                         r_ot=r_ot, p_ot_p=p_ot_p,
                         rho_rt=rho_rt, p_rt=p_rt, rho_acc=rho_acc, p_acc=p_acc))

        s_ot  = "***" if p_ot_s<.001  else "**" if p_ot_s<.01  else "*" if p_ot_s<.05  else ""
        s_rt  = "***" if p_rt<.001    else "**" if p_rt<.01    else "*" if p_rt<.05    else ""
        s_acc = "***" if p_acc<.001   else "**" if p_acc<.01   else "*" if p_acc<.05   else ""

        print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)})")
        print(f"    Anxiety ~ OT:              rho={rho_ot:+.3f}, p={p_ot_s:.4f} {s_ot:<3s}  "
              f"|  Pearson r={r_ot:+.3f}, p={p_ot_p:.4f}")
        print(f"    Anxiety ~ z(log_delib):    rho={rho_rt:+.3f},  p={p_rt:.4f} {s_rt}")
        print(f"    Anxiety ~ z(accuracy):     rho={rho_acc:+.3f},  p={p_acc:.4f} {s_acc}")
        if p_rt < .05 and p_acc > .1:
            print(f"    → OVERTHINKING signature: anxiety increases RT but NOT accuracy.")
        elif p_rt < .05 and p_acc < .05 and rho_acc > 0:
            print(f"    → Productive deliberation: anxiety increases both RT and accuracy.")
        elif p_rt > .1:
            print(f"    → No anxiety → RT effect in this group.")

    # Also a between-groups test: does OT differ between MF and MB?
    from scipy.stats import mannwhitneyu
    g_mf = ot[ot["model"] == "MF"]["ot"]
    g_mb = ot[ot["model"] == "MB"]["ot"]
    u, p_u = mannwhitneyu(g_mf, g_mb, alternative="two-sided")
    print(f"\n  Between-group OT: MF median={g_mf.median():+.3f}, MB median={g_mb.median():+.3f}")
    print(f"    Mann-Whitney U={u:.0f}, p={p_u:.4f}")

    # ══════════════════════════════════════════════════════════════════════
    # PLOT 1: OT ~ Anxiety (rank-rank), one panel per group + RT/Acc scatter
    # ══════════════════════════════════════════════════════════════════════

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG_COLOR)

    for j, grp in enumerate(MODEL_GROUPS):
        ax = axes[j]; ax.set_facecolor(BG_COLOR)
        g = ot[ot["model"] == grp]
        xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
        ax.scatter(xr, yr, alpha=0.65, s=50,
                   color=MODEL_COLORS[grp], edgecolors="white", linewidths=0.6)
        z = np.polyfit(xr, yr, 1)
        xs = np.linspace(xr.min(), xr.max(), 100)
        ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
        rho, p = spearmanr(g["somatic_anxiety"], g["ot"])
        star = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_xlabel("Anxiety (rank)", fontsize=11)
        if j == 0:
            ax.set_ylabel("OT = z(acc) − z(log Delib) (rank)", fontsize=11)
        ax.set_title(f"{MODEL_LABELS[grp]}\nn={len(g)},  rho={rho:.3f},  p={p:.3f} {star}",
                     fontweight="bold", fontsize=11)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # Third panel: z(RT) vs z(acc), colored by anxiety, shaped by group
    ax = axes[2]; ax.set_facecolor(BG_COLOR)
    markers = {"MF": "o", "MB": "s"}
    for grp in MODEL_GROUPS:
        g = ot[ot["model"] == grp]
        sc = ax.scatter(g["z_rt"], g["z_acc"], c=g["somatic_anxiety"], cmap="RdYlBu_r",
                        alpha=0.8, s=65, marker=markers[grp],
                        edgecolors=MODEL_COLORS[grp], linewidths=1.8,
                        label=f"{MODEL_SHORT[grp]} (n={len(g)})")
    plt.colorbar(sc, ax=ax, label="Anxiety", shrink=0.8)
    ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
    ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
    ax.set_xlabel("z(log Delib RT)   ← faster | slower →", fontsize=10)
    ax.set_ylabel("z(accuracy)   ← less | more →", fontsize=10)
    ax.set_title("RT × Accuracy (pooled z)\nupper-left = efficient  |  lower-right = overthinking",
                 fontweight="bold", fontsize=10)
    ax.legend(fontsize=9, loc="best")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("Simple OT = z(accuracy) − z(log Delib RT)   —   pooled z, per-group correlation",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_simple_pooled_z_{OUT_PREFIX}_cbmconfirm.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H1 (preregistered) — Somatic Anxiety × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Predictor : Survey_ANXIETY_2_Total_Score (10-item Physiological Anxiety
#             Checklist = somatic anxiety), assigned to df['somatic_anxiety']
#             in cell 3 of the V4 notebook (post-rename 2026-05-27).
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across the
#             full MF+MB sample, computed in the simple-OT cell above and
#             held in `ot`. Positive OT = overthinking (prereg sign).
# Test      : Spearman ρ, one-sided positive p (per prereg, α = .05).
# Partition : mb_classif_agnostic — model-agnostic MB classifier per prereg.
# Panels    : All, mb_score-MB (PRIMARY, H1), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#
# Additive cell: parallels the H2 cell but substitutes the somatic-anxiety
# predictor. Does not modify any earlier analysis.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError("`ot` not defined. Run the simple OT × Anxiety cell first.")
if "somatic_anxiety" not in df.columns:
    raise RuntimeError("df['somatic_anxiety'] missing — re-run cell 3 (it sets "
                       "Anxiety = Survey_ANXIETY_2_Total_Score, the somatic scale).")
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score classifier cell.")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join somatic anxiety into the OT table
somatic_map = df.set_index("subject_id")["somatic_anxiety"]
ot_h1 = ot.copy()
ot_h1["somatic_anxiety"] = ot_h1["subject_id"].map(somatic_map)
ot_h1 = ot_h1.dropna(subset=["somatic_anxiety", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).

print("=" * 70)
print("H1 — Somatic Anxiety × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN somatic/OT: N = {len(ot_h1)}")

analysis_groups_h1 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
]
if bool(subj['model'].notna().any()):   # CBM panels only after the Modeling section
    analysis_groups_h1 += [
        ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory,
        ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory,
    ]
rows_h1 = []
for label, kind, val, is_primary in analysis_groups_h1:
    if kind == "all":
        g = ot_h1
    elif kind == "mb_agn":
        g = ot_h1[ot_h1["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h1[ot_h1["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h1.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["somatic_anxiety"], g["ot"])
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["somatic_anxiety"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h1_tag = "   ← H1 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h1_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h1.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot ──
_BG_H1 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H1 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#E74C3C",    # red — primary
    "nonMB (mb_score)": "#F1948A",
    "MF  (CBM)":        "#FAD7A0",    # exploratory
    "MB  (CBM)":        "#F5CBA7",    # exploratory
}

n_panels = len(rows_h1)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H1)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h1):
    ax = axes[j]; ax.set_facecolor(_BG_H1)
    ax.set_xlabel("Somatic Anxiety (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h1_tag = "  ← H1" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h1_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H1.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h1_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H1 — Somatic Anxiety × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h1_somatic_ot_{OUT_PREFIX}_cbmconfirm.png", dpi=200, bbox_inches="tight", facecolor=_BG_H1)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H2 (preregistered) — Worry × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Additive cell: does not modify any earlier analysis. Parallels the
# simple OT × Anxiety cell above but substitutes the Worry predictor.
#
# Predictor : Survey_ANXIETY_1_Total_Score (8-item PSWQ trait Worry),
#             already assigned to df["worry"] in cell 3.
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across
#             MF+MB, computed in the simple-OT cell above and held in `ot`.
#             Positive OT = overthinking (prereg sign convention), so the
#             predicted one-sided positive Spearman ρ tests the H2 claim
#             that higher worry → more overthinking.
# Test      : Spearman rho with a one-sided positive p (per prereg, α=.05).
# Panels    : All, mb_score-MB (PRIMARY, H2), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#             Per prereg, H2 is the mb_score-MB panel.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError(
        "`ot` not defined. Run the simple OT × Anxiety cell first "
        "(it builds the pooled-z OT table this analysis reuses).")
if "worry" not in df.columns:
    raise RuntimeError(
        "df['worry'] missing — re-run cell 3 (it sets "
        "worry = sum of PSWQ-8 items = Survey_ANXIETY_1_Total_Score).")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join Worry into the OT table
worry_map = df.set_index("subject_id")["worry"]
ot_h2 = ot.copy()
ot_h2["worry"] = ot_h2["subject_id"].map(worry_map)
ot_h2 = ot_h2.dropna(subset=["worry", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score "
                       "classifier cell near the top of the notebook.")

print("=" * 70)
print("H2 — Worry (PSWQ trait) × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN Worry/OT: N = {len(ot_h2)}")

# Panel definitions: (label, partition_kind, partition_value, is_h2_primary)
#   partition_kind ∈ {'all', 'mb_agn', 'cbm'}
analysis_groups_h2 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
]
if bool(subj['model'].notna().any()):   # CBM panels only after the Modeling section
    analysis_groups_h2 += [
        ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory,
        ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory,
    ]
rows_h2 = []
for label, kind, val, is_primary in analysis_groups_h2:
    if kind == "all":
        g = ot_h2
    elif kind == "mb_agn":
        g = ot_h2[ot_h2["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h2[ot_h2["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h2.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["worry"], g["ot"])
    # one-sided positive: halve two-sided when rho>0, else 1 - half
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["worry"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h2_tag = "   ← H2 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h2_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h2.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot: rank-rank scatter, one panel per analysis group ──
_BG_H2 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H2 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#2ECC71",    # green — primary
    "nonMB (mb_score)": "#3498DB",
    "MF  (CBM)":        "#85C1E9",    # light blue — exploratory
    "MB  (CBM)":        "#A9DFBF",    # light green — exploratory
}

n_panels = len(rows_h2)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H2)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h2):
    ax = axes[j]; ax.set_facecolor(_BG_H2)
    ax.set_xlabel("Worry / PSWQ (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h2_tag = "  ← H2" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h2_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["worry"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H2.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h2_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H2 — Worry (PSWQ trait) × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h2_worry_ot_{OUT_PREFIX}_cbmconfirm.png", dpi=200, bbox_inches="tight", facecolor=_BG_H2)
plt.show()


In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# OVERTHINKING — prereg formula, per-subgroup z-scoring (descriptive)
# ══════════════════════════════════════════════════════════════════════
# For the whole group AND for each subgroup in MODEL_GROUPS:
#   1. Take per-subject mean log(deliberation_RT) and mean correct_canon
#   2. Z-score BOTH between subjects WITHIN that analysis group
#   3. OT = z(delib_log) − z(logit(acc_canon))   [prereg formula]
#      Positive OT = overthinking (slow AND inaccurate)
#      Negative OT = efficient    (fast AND accurate)
#   4. Relate OT to anxiety (Spearman + Pearson)
# Note: this cell uses per-subgroup z-scoring, distinct from the H1/H2
# tests (which use pooled MF+MB z-scoring, per prereg).

from scipy.stats import spearmanr, pearsonr, rankdata

BG_COLOR = "#FAFBFC"

# Pool over the subgroups defined in MODEL_GROUPS (from the descriptives cell)
ot_pool = subj[subj["model"].isin(MODEL_GROUPS)].copy()
print(f"OT = z(acc) − z(RT): N pooled = {len(ot_pool)} subjects across {len(MODEL_GROUPS)} subgroups")

# Define analysis groups: 'All' first, then each subgroup
analysis_groups = [("All", None)] + [(MODEL_SHORT[m], m) for m in MODEL_GROUPS]

def _analysis_color(key):
    """All = slate; subgroups take MODEL_COLORS."""
    if key is None:
        return "#34495E"
    return MODEL_COLORS.get(key, "#555555")

def _analysis_label(short, key):
    if key is None:
        return "All subjects (pooled)"
    return f"{short}: {MODEL_LABELS[key]}"

rows = []
for short, mkey in analysis_groups:
    g = ot_pool.copy() if mkey is None else ot_pool[ot_pool["model"] == mkey].copy()
    if len(g) < 10:
        print(f"\n  {short}: n={len(g)}, too few")
        continue

    # Z-score between subjects WITHIN this analysis group
    # Prereg formula (positive OT = overthinking)
    def _logit_clip(p, lo=0.025, hi=0.975):
        p_c = np.clip(p, lo, hi)
        return np.log(p_c / (1.0 - p_c))
    g["z_delib_log"] = (g["mean_log_delib"] - g["mean_log_delib"].mean()) / g["mean_log_delib"].std()
    g["_logit_acc"]  = _logit_clip(g["accuracy_canon"])
    g["z_logit_acc"] = (g["_logit_acc"] - g["_logit_acc"].mean()) / g["_logit_acc"].std()
    g["ot"]          = g["z_delib_log"] - g["z_logit_acc"]   # POSITIVE = overthinking
    g["z_rt"]  = g["z_delib_log"]    # legacy alias for plotting code below
    g["z_acc"] = g["z_logit_acc"]   # legacy alias for plotting code below

    v = g.dropna(subset=["somatic_anxiety"])

    rho_ot,  p_ot   = spearmanr(v["somatic_anxiety"], v["ot"])
    r_ot,    p_ot_p = pearsonr(v["somatic_anxiety"], v["ot"])
    rho_rt,  p_rt   = spearmanr(v["somatic_anxiety"], v["z_rt"])
    rho_acc, p_acc  = spearmanr(v["somatic_anxiety"], v["z_acc"])

    s_ot  = "***" if p_ot<.001  else "**" if p_ot<.01  else "*" if p_ot<.05  else ""
    s_rt  = "***" if p_rt<.001  else "**" if p_rt<.01  else "*" if p_rt<.05  else ""
    s_acc = "***" if p_acc<.001 else "**" if p_acc<.01 else "*" if p_acc<.05 else ""

    print(f"\n  {_analysis_label(short, mkey)} (n={len(v)})")
    print(f"    Anxiety ~ OT = z(acc) − z(RT):  Spearman rho={rho_ot:+.3f} p={p_ot:.4f} {s_ot}  "
          f"Pearson r={r_ot:+.3f} p={p_ot_p:.4f}")
    print(f"      Component → Anxiety ~ z(RT):  rho={rho_rt:+.3f} p={p_rt:.4f} {s_rt}")
    print(f"      Component → Anxiety ~ z(acc): rho={rho_acc:+.3f} p={p_acc:.4f} {s_acc}")

    g["__short"] = short
    g["__mkey"]  = mkey if mkey is not None else "__all__"
    g["__anxiety_rank"] = rankdata(g["somatic_anxiety"].fillna(g["somatic_anxiety"].median()))
    g["__ot_rank"]      = rankdata(g["ot"])
    rows.append((short, mkey, g))

# ── Plot 1: OT ~ Anxiety (rank-rank), one panel per analysis group ──
if len(rows) > 0:
    n_panels = len(rows)
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=BG_COLOR)
    if n_panels == 1:
        axes = [axes]

    for i, (short, mkey, g) in enumerate(rows):
        ax = axes[i]; ax.set_facecolor(BG_COLOR)
        v = g.dropna(subset=["somatic_anxiety", "ot"])
        xr, yr = v["__anxiety_rank"], v["__ot_rank"]
        ax.scatter(xr, yr, alpha=0.55, s=40, color=_analysis_color(mkey),
                   edgecolors="white", linewidths=0.5)
        z = np.polyfit(xr, yr, 1)
        xs = np.linspace(xr.min(), xr.max(), 100)
        ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
        rho, p = spearmanr(v["somatic_anxiety"], v["ot"])
        s = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_xlabel("Anxiety (rank)", fontsize=11)
        if i == 0:
            ax.set_ylabel("OT = z(acc) − z(RT) (rank)", fontsize=11)
        ax.set_title(f"{_analysis_label(short, mkey)}\nn={len(v)}, rho={rho:.3f}, p={p:.3f} {s}",
                     fontweight="bold", fontsize=10)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("OT = z(accuracy) − z(log Delib RT), z-scored within group ~ Anxiety\n"
                 "(positive OT = efficient; negative OT = slow + inaccurate = overthinking)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_zscore_anxiety_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()

    # ── Plot 2: z(RT) vs z(acc), colored by anxiety ──
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=BG_COLOR)
    if n_panels == 1:
        axes = [axes]

    for i, (short, mkey, g) in enumerate(rows):
        ax = axes[i]; ax.set_facecolor(BG_COLOR)
        v = g.dropna(subset=["somatic_anxiety"])
        sc = ax.scatter(v["z_rt"], v["z_acc"], c=v["somatic_anxiety"],
                        cmap="RdYlBu_r", alpha=0.7, s=45,
                        edgecolors="white", linewidths=0.5)
        plt.colorbar(sc, ax=ax, label="Anxiety", shrink=0.8)
        ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
        ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
        ax.set_xlabel("z(log Delib RT)\n← faster  |  slower →", fontsize=10)
        if i == 0:
            ax.set_ylabel("z(accuracy)\n← less accurate  |  more accurate →", fontsize=10)
        ax.set_title(f"{_analysis_label(short, mkey)}\nupper-left = efficient | lower-right = overthinking",
                     fontweight="bold", fontsize=10)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("RT vs Accuracy (z-scored within group), colored by anxiety",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_zscore_components_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Convergent validity (preregistered) — Cohen's κ between classifiers
# ROBUST VERSION (2026-07-02): computes the HBI allocation directly from
# the responsibility matrix in cbm_output_<OUT_PREFIX>/hbi_choice.pkl
# instead of relying on in-memory `subj`/`winning_params` state, and uses
# the prereg-correct FULL 11-model collapse (nothing dropped, N = 277).
# The old in-memory path silently dropped subjects whose winner was
# outside {M2,M3,M7,M9} → stale N=118, κ=0.966. Verified externally:
# κ = +0.764, agreement 246/277 (88.8%) — see
# verify_convergence_agnostic_vs_hbi.py / LOGBOOK 2026-07-02.
#
# Prereg flag: if κ < 0.5, the two classifiers no longer measure the
# same construct → reconsider before interpreting H1/H2/H3.
# V3 pilot reference: 85.6% agreement, κ = 0.71 (N = 139).
# ══════════════════════════════════════════════════════════════════════
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score classifier cell.")
if 'trial_df' not in dir():
    raise RuntimeError("`trial_df` not defined — run the trial-level data cell.")

import os, sys, types, pickle
from scipy.stats import spearmanr

# ── 1. Load HBI responsibilities (lenient unpickler: no cbm import needed) ──
_cbm_dir = f"cbm_output_{OUT_PREFIX}" if 'OUT_PREFIX' in dir() else "cbm_output_v4_1_all"
_hbi_path = os.path.join(_cbm_dir, "hbi_choice.pkl")

class _StubObj:
    def __init__(self, *a, **kw): pass
    def __setstate__(self, st):
        self.__dict__.update(st if isinstance(st, dict) else {"_state": st})

class _LenientUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            mod = sys.modules.setdefault(module, types.ModuleType(module))
            if not hasattr(mod, name):
                setattr(mod, name, type(name, (_StubObj,), {}))
            return getattr(mod, name)

with open(_hbi_path, "rb") as _f:
    _hbi = _LenientUnpickler(_f).load()

_resp   = _hbi.output.responsibility          # (N, K)
_mnames = [m if isinstance(m, str) else getattr(m, "__name__", type(m).__name__)
           for m in _hbi.input.models]
print(f"HBI responsibilities loaded from {_hbi_path}: {_resp.shape}, "
      f"models = {_mnames}")

# ── 2. Subject order = CBM fit order (mirror of the data_list cell) ──
if 'FIT_ALL_TRIALS' in dir() and not FIT_ALL_TRIALS:
    _fit = trial_df[trial_df["is_anti_mf"] == 1].dropna(subset=["choice"])
else:
    _fit = trial_df.dropna(subset=["choice"])
_sids = _fit["subject_id"].unique().tolist()
assert len(_sids) == _resp.shape[0], (
    f"alignment failure: {len(_sids)} subjects in fit order vs "
    f"{_resp.shape[0]} rows in HBI responsibility matrix")

# ── 3. Full prereg-correct MF/MB collapse (MF iff no EV term; hybrids→MB) ──
_GROUP_MAP_FN = {
    "m1_mf_only": "MF", "m2_mf_alpha": "MF",
    "m3_mb_only": "MB", "m4_mb_mf": "MB", "m5_mb_sep_stim": "MB",
    "m7_mb_sep_reval": "MB", "m9_sep_w_stim": "MB", "m10_sep_w_reval": "MB",
    "m11_sep_beta_w_reval": "MB", "m12_mb_mf_sep_stim": "MB",
    "m16_sep_beta_w_stim": "MB",
}
def _to_group(name):
    if name in _GROUP_MAP_FN: return _GROUP_MAP_FN[name]
    low = name.lower()
    if "mb" in low or "w_" in low or "det" in low: return "MB"
    if "mf" in low: return "MF"
    raise ValueError(f"unmapped model name: {name}")

_winner   = _resp.argmax(axis=1)
_max_resp = _resp[np.arange(len(_winner)), _winner]
_win_nm   = [_mnames[k] for k in _winner]
_grp      = np.array([_to_group(n) for n in _win_nm])
_mb_cols  = [i for i, n in enumerate(_mnames) if _to_group(n) == "MB"]
_fam_mb   = _resp[:, _mb_cols].sum(axis=1)     # P(MB family) per subject

hbi_alloc = pd.DataFrame({
    "subject_id":  _sids,
    "hbi_model":   _win_nm,
    "hbi_group":   _grp,
    "hbi_max_resp": _max_resp,
    "hbi_P_MB_family": _fam_mb,
})
print(f"HBI collapse: MB = {(_grp=='MB').sum()}, MF = {(_grp=='MF').sum()} "
      f"(N = {len(_grp)}); mean max responsibility = {_max_resp.mean():.3f}")

# ── 4. κ against the model-agnostic classifier (all subjects, none dropped) ──
kdf = (mb_score_df[["subject_id", "mb_score_F", "mb_score_T",
                    "mb_classif_agnostic"]]
       .merge(hbi_alloc, on="subject_id", how="inner"))
kdf["agn_mb"] = kdf["mb_classif_agnostic"].astype(int)
kdf["cbm_mb"] = (kdf["hbi_group"] == "MB").astype(int)

n_joint = len(kdf)
n_agree = int((kdf["agn_mb"] == kdf["cbm_mb"]).sum())
obs     = n_agree / n_joint
_pa, _pc = kdf["agn_mb"].mean(), kdf["cbm_mb"].mean()
exp     = _pa*_pc + (1-_pa)*(1-_pc)
kappa   = (obs - exp) / (1 - exp)
kappa_se = np.sqrt(obs*(1-obs)/n_joint) / (1-exp)
ci_lo, ci_hi = kappa - 1.96*kappa_se, min(kappa + 1.96*kappa_se, 1.0)

print("=" * 70)
print("Convergent validity — model-agnostic vs HBI/CBM (full collapse)")
print("=" * 70)
print(f"  N with both classifications : {n_joint}")
print(f"  Agreement                   : {n_agree}/{n_joint} ({100*obs:.1f}%)")
print(f"  Cohen's κ                   : {kappa:+.3f}  (95% CI {ci_lo:.3f} to {ci_hi:.3f})")
print(f"  Prior-pilot reference (V3)  : 85.6% agreement, κ = 0.71  (N = 139)")
if kappa < 0.5:
    print("  ⚠ κ < 0.5 — per prereg, reconsider before interpreting H1/H2/H3.")
elif kappa < 0.71:
    print("  • κ ≥ 0.5 (prereg OK); slightly below/near V3 pilot 0.71." if kappa < 0.71 else "")
else:
    print("  ✓ κ ≥ V3 pilot — classifiers agree at least as well as in V3.")

# 2×2 confusion
cm = np.zeros((2, 2), dtype=int)
for _c, _a in zip(kdf["cbm_mb"], kdf["agn_mb"]):
    cm[_c, _a] += 1
print()
print("  Confusion matrix  (rows: HBI/CBM, cols: model-agnostic)")
print("                       agn=nonMB   agn=MB")
print(f"    HBI = MF (pure)      {cm[0,0]:4d}       {cm[0,1]:4d}")
print(f"    HBI = MB (+hybrid)   {cm[1,0]:4d}       {cm[1,1]:4d}")

# Continuous convergence: graded mb_score vs graded HBI family probability
kdf["mb_score_max"] = kdf[["mb_score_F", "mb_score_T"]].max(axis=1)
_rho, _p = spearmanr(kdf["mb_score_max"], kdf["hbi_P_MB_family"])
print(f"\n  Continuous check: Spearman(mb_score_max, P(MB family)) "
      f"ρ = {_rho:+.3f}, p = {_p:.3g}")

# Disagreement anatomy (hybrid caveat: MB-by-identity with possibly w≈0)
_dis = kdf[kdf["agn_mb"] != kdf["cbm_mb"]]
print(f"\n  Disagreements: {len(_dis)} "
      f"(HBI-MB/agn-nonMB: {int(((_dis['cbm_mb']==1)).sum())}, "
      f"HBI-MF/agn-MB: {int(((_dis['cbm_mb']==0)).sum())})")
if len(_dis):
    print(_dis.groupby("hbi_model")
              .agg(n=("agn_mb", "size"),
                   mean_max_resp=("hbi_max_resp", "mean"),
                   mean_P_MB_fam=("hbi_P_MB_family", "mean"))
              .round(3).to_string())

# ── 5. Plot + save ──
fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.5))
ax.set_facecolor('#FAFBFC')
im = ax.imshow(cm, cmap='YlGnBu', aspect='auto')
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-MB', 'MB'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['non-MB', 'MB'])
ax.set_xlabel('Model-agnostic classifier')
ax.set_ylabel('CBM (HBI) classifier')
for _i in range(2):
    for _j in range(2):
        ax.text(_j, _i, str(cm[_i, _j]), ha='center', va='center',
                fontsize=16, fontweight='bold',
                color='white' if cm[_i, _j] > cm.max()/2 else 'black')
ax.set_title(f"Convergent validity — Cohen's κ = {kappa:+.3f}\n"
             f"({100*obs:.1f}% agreement, N = {n_joint})",
             fontweight='bold')
plt.tight_layout()
plt.savefig(f"fig_kappa_convergent_validity_{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}_CORRECTED.png",
            dpi=180, bbox_inches='tight')
plt.show()

kdf.to_csv("convergence_agnostic_vs_hbi_"
           f"{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}.csv", index=False)
print("\nSaved corrected κ figure + convergence CSV.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Sensitivity: hybrid winners reclassified by FITTED w (resolves the
# 2026-06-10 open item "hybrid-by-identity vs hybrid-by-w")
# ══════════════════════════════════════════════════════════════════════
# The identity rule labels every hybrid winner MB even when fitted w≈0
# (i.e. the subject puts ~all decision weight on the cached MF value).
# Here each hybrid winner is relabeled MB iff any of their fitted mixing
# weights exceeds 0.5 (OR-rule, mirroring the agnostic classifier).
# w extraction per likelihood (cell 8A): m4 w=expit(p1); m9 wf/wt=expit(p1,p2);
# m10 w1/w2=expit(p1,p2); m11 w=p2 RAW (no sigmoid in likelihood);
# m16 wf/wt=expit(p2,p3); m12 malformed (w=beta_f bug) → kept MB by identity.
# Requires the robust κ cell above (uses _hbi, _mnames, _resp, _winner, kdf).
# Verified 2026-07-02: κ 0.764 → 0.933; splits converge to 160/117 vs 159/118.
# ══════════════════════════════════════════════════════════════════════
_params = _hbi.output.parameters
_midx = {n: i for i, n in enumerate(_mnames)}

def _w_max(sub_i, model):
    p = _params[_midx[model]][sub_i]
    if model == "m4_mb_mf":             return expit(p[1])
    if model == "m9_sep_w_stim":        return max(expit(p[1]), expit(p[2]))
    if model == "m10_sep_w_reval":      return max(expit(p[1]), expit(p[2]))
    if model == "m11_sep_beta_w_reval": return p[2]      # raw w in likelihood
    if model == "m12_mb_mf_sep_stim":   return np.nan    # malformed w (bug)
    if model == "m16_sep_beta_w_stim":  return max(expit(p[2]), expit(p[3]))
    return None                                          # not a hybrid

_w_df = pd.DataFrame({
    "subject_id": _sids,
    "w_fitted": [_w_max(i, m) for i, m in enumerate(_win_nm)],
})
kdf_w = kdf.merge(_w_df, on="subject_id", how="left")
_is_hyb = kdf_w["hbi_model"].isin(
    ["m4_mb_mf", "m9_sep_w_stim", "m10_sep_w_reval",
     "m11_sep_beta_w_reval", "m12_mb_mf_sep_stim", "m16_sep_beta_w_stim"])

kdf_w["hbi_group_w"] = np.where(
    ~_is_hyb, kdf_w["hbi_group"],
    np.where(kdf_w["w_fitted"].isna(), "MB",           # m12 kept by identity
             np.where(kdf_w["w_fitted"] > 0.5, "MB", "MF")))

_h = kdf_w[_is_hyb & kdf_w["w_fitted"].notna()]
print("Fitted w among hybrid winners (max across w-params):")
for _lab, _sub in [("agnostic-MB", _h[_h["agn_mb"] == 1]),
                   ("agnostic-nonMB", _h[_h["agn_mb"] == 0])]:
    print(f"  {_lab:15s} n={len(_sub):3d}  mean w={_sub['w_fitted'].mean():.3f} "
          f"median={_sub['w_fitted'].median():.3f}  "
          f"w<=0.5: {(_sub['w_fitted']<=.5).sum()}  w>0.5: {(_sub['w_fitted']>.5).sum()}")
_rho_w, _p_w = spearmanr(_h["w_fitted"], _h["mb_score_max"])
print(f"  Spearman(w, mb_score_max) among hybrid winners: "
      f"ρ = {_rho_w:+.3f}, p = {_p_w:.3g}")

_cw = (kdf_w["hbi_group_w"] == "MB").astype(int)
_aw = kdf_w["agn_mb"]
_obs = (_aw == _cw).mean()
_exp = _aw.mean()*_cw.mean() + (1-_aw.mean())*(1-_cw.mean())
_kap_w = (_obs - _exp) / (1 - _exp)
_se_w = np.sqrt(_obs*(1-_obs)/len(kdf_w)) / (1-_exp)
print(f"\nHybrid-by-w split: MB = {(kdf_w['hbi_group_w']=='MB').sum()}, "
      f"MF = {(kdf_w['hbi_group_w']=='MF').sum()}   "
      f"(identity rule: {(kdf_w['hbi_group']=='MB').sum()}/"
      f"{(kdf_w['hbi_group']=='MF').sum()}; agnostic: "
      f"{int(_aw.sum())}/{int((1-_aw).sum())})")
print(f"κ (identity rule)   : {kappa:+.3f}  ({100*obs:.1f}% agreement)")
print(f"κ (fitted-w rule)   : {_kap_w:+.3f}  (95% CI {_kap_w-1.96*_se_w:.3f} to "
      f"{min(_kap_w+1.96*_se_w,1):.3f}; {100*_obs:.1f}% agreement; N = {len(kdf_w)})")
_rem = kdf_w[(_cw != _aw)]
print(f"Remaining disagreements: {len(_rem)}")
if len(_rem):
    print(_rem.groupby("hbi_model").size().to_string())

kdf_w.to_csv(f"hybrid_w_sensitivity_"
             f"{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}.csv", index=False)
print("\nSaved hybrid-w sensitivity CSV.")

---
## CBM (HBI-winning) vs model-agnostic MB classification — overlap

Imports the CBM-derived MF/MB label per subject from
`Modelling_AllChoices.ipynb` (which writes it to
`pymc_modelA_subject_params.csv` or `ot_subj_per_subject.csv` when those
cells run) and compares it to the model-agnostic `mb_group`
(per-stim above-chance criterion: MB = `mb_score > 0.5` on either Feather or Triangle).

If neither file is found, run cells 28 + 31 in
`Modelling_AllChoices.ipynb` to produce one — those cells collapse
HBI-winning models into MF / MB and write the per-subject summary
table.

Reports the contingency table, Cohen's κ, Mann-Whitney U comparing
continuous `mb_score` between the two CBM groups, and a 2-panel figure
(violin + confusion-matrix heatmap).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CBM × MODEL-AGNOSTIC MB OVERLAP
# ══════════════════════════════════════════════════════════════════════
# Loader tries (in order):
#   1. Any per-subject CBM-label CSV in the working directory.
#   2. The HBI pickle at cbm_output/hbi_choice.pkl (argmax over the
#      responsibility matrix — no cbm package needed, lenient unpickler).
# Whichever wins yields a `cbm_labels` DataFrame with columns
#   subject_id (str), cbm_model ∈ {'MF','MB'}.
import os, pickle, sys, types

cbm_labels = None
N_SUBJ_RAW = len(rt_df)   # rt_df was loaded in cell 1 (filtered_data_v4_1.csv)

# ───────────────────────── (1) CSV path (unchanged) ─────────────────────
_cbm_candidates = [
    ('pymc_modelA_subject_params.csv', 'group'),
    ('ot_subj_per_subject.csv',        'model'),
    ('pymc_modelA_cost_anxiety.csv',   'group'),
    ('pymc_subj_alpha_anxiety.csv',    'group'),
]
for fname, col in _cbm_candidates:
    if not os.path.exists(fname): continue
    try:
        d = pd.read_csv(fname)
        if 'subject_id' in d.columns and col in d.columns:
            cbm_labels = (d[['subject_id', col]]
                          .rename(columns={col: 'cbm_model'})
                          .copy())
            cbm_labels['subject_id'] = cbm_labels['subject_id'].astype(str)
            cbm_labels = cbm_labels.dropna(subset=['cbm_model'])
            print(f"Loaded CBM labels from {fname}: "
                  f"{len(cbm_labels)} subjects, "
                  f"counts={cbm_labels['cbm_model'].value_counts().to_dict()}")
            break
    except Exception as e:
        print(f"  could not parse {fname}: {e}")

# ───────────────────────── (2) HBI pickle fallback ──────────────────────
if cbm_labels is None:
    _hbi_path = os.path.join('cbm_output', 'hbi_choice.pkl')
    if os.path.exists(_hbi_path):
        # Lenient unpickler: stub out the `cbm` package so we can read the
        # pickle even if the cbm library is not installed in this kernel.
        for _sub in ['cbm', 'cbm.fitting', 'cbm.hbi', 'cbm.models',
                     'cbm.utils', 'cbm.data', 'cbm.core']:
            sys.modules.setdefault(_sub, types.ModuleType(_sub))

        class _StubObj:
            def __init__(self, *a, **kw): pass
            def __setstate__(self, state):
                self.__dict__.update(state if isinstance(state, dict)
                                     else {'_state': state})

        class _LenientUnpickler(pickle.Unpickler):
            def find_class(self, module, name):
                try:
                    return super().find_class(module, name)
                except Exception:
                    mod = sys.modules.setdefault(module,
                                                 types.ModuleType(module))
                    if not hasattr(mod, name):
                        setattr(mod, name, type(name, (_StubObj,), {}))
                    return getattr(mod, name)

        try:
            with open(_hbi_path, 'rb') as f:
                _hbi = _LenientUnpickler(f).load()

            _resp = _hbi.output.responsibility   # (N, K)
            _models = list(_hbi.input.models)    # K function-objects / names

            def _mname(m):
                if isinstance(m, str): return m
                return getattr(m, '__name__', type(m).__name__)

            _model_names = [_mname(m) for m in _models]

            # Map model-name → 'MF' / 'MB'.  Names in this HBI run are
            # lowercase function names:
            #   m2_mf_alpha → MF;  everything else (m3_mb_only,
            #   m7_mb_sep_reval, m9_sep_w_stim, m_mb_det, m_mb_det_stim) → MB.
            # Heuristic: 'mb' or 'det' or 'w_stim' anywhere → MB; else
            # 'mf' → MF.  Names without 'mb'/'mf' default to 'MB' so a new
            # MB-style model added later doesn't silently misclassify.
            def _to_group(name):
                low = name.lower()
                if any(t in low for t in ('mb', 'det', 'w_stim')): return 'MB'
                if 'mf' in low: return 'MF'
                return 'MB'

            _grp = np.array([_to_group(n) for n in _model_names])
            _winner = _resp.argmax(axis=1)
            _subj_groups = _grp[_winner]   # length N

            # Align with subject_id order from filtered_data_v4_1.csv (same load
            # order used by Modelling_AllChoices_new.ipynb to build data_list).
            _sids = rt_df['subject_id'].astype(str).tolist()
            if len(_sids) != _resp.shape[0]:
                # Possible explanation: Modelling notebook filters further.
                # Trim/pad to the responsibility length.
                print(f"  warn: filtered_data has {len(_sids)} subjects but "
                      f"HBI responsibility has {_resp.shape[0]}. "
                      "Using min length; CBM-label alignment may drift.")
                _n = min(len(_sids), _resp.shape[0])
                _sids = _sids[:_n]; _subj_groups = _subj_groups[:_n]
                _winner = _winner[:_n]

            cbm_labels = pd.DataFrame({
                'subject_id': _sids,
                'cbm_model':  _subj_groups,
                'cbm_orig_model': [_model_names[k] for k in _winner],
                'cbm_max_resp':   _resp[np.arange(len(_winner)), _winner],
            })
            print(f"Loaded CBM labels from {_hbi_path} "
                  f"({len(cbm_labels)} subjects).")
            print(f"  HBI models: {_model_names}")
            print(f"  Per-model winners: "
                  f"{dict(pd.Series([_model_names[k] for k in _winner]).value_counts())}")
            print(f"  Collapsed counts: "
                  f"{cbm_labels['cbm_model'].value_counts().to_dict()}")
        except Exception as e:
            print(f"  could not parse {_hbi_path}: {e}")

if cbm_labels is None:
    print("\n!!! No CBM-label CSV or HBI pickle found.")
    print("    Either drop one of these CSVs in the working dir")
    print("    (pymc_modelA_subject_params.csv, ot_subj_per_subject.csv,")
    print("    pymc_modelA_cost_anxiety.csv, pymc_subj_alpha_anxiety.csv)")
    print("    OR ensure cbm_output/hbi_choice.pkl exists (from")
    print("    Modelling_AllChoices_new.ipynb cell 25, HBI step).")
else:
    # Merge with subj_meta from earlier (has mb_score + mb_group + anxiety)
    overlap = subj_meta.merge(cbm_labels, on='subject_id', how='inner')
    print(f"\nOverlap: {len(overlap)} subjects with both CBM label and "
          f"model-agnostic mb_score")

    # ── Contingency table ──
    ct_full   = pd.crosstab(overlap['cbm_model'], overlap['mb_group'], margins=True)
    ct_inner  = pd.crosstab(overlap['cbm_model'], overlap['mb_group'])
    print(f"\nContingency: CBM model × model-agnostic mb_group")
    print(ct_full)

    # ── Agreement metrics (mapping: CBM MF↔non-MB,  CBM MB↔MB) ──
    cbm_bin = (overlap['cbm_model'] == 'MB').astype(int)
    ma_bin  = (overlap['mb_group']  == 'MB').astype(int)   # MA "MB" = above chance on either stim
    n_agree = int((cbm_bin == ma_bin).sum())
    pct_agree = n_agree / len(overlap)

    # Cohen's kappa (manual, no sklearn dependency)
    obs = pct_agree
    p_cbm_mb = cbm_bin.mean()
    p_ma_mb  = ma_bin.mean()
    exp = p_cbm_mb*p_ma_mb + (1-p_cbm_mb)*(1-p_ma_mb)
    kappa = (obs - exp) / (1 - exp) if (1 - exp) > 0 else np.nan
    print(f"\nAgreement: {n_agree}/{len(overlap)} = {pct_agree:.1%},  "
          f"Cohen's kappa = {kappa:.3f}")

    # ── Continuous mb_score split by CBM group ──
    mf_scores = overlap.loc[overlap['cbm_model']=='MF', 'mb_score']
    mb_scores = overlap.loc[overlap['cbm_model']=='MB', 'mb_score']
    u, pu = mannwhitneyu(mb_scores, mf_scores, alternative='two-sided')
    print(f"\nContinuous mb_score by CBM group:")
    print(f"  CBM MF (n={len(mf_scores)}): M={mf_scores.mean():.3f}, "
          f"median={mf_scores.median():.3f}")
    print(f"  CBM MB (n={len(mb_scores)}): M={mb_scores.mean():.3f}, "
          f"median={mb_scores.median():.3f}")
    star = '***' if pu<.001 else '**' if pu<.01 else '*' if pu<.05 else ''
    print(f"  Mann-Whitney U={u:.0f}, p={pu:.3g} {star}")

    # Spearman of mb_score with binary CBM label (point-biserial via Spearman)
    rs, ps = spearmanr(overlap['mb_score'], cbm_bin)
    print(f"\nSpearman(mb_score, CBM is MB):  rho={rs:+.3f}, p={ps:.3g}")

    # ── Plot: 2 panels ──
    BG_COLOR = "#FAFBFC"
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor=BG_COLOR)

    # Panel 1: violin of mb_score by CBM group
    ax = axes[0]; ax.set_facecolor(BG_COLOR)
    parts = ax.violinplot([mf_scores.values, mb_scores.values], positions=[1,2],
                           showmeans=True, showextrema=False)
    for pc, color in zip(parts['bodies'], ['#3498DB','#2ECC71']):
        pc.set_facecolor(color); pc.set_alpha(0.55)
    for j, (y, c) in enumerate([(mf_scores.values, '#3498DB'),
                                (mb_scores.values, '#2ECC71')]):
        ax.scatter(np.random.uniform(j+1-.08, j+1+.08, len(y)), y,
                    s=22, color=c, alpha=0.7,
                    edgecolors='white', linewidths=0.5)
    ax.axhline(0.5, color='red', ls='--', alpha=.6,
                label='above-chance threshold = 0.50')
    ax.set_xticks([1,2]); ax.set_xticklabels(['CBM MF','CBM MB'])
    ax.set_ylabel('Model-agnostic mb_score  (P chose MB | anti-MF trial)')
    star = '***' if pu<.001 else '**' if pu<.01 else '*' if pu<.05 else ''
    ax.set_title(f'mb_score by CBM group\nU p={pu:.3g} {star}',
                 fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # Panel 2: confusion-matrix heatmap
    ax = axes[1]; ax.set_facecolor(BG_COLOR)
    sns.heatmap(ct_inner, annot=True, fmt='d', ax=ax, cmap='Blues',
                cbar=False, linewidths=1, linecolor='white')
    ax.set_xlabel('Model-agnostic mb_group')
    ax.set_ylabel('CBM model')
    ax.set_title(f'Agreement: {n_agree}/{len(overlap)} = {pct_agree:.0%}, '
                 f'kappa={kappa:.2f}', fontweight='bold', fontsize=10)

    fig.suptitle("CBM (HBI-winning) vs model-agnostic MB classification",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig("fig_cbm_vs_modelagnostic_overlap.png", dpi=180,
                bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()

    overlap.to_csv("rt_analyses_cbm_vs_modelagnostic.csv", index=False)
    print(f"\nSaved fig_cbm_vs_modelagnostic_overlap.png + "
          f"rt_analyses_cbm_vs_modelagnostic.csv")


---
## 9 · Model and Parameter Recovery

Simulate data from each of the top 5 models, re-fit all 5 candidate models to
the simulated data using the exact same CBM + HBI pipeline, and check whether
each generating model is recovered as the HBI winner.

**Recovery model set (5×5 confusion):**
- `M2_MF_alpha`  — pure MF, learned scaling of cached values
- `M3_MB`        — pure MB, single β
- `M5_MB_stim`   — MB with β per stimulus (Feather vs Triangle)
- `M7_MB_reval`  — MB with β per revaluation phase (Q1 vs Q2)
- `M9_w_stim`    — MB/MF mixture with per-stimulus weight w

**Simulation protocol:** for each generating model we draw 200 simulated
subjects. Each simulated subject inherits a real subject's trial structure
(ev, offer, stimulus, query) and receives parameters sampled *with replacement*
from the empirical distribution of that model's per-subject CBM MAPs — so the
simulated parameter range matches what we actually see in the data.

**Match the pipeline exactly:** CBM individual_fit + HBI + hbi_null (no MLE
shortcuts). This is slow (~30–60 min per filter setting) but is the
methodologically correct benchmark for our real results.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9A  RECOVERY CONFIG: model set, sample sizes, param pools, rng
# ══════════════════════════════════════════════════════════════════════
# Depends on: MODELS, cbm_fits, data_list, N_SUBJ, CBM_DIR, FILTER_SUFFIX, SEED
# All of the above are defined by cells 21–22 (8A–8B).
# ══════════════════════════════════════════════════════════════════════

RECOVERY_MODELS = ["M2_MF_alpha", "M3_MB", "M5_MB_stim", "M7_MB_reval", "M9_w_stim"]
N_SIM_PER_MODEL = 200
RECOVERY_PRIOR_VARIANCE = 6.25  # matches real-data fits

RECOVERY_DIR = f"recovery_output_{OUT_PREFIX}"
os.makedirs(RECOVERY_DIR, exist_ok=True)

# Separate RNG so running recovery doesn't perturb any other stochastic step
recovery_rng = np.random.default_rng(SEED)

# ── Build parameter pools from empirical CBM MAPs ──
# pool[mname] has shape (N_SUBJ, k_m); we draw rows with replacement to build
# realistic synthetic subjects.
param_pools = {}
for mname in RECOVERY_MODELS:
    assert mname in cbm_fits, f"{mname} not in cbm_fits — run cell 22 (CBM fits) first"
    param_pools[mname] = cbm_fits[mname].output.parameters  # (N, k)

print(f"[RECOVERY] Filter: {FILTER_LABEL}")
print(f"[RECOVERY] Dir:    {RECOVERY_DIR}")
print(f"[RECOVERY] Models: {RECOVERY_MODELS}")
print(f"[RECOVERY] N per generating model: {N_SIM_PER_MODEL}")
print(f"[RECOVERY] Param pools:")
for m, p in param_pools.items():
    print(f"  {m:15s}  shape={p.shape}  param_means={np.round(p.mean(axis=0), 3)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9B  SIMULATORS: generate binary choice arrays from each model
# ══════════════════════════════════════════════════════════════════════
# Each simulator takes (params, d, rng) and returns an array of 0/1 choices
# the same length as d["ev"]. The probability expression MUST match the
# log-likelihood in cell 21 (8A) exactly — copy-paste carefully.

def sim_M2_MF_alpha(params, d, rng):
    beta, alpha_raw = params
    alpha = expit(alpha_raw)
    p = expit(beta * (alpha * d["mf_value"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M3_MB(params, d, rng):
    beta = params[0]
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M5_MB_stim(params, d, rng):
    beta_f, beta_t = params
    beta = np.where(d["is_feather"], beta_f, beta_t)
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M7_MB_reval(params, d, rng):
    beta_q1, beta_q2 = params
    beta = np.where(d["is_q1"], beta_q1, beta_q2)
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M9_w_stim(params, d, rng):
    beta, wf_raw, wt_raw = params
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

SIMULATORS = {
    "M2_MF_alpha":  sim_M2_MF_alpha,
    "M3_MB":        sim_M3_MB,
    "M5_MB_stim":   sim_M5_MB_stim,
    "M7_MB_reval":  sim_M7_MB_reval,
    "M9_w_stim":    sim_M9_w_stim,
}
print("[RECOVERY] Simulators ready:", list(SIMULATORS.keys()))


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9C  RUN RECOVERY  (~30–60 min for this cell; HBI does most of the waiting)
# ══════════════════════════════════════════════════════════════════════
# For each generating model we:
#   (1) Sample N_SIM_PER_MODEL parameter sets from its empirical pool.
#   (2) For each simulated subject, donor a real trial structure from data_list,
#       then generate binary choices via the matching simulator.
#   (3) Fit ALL 5 candidate models to the simulated dataset via CBM individual_fit.
#   (4) Run HBI (+ hbi_null for PXP) over those 5 candidate fits.
#   (5) Record the HBI-winner for each simulated subject + save the true params.
# Artifacts go to RECOVERY_DIR/gen_<model>/.

import time

recovery_results = {}  # keyed by generating model name

for gen_name in RECOVERY_MODELS:
    t0 = time.time()
    print("\n" + "=" * 72)
    print(f"[RECOVERY] Generating model: {gen_name}")
    print("=" * 72)

    gen_dir = os.path.join(RECOVERY_DIR, f"gen_{gen_name}")
    os.makedirs(gen_dir, exist_ok=True)

    pool = param_pools[gen_name]            # (N_SUBJ, k_gen)
    sim_func = SIMULATORS[gen_name]

    # ── (1) + (2): build sim_data_list + record true params ──
    sim_data_list = []
    true_params = np.zeros((N_SIM_PER_MODEL, pool.shape[1]))
    donor_idx = recovery_rng.integers(0, N_SUBJ, size=N_SIM_PER_MODEL)
    param_idx = recovery_rng.integers(0, pool.shape[0], size=N_SIM_PER_MODEL)

    for i in range(N_SIM_PER_MODEL):
        params = pool[param_idx[i]].copy()
        donor  = data_list[donor_idx[i]]
        d_sim = {k: np.asarray(v).copy() for k, v in donor.items()}
        d_sim["choice"] = sim_func(params, d_sim, recovery_rng)
        sim_data_list.append(d_sim)
        true_params[i] = params

    # ── (3): fit each candidate model to sim_data_list ──
    sim_fits = {}
    for fit_name in RECOVERY_MODELS:
        mspec = MODELS[fit_name]
        k = mspec["k"]
        fname = os.path.join(gen_dir, f"fit_{fit_name}.pkl")
        print(f"  Fitting {fit_name} (k={k})...")
        sim_fits[fit_name] = individual_fit(
            sim_data_list, mspec["func"], np.zeros(k), RECOVERY_PRIOR_VARIANCE,
            fname=fname, config={"verbose": False}
        )

    # ── (4): HBI across the 5 candidates ──
    fit_funcs = [MODELS[m]["func"] for m in RECOVERY_MODELS]
    fit_paths = [os.path.join(gen_dir, f"fit_{m}.pkl") for m in RECOVERY_MODELS]
    hbi_path = os.path.join(gen_dir, "hbi.pkl")
    print(f"  Running HBI over {len(RECOVERY_MODELS)} models...")
    hbi_res = hbi_main(sim_data_list, fit_funcs, fit_paths,
                       fname=hbi_path, config={"verbose": 0, "save_prog": False})
    hbi_res = hbi_null(sim_data_list, hbi_path)

    # ── (5): extract per-subject winners and frequencies ──
    resp = hbi_res.output.responsibility  # (N_SIM_PER_MODEL, 5)
    winners_idx = np.argmax(resp, axis=1)
    winners = [RECOVERY_MODELS[i] for i in winners_idx]

    recovery_results[gen_name] = {
        "true_params":     true_params,
        "responsibility":  resp,
        "winners":         winners,
        "winners_idx":     winners_idx,
        "model_frequency": hbi_res.output.model_frequency,
        "pxp":             hbi_res.output.protected_exceedance_prob,
        "sim_fits":        sim_fits,
        "hbi":             hbi_res,
        "donor_idx":       donor_idx,
        "param_idx":       param_idx,
    }

    # Quick summary printout
    from collections import Counter
    counts = Counter(winners)
    elapsed = time.time() - t0
    print(f"  -> {N_SIM_PER_MODEL} simulated subjects fit in {elapsed/60:.1f} min")
    for m in RECOVERY_MODELS:
        c = counts.get(m, 0)
        marker = "  <- TRUE" if m == gen_name else ""
        print(f"    {m:15s}: {c:3d} winners ({100*c/N_SIM_PER_MODEL:5.1f}%){marker}")

print("\n[RECOVERY] ✅ All generating models done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9D  MODEL-RECOVERY CONFUSION MATRIX  (row = generating, col = recovered)
# ══════════════════════════════════════════════════════════════════════

K = len(RECOVERY_MODELS)
confusion = np.zeros((K, K), dtype=int)
for i, gen_name in enumerate(RECOVERY_MODELS):
    winners_idx = recovery_results[gen_name]["winners_idx"]
    for j in range(K):
        confusion[i, j] = int(np.sum(winners_idx == j))

confusion_prop = confusion / confusion.sum(axis=1, keepdims=True)

# ── Save CSV ──
conf_csv = os.path.join(RECOVERY_DIR, "confusion_matrix.csv")
import pandas as _pd
_pd.DataFrame(confusion_prop, index=RECOVERY_MODELS, columns=RECOVERY_MODELS)\
    .to_csv(conf_csv, float_format="%.3f")
print(f"[RECOVERY] saved: {conf_csv}")

# ── Plot heatmap ──
fig, ax = plt.subplots(figsize=(7.5, 6.3))
im = ax.imshow(confusion_prop, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(RECOVERY_MODELS, rotation=35, ha="right")
ax.set_yticklabels(RECOVERY_MODELS)
ax.set_xlabel("Recovered model (HBI winner)", fontweight="bold")
ax.set_ylabel("Generating model", fontweight="bold")
ax.set_title(f"Model Recovery — {FILTER_LABEL}\n(N={N_SIM_PER_MODEL} per generating model)",
             fontweight="bold")
for i in range(K):
    for j in range(K):
        val = confusion_prop[i, j]
        count = confusion[i, j]
        color = "white" if val > 0.45 else "black"
        ax.text(j, i, f"{val:.2f}\n({count})", ha="center", va="center",
                color=color, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.85, label="P(recovered | generating)")
plt.tight_layout()
fig_path = os.path.join(RECOVERY_DIR, "confusion_matrix.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"[RECOVERY] saved: {fig_path}")

# ── Print key recovery diagnostics ──
print("\nDiagonal recoveries (P(recovered = generating)):")
for i, m in enumerate(RECOVERY_MODELS):
    print(f"  {m:15s}: {confusion_prop[i, i]:.2f}")
print(f"\nMean diagonal: {np.mean(np.diag(confusion_prop)):.2f}")
print(f"Worst off-diagonal confusion: {np.max(confusion_prop - np.eye(K) * confusion_prop):.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9E  PARAMETER RECOVERY — recovered vs true, per model & parameter
# ══════════════════════════════════════════════════════════════════════
# For each generating model we pull the FITTED parameters of that same
# model (from the CBM individual_fit it ran against the simulated data of its
# own generator) and scatter them against the TRUE parameters we sampled in.
# This is the cleanest parameter-identifiability check — it ignores the
# model-comparison question (handled by the confusion matrix above) and
# asks just: "if we know the right model, can we recover its parameters?"

from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(len(RECOVERY_MODELS), 4, figsize=(14, 3.1 * len(RECOVERY_MODELS)),
                         squeeze=False)
for row, gen_name in enumerate(RECOVERY_MODELS):
    rec = recovery_results[gen_name]
    true_p = rec["true_params"]
    fit_result = rec["sim_fits"][gen_name]
    fit_p = fit_result.output.parameters
    names = MODELS[gen_name]["names"]
    k = true_p.shape[1]

    for col in range(4):
        ax = axes[row][col]
        if col >= k:
            ax.axis("off")
            continue
        t = true_p[:, col]
        r = fit_p[:, col]
        rho_p, pval_p = pearsonr(t, r)
        rho_s, _     = spearmanr(t, r)
        ax.scatter(t, r, s=14, alpha=0.55, edgecolor="none")
        lims = [min(t.min(), r.min()), max(t.max(), r.max())]
        ax.plot(lims, lims, "k--", lw=0.9, alpha=0.6, label="y = x")
        ax.set_xlabel(f"True {names[col]}")
        ax.set_ylabel(f"Recovered {names[col]}")
        ax.set_title(f"{gen_name} - {names[col]}\nrho_P={rho_p:.2f}  rho_S={rho_s:.2f}", fontsize=9)
        if col == 0:
            ax.text(-0.28, 0.5, gen_name, transform=ax.transAxes,
                    rotation=90, va="center", ha="center",
                    fontsize=11, fontweight="bold")

fig.suptitle(f"Parameter Recovery - {FILTER_LABEL}",
             fontsize=13, fontweight="bold", y=1.005)
plt.tight_layout()
pr_path = os.path.join(RECOVERY_DIR, "param_recovery.png")
plt.savefig(pr_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"[RECOVERY] saved: {pr_path}")

import pandas as _pd
print("\nParameter recovery summary (Pearson r of true vs recovered):")
rows = []
for gen_name in RECOVERY_MODELS:
    rec = recovery_results[gen_name]
    true_p = rec["true_params"]
    fit_p = rec["sim_fits"][gen_name].output.parameters
    names = MODELS[gen_name]["names"]
    for j, pname in enumerate(names):
        r, p = pearsonr(true_p[:, j], fit_p[:, j])
        rows.append({"model": gen_name, "parameter": pname, "r_pearson": r, "p_value": p})
        print(f"  {gen_name:15s} {pname:12s}: r = {r:+.3f}  (p = {p:.3g})")

pr_summary = _pd.DataFrame(rows)
pr_csv = os.path.join(RECOVERY_DIR, "param_recovery_summary.csv")
pr_summary.to_csv(pr_csv, index=False, float_format="%.4f")
print(f"[RECOVERY] saved: {pr_csv}")


---
## Section 2 — Bayesian hierarchical LMM predicting log(delib RT) — **MB subjects only**

Per user (2026-04-28) the LMM is fit **only on MB subjects** (n = 67).
The previous group-specific intercept population (MB vs non-MB) is
removed since there is now only one group, and `ev_pos` (the binary
EV-vs-offer sign indicator) has been **removed** from predictors.

### Model

For trial $t$ of MB-subject $i$:

$$
\begin{aligned}
\log\,\text{delib}_{t} &\sim \mathcal{N}(\mu_t,\,\sigma_y) \\
\mu_t &= \alpha_i + \beta^{q2}_i\,\text{is\_q2}_t + \beta^{\Delta}_i\,\text{is\_triangle}_t
       + \beta^{|ev|}_i\,z|ev\!-\!offer|_t + \beta^{acc}\,\text{correct}_t \\
\alpha_i &\sim \mathcal{N}(\mu^{\alpha},\,\sigma^{\alpha}) \\
\beta^{\bullet}_i &\sim \mathcal{N}(\mu^{\bullet},\,\sigma^{\bullet})
\end{aligned}
$$

Sampled with PyMC + nutpie (1000 tune + 1000 draw × 4 chains).

Per subject we extract posterior means of every random effect
($\alpha_i$, $\beta^{q2}_i$, $\beta^{\Delta}_i$, $\beta^{|ev|}_i$) and
correlate each with **somatic_anxiety** and **worry**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2A.  BUILD LMM INPUT FRAME (PyMC) — canonical EV, log(delib_s) outcome,
#       restricted to MB subjects only.
# ══════════════════════════════════════════════════════════════════════
# Predictors retained: is_q2, is_triangle, z_abs_ev_diff, correct_canon.
# `ev_pos` removed per user request.
def _z(s, mu=None, sd=None):
    if mu is None: mu = s.mean()
    if sd is None: sd = s.std()
    return (s - mu) / (sd if sd else 1.0)

# Attach mb_group from subj_meta and KEEP ONLY MB subjects.
mb_lookup = subj_meta.set_index('subject_id')['mb_group']
lmm_canon = rt_trial.dropna(
    subset=['log_delib','choice','ev_diff','correct_canon']).copy()
lmm_canon['mb_group'] = lmm_canon['subject_id'].astype(str).map(mb_lookup)
lmm_canon = lmm_canon[lmm_canon['mb_group'] == 'MB'].copy()

# Standardise predictors AFTER restricting to MB so the z-scoring is
# centred on the MB sample.
lmm_canon['z_abs_ev_diff'] = _z(lmm_canon['abs_ev_diff'])
# correct_canon ∈ {0, 0.5, 1}; centre at 0.5 so the intercept stays at
# "baseline log-delib for an at-chance trial".
lmm_canon['correct_c'] = lmm_canon['correct_canon'] - 0.5

print(f"LMM input — MB subjects only:")
print(f"  {len(lmm_canon)} trials, {lmm_canon['subject_id'].nunique()} subjects")
print(f"  correct_canon distribution: 0={int((lmm_canon['correct_canon']==0).sum())}, "
      f"0.5={int((lmm_canon['correct_canon']==0.5).sum())}, "
      f"1={int((lmm_canon['correct_canon']==1).sum())}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2B.  FIT THE PyMC HIERARCHICAL MODEL — MB SUBJECTS ONLY
#       log_delib_t ~ N(alpha_i + sum(beta_i * X_t) + b_acc * correct_c_t, sigma_y)
#       alpha_i, beta_i ~ N(mu_*, sigma_*)   single (MB) population
# ══════════════════════════════════════════════════════════════════════
import pytensor, os
def _can_compile_cxx():
    try:
        import pytensor.tensor as pt
        x = pt.dvector('x')
        f = pytensor.function([x], x.sum())
        f([1.0, 2.0])
        return True
    except Exception:
        return False
if not _can_compile_cxx():
    print("PyTensor C++ compilation failed -- using Python VM (slower).")
    pytensor.config.cxx = ""
else:
    print("PyTensor C++ compilation OK.")

import pymc as pm
import arviz as az

subj_codes, subj_labels = pd.factorize(lmm_canon['subject_id'])
n_subj = len(subj_labels)
print(f"MB-only LMM: {len(lmm_canon)} trials, {n_subj} subjects")

X_q2     = lmm_canon['is_q2'].values.astype(float)
X_tri    = lmm_canon['is_triangle'].values.astype(float)
X_evdif  = lmm_canon['z_abs_ev_diff'].values.astype(float)
X_acc    = lmm_canon['correct_c'].values.astype(float)
y        = lmm_canon['log_delib'].values.astype(float)

with pm.Model() as model:
    # ── Single INTERCEPT population (MB only) ──
    mu_int    = pm.Normal('mu_int', mu=0., sigma=2.)
    sigma_int = pm.HalfNormal('sigma_int', sigma=1.)
    z_alpha   = pm.Normal('z_alpha', mu=0., sigma=1., shape=n_subj)
    alpha     = pm.Deterministic('alpha', mu_int + sigma_int * z_alpha)

    # ── Subject-level random slopes for trial features ──
    def _slope(name):
        mu_b    = pm.Normal(f'mu_{name}', mu=0., sigma=1.)
        sigma_b = pm.HalfNormal(f'sigma_{name}', sigma=0.5)
        z_b     = pm.Normal(f'z_{name}', mu=0., sigma=1., shape=n_subj)
        return pm.Deterministic(name, mu_b + sigma_b * z_b)

    b_q2    = _slope('b_q2')
    b_tri   = _slope('b_tri')
    b_evdif = _slope('b_evdif')

    # ── Fixed effect of accuracy ──
    b_acc   = pm.Normal('b_acc', mu=0., sigma=1.)

    # ── Linear predictor + Gaussian likelihood ──
    mu_t = (alpha[subj_codes]
            + b_q2[subj_codes]    * X_q2
            + b_tri[subj_codes]   * X_tri
            + b_evdif[subj_codes] * X_evdif
            + b_acc               * X_acc)
    sigma_y = pm.HalfNormal('sigma_y', sigma=1.)
    pm.Normal('y_obs', mu=mu_t, sigma=sigma_y, observed=y)

    try:
        import nutpie  # noqa: F401
        _SAMPLER = 'nutpie'
    except ImportError:
        _SAMPLER = 'pymc'
    print(f"Sampling with nuts_sampler='{_SAMPLER}'")
    idata = pm.sample(draws=1000, tune=1000, chains=4, cores=4,
                      target_accept=0.92, random_seed=20260428,
                      nuts_sampler=_SAMPLER, progressbar=False)

print(idata.sample_stats['diverging'].sum().values, "divergences")

summ_pop = az.summary(
    idata,
    var_names=['mu_int','sigma_int',
               'mu_b_q2','sigma_b_q2',
               'mu_b_tri','sigma_b_tri',
               'mu_b_evdif','sigma_b_evdif',
               'b_acc','sigma_y'],
    hdi_prob=0.95)
print("\n=== Population-level posterior summary (MB-only) ===")
print(summ_pop.round(3).to_string())

idata.to_netcdf("pymc_lmm_log_delib_MBonly.nc")
print("\nSaved pymc_lmm_log_delib_MBonly.nc")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2C.  EXTRACT POSTERIOR MEAN INTERCEPT + Triangle SLOPE PER SUBJECT
#       Relate to BOTH somatic_anxiety AND worry.
# ══════════════════════════════════════════════════════════════════════
post = idata.posterior

# Per-subject posterior means (MB-only fit)
alpha_mean = post['alpha'].mean(dim=('chain','draw')).values
btri_mean  = post['b_tri'].mean(dim=('chain','draw')).values
bq2_mean   = post['b_q2'].mean(dim=('chain','draw')).values
bevdif_m   = post['b_evdif'].mean(dim=('chain','draw')).values

blups = pd.DataFrame({
    'subject_id'        : subj_labels.astype(str),
    'intercept'         : alpha_mean,
    'b_q2'              : bq2_mean,
    'b_triangle'        : btri_mean,
    'b_z_abs_ev_diff'   : bevdif_m,
    'intercept_triangle': alpha_mean + btri_mean,
})
for anx_col in ['somatic_anxiety','worry']:
    blups[anx_col] = blups['subject_id'].map(
        subj_meta.set_index('subject_id')[anx_col])

print(f"Per-subject posterior means: {len(blups)} MB subjects")
print()
print("=== Random-effect × {somatic_anxiety, worry} (MB only) ===")
print(f"  {'param':>22s}  rho_S    p_S         |  rho_W    p_W")
re_summary_rows = []
PARAMS = ['intercept','b_q2','b_triangle','b_z_abs_ev_diff','intercept_triangle']
for col in PARAMS:
    g = blups.dropna(subset=['somatic_anxiety','worry', col])
    rs_s, p_s = spearmanr(g['somatic_anxiety'], g[col])
    rs_w, p_w = spearmanr(g['worry'],           g[col])
    rp_s, pp_s = pearsonr(g['somatic_anxiety'], g[col])
    rp_w, pp_w = pearsonr(g['worry'],           g[col])
    star_s = '*' if p_s<.05 else ('.' if p_s<.10 else '')
    star_w = '*' if p_w<.05 else ('.' if p_w<.10 else '')
    print(f"  {col:>22s}  {rs_s:+.3f}  {p_s:.4g}{star_s:>2s}    |  "
          f"{rs_w:+.3f}  {p_w:.4g}{star_w:>2s}")
    re_summary_rows.append({'param':col, 'n':len(g),
                            'rho_somatic':rs_s, 'p_somatic':p_s,
                            'r_somatic':rp_s,   'p_r_somatic':pp_s,
                            'rho_worry':rs_w,   'p_worry':p_w,
                            'r_worry':rp_w,     'p_r_worry':pp_w})

pd.DataFrame(re_summary_rows).to_csv("rt_analyses_lmm_re_anxiety.csv", index=False)
blups.to_csv("rt_analyses_lmm_blups.csv", index=False)
print("\nSaved rt_analyses_lmm_re_anxiety.csv  &  rt_analyses_lmm_blups.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2D.  PLOT posterior intercept and intercept+triangle vs each anxiety
#       Plus posterior of mu_int[MB] - mu_int[non-MB] contrast.
# ══════════════════════════════════════════════════════════════════════
BG_COLOR = "#FAFBFC"
MB_COLOR = "#2ECC71"

def _scatter_one(ax, rt_df, anx_col, ycol, axis_label, title, color=MB_COLOR):
    ax.set_facecolor(BG_COLOR)
    g = rt_df.dropna(subset=[anx_col, ycol])
    if len(g) < 5:
        ax.set_title(f"{title}: n={len(g)} (too few)"); return
    ax.scatter(g[anx_col], g[ycol], alpha=.7, s=42,
               color=color, edgecolors='white', linewidths=0.6)
    m, b = np.polyfit(g[anx_col], g[ycol], 1)
    xr = np.linspace(g[anx_col].min(), g[anx_col].max(), 50)
    ax.plot(xr, m*xr+b, color=color, lw=2)
    rs, ps = spearmanr(g[anx_col], g[ycol])
    rp, pp = pearsonr (g[anx_col], g[ycol])
    star = '***' if ps<.001 else '**' if ps<.01 else '*' if ps<.05 else ''
    ax.set_xlabel(anx_col); ax.set_ylabel(axis_label)
    ax.set_title(f"{title}\n"
                 f"Spearman rho={rs:+.3f}, p={ps:.3g} {star}  |  "
                 f"Pearson r={rp:+.3f}, p={pp:.3g}",
                 fontweight='bold', fontsize=9.5)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# 2 anxieties × 5 random effects = 2 figures, 1 × 5 grid each
RE_PARAMS = [
    ('intercept',          'alpha_i (baseline log-delib)'),
    ('b_q2',               'b_q2_i (Q2 vs Q1 slope)'),
    ('b_triangle',         'b_triangle_i (Triangle vs Feather)'),
    ('b_z_abs_ev_diff',    'b_evdif_i (RT × difficulty slope)'),
    ('intercept_triangle', 'alpha + b_triangle (Triangle baseline)'),
]
for anx_col in ['somatic_anxiety','worry']:
    fig, axes = plt.subplots(1, len(RE_PARAMS),
                             figsize=(5.2*len(RE_PARAMS), 4.5),
                             facecolor=BG_COLOR)
    for ax, (col_name, ylab) in zip(axes, RE_PARAMS):
        _scatter_one(ax, blups, anx_col, col_name, ylab,
                     f"{col_name} × {anx_col}  (n={len(blups)} MB)", MB_COLOR)
    fig.suptitle(f"MB-only LMM: per-subject random effects × {anx_col}",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = f"fig_section2_lmm_re_{anx_col}.png"
    plt.savefig(out, dpi=160, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print(f"Saved {out}")

blups.to_csv("rt_analyses_lmm_blups.csv", index=False)
print("\nSaved rt_analyses_lmm_blups.csv")

---
## Section 3 — DDM in MB subjects only (simplified)

Why MB-only: the preregistered hypothesis (somatic × OT) lives in the
MB subgroup. The §2 worry findings live in non-MB but they don't
manifest as DDM parameter shifts (they're about the gap between
canonical EV and subjective rating, which the DDM doesn't model).
Restricting to MB subjects sharpens the test of the one mechanistic
claim the DDM can adjudicate: **does somatic anxiety raise the
boundary `a` in MB subjects?**

### Approach

Per MB subject, compute closed-form EZ-DDM parameters
(Wagenmakers, van der Maas & Grasman 2007) from accuracy +
correct-RT mean + correct-RT variance:

| Parameter | What it captures |
|---|---|
| **`v` — drift rate** | quality of evidence accumulation (higher = faster + more accurate) |
| **`a` — boundary separation** | response caution / decision threshold (higher = slower + more accurate) |
| **`Ter` — non-decision time** | encoding + motor time (higher = slower physical response, regardless of deliberation) |

Then test, in MB only, whether each parameter correlates with **worry**
and **somatic anxiety** (Spearman ρ, two-sided). One small Bayesian
regression per parameter to formalise: `param ~ worry + somatic`.

### A priori prediction

The somatic × OT effect in MB should be carried by **boundary `a`**
(extra caution), not by drift `v` (which would imply an accuracy
change). Worry should not show a clean DDM-parameter signature in MB
since worry's effects in this dataset live in non-MB on subjective
measures.

### ⚠ Reliability caveat (see §3b for parameter recovery)

A simulation of 300 synthetic subjects (true v, a, Ter known)
shows that at our n = 16 trials per subject:

* **`v` recovery: ρ = 0.88** — excellent, near-zero bias.
* **`a` recovery: ρ = 0.84** — very good, ~−0.08 downward bias.
* **`Ter` recovery: ρ = 0.28** — *essentially random*, with a
  large upward bias (+0.72 s).

So **inferences about individual differences in v and a are
trustworthy**, but **any Ter × anxiety effect at this N should be
discounted** — Ter is too noisily estimated. The recovery analysis
is in §3b.

EXPLORATORY — not preregistered.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.  EZ-DDM in MB subjects only  (simplified)
# ══════════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings('ignore', category=RuntimeWarning)
from scipy.stats import spearmanr

# ─── EZ-DDM closed-form (Wagenmakers, van der Maas & Grasman 2007) ───
def ez_ddm(rt_correct_s, n_trials, n_correct, s=0.1):
    '''Return dict(v, a, Ter) from accuracy + correct-RT. NaN if degenerate.'''
    if n_trials < 4 or len(rt_correct_s) < 2:
        return dict(v=np.nan, a=np.nan, Ter=np.nan)
    Pc = n_correct / n_trials
    if Pc >= 1.0: Pc = 1.0 - 1.0/(2*n_trials)
    if Pc <= 0.0: Pc = 0.0 + 1.0/(2*n_trials)
    if Pc == 0.5: Pc = 0.5 + 1.0/(2*n_trials)
    MRT = float(np.mean(rt_correct_s))
    VRT = float(np.var(rt_correct_s, ddof=1))
    if VRT <= 0: return dict(v=np.nan, a=np.nan, Ter=np.nan)
    L = np.log(Pc / (1.0 - Pc))
    v_arg = (L * (Pc*Pc*L - Pc*L + Pc - 0.5)) / VRT
    if v_arg <= 0: return dict(v=np.nan, a=np.nan, Ter=np.nan)
    v = np.sign(Pc - 0.5) * s * (v_arg ** 0.25)
    if v == 0: return dict(v=np.nan, a=np.nan, Ter=np.nan)
    a = (s * s * L) / v
    y = -v * a / (s * s)
    mdt = (a / (2.0 * v)) * ((1.0 - np.exp(y)) / (1.0 + np.exp(y)))
    return dict(v=float(v), a=float(a), Ter=float(MRT - mdt))

# ─── Compute per-subject params, then SUBSET TO MB ONLY ───
ddm_rows = []
for sid, g in rt_trial.groupby('subject_id'):
    g = g.dropna(subset=['rt_s','correct_canon'])
    n_trials = len(g)
    n_correct = int(g['correct_canon'].sum())
    rt_correct = g[g['correct_canon']==1]['rt_s'].values
    p = ez_ddm(rt_correct, n_trials, n_correct)
    p['subject_id'] = sid
    p['n_trials']   = n_trials
    p['n_correct']  = n_correct
    ddm_rows.append(p)
ddm_df = pd.DataFrame(ddm_rows)
ddm_df = ddm_df.merge(rt_per_subj[['subject_id','mb_group','worry','somatic_anxiety']],
                        on='subject_id', how='inner')

# ── MB-ONLY restriction ──
ddm_mb = ddm_df[ddm_df['mb_group'] == 'MB'].copy()
ddm_mb = ddm_mb.dropna(subset=['v','a','Ter','worry','somatic_anxiety'])
print(f"\nEZ-DDM in MB subjects only: n = {len(ddm_mb)}")

# Winsorize at 5th/95th to suppress one-or-two-subject outliers in EZ
for p in ['v','a','Ter']:
    x = ddm_mb[p]
    lo, hi = np.percentile(x.dropna(), [5, 95])
    ddm_mb[f'{p}_w'] = x.clip(lower=lo, upper=hi)

print("\nMB-only parameter descriptives (raw, then winsorized for analyses):")
for p in ['v','a','Ter']:
    x = ddm_mb[p]; xw = ddm_mb[f'{p}_w']
    print(f"  {p}: raw M={x.mean():+.3f} med={x.median():+.3f} range=[{x.min():+.2f},{x.max():+.2f}]  "
          f"→ winsor M={xw.mean():+.3f}")

ddm_mb.to_csv("rt_analyses_ez_ddm_MB.csv", index=False)
print("\nSaved rt_analyses_ez_ddm_MB.csv")

# ─── Spearman correlations: each DDM param × {worry, somatic} in MB ───
def _star(p):
    return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '.' if p<.10 else ' '

print("\n" + "="*82)
print("EZ-DDM in MB only:  parameter × {worry, somatic_anxiety}  (Spearman ρ)")
print("="*82)
# print(f"  {'param':>6s} {'n':>4s}  {'ρ_worry':>+8s} {'p':>8s}    "
#       f"{'ρ_somatic':>+10s} {'p':>8s}")
ez_corr_rows = []
for param in ['v','a','Ter']:
    col = f'{param}_w'
    g = ddm_mb.dropna(subset=['worry','somatic_anxiety', col])
    rs_w, pw = spearmanr(g['worry'],            g[col])
    rs_s, ps = spearmanr(g['somatic_anxiety'],  g[col])
    ez_corr_rows.append(dict(parameter=param, n=len(g),
                                rho_worry=rs_w, p_worry=pw,
                                rho_somatic=rs_s, p_somatic=ps))
    # print(f"  {param:>6s} {len(g):>4d}  "
    #       f"{rs_w:>+8.3f} {pw:>8.4g}{_star(pw):>3s}  "
    #       f"{rs_s:>+10.3f} {ps:>8.4g}{_star(ps):>3s}")
pd.DataFrame(ez_corr_rows).to_csv("rt_analyses_ez_ddm_MB_corr.csv", index=False)
print("\nSaved rt_analyses_ez_ddm_MB_corr.csv")

# ─── Simple Bayesian regression per parameter (MB-only): param ~ worry + somatic ───
import pymc as pm
import arviz as az

print("\n" + "="*78)
print("Bayesian regression (MB-only):  z(param) ~ z(worry) + z(somatic)")
print("="*78)

ddm_reg = ddm_mb.copy()
ddm_reg['z_worry']   = (ddm_reg['worry']           - ddm_reg['worry'].mean())           / ddm_reg['worry'].std()
ddm_reg['z_somatic'] = (ddm_reg['somatic_anxiety'] - ddm_reg['somatic_anxiety'].mean()) / ddm_reg['somatic_anxiety'].std()

results_3b = {}
for param in ['v','a','Ter']:
    y = ddm_reg[f'{param}_w'].values.astype(float)
    z_y = (y - y.mean()) / y.std()
    z_w = ddm_reg['z_worry'].values
    z_s = ddm_reg['z_somatic'].values
    with pm.Model() as m:
        intercept = pm.Normal('intercept', 0, 1)
        b_worry   = pm.Normal('b_worry',   0, 1)
        b_somatic = pm.Normal('b_somatic', 0, 1)
        sigma     = pm.HalfNormal('sigma', 1)
        mu_y = intercept + b_worry * z_w + b_somatic * z_s
        pm.Normal('obs', mu=mu_y, sigma=sigma, observed=z_y)
        idata = pm.sample(draws=600, tune=600, chains=2, cores=1,
                          random_seed=42, progressbar=False, target_accept=0.95)
    summ = az.summary(idata, var_names=['intercept','b_worry','b_somatic'],
                       hdi_prob=0.94, round_to=3)
    results_3b[param] = dict(idata=idata, summary=summ)
    print(f"\n--- {param} ---")
    print(summ.to_string())

# Save Bayesian summary
all_summ = []
for param, r in results_3b.items():
    s = r['summary'].copy(); s['parameter'] = param; s['effect'] = s.index
    all_summ.append(s.reset_index(drop=True))
all_summ = pd.concat(all_summ, ignore_index=True)[
    ['parameter','effect','mean','sd','hdi_3%','hdi_97%','r_hat']]
all_summ.to_csv("rt_analyses_ez_ddm_MB_bayes.csv", index=False)
print("\nSaved rt_analyses_ez_ddm_MB_bayes.csv")

# ─── Plots ───
BG = "#FAFBFC"

# Plot 1: parameter distributions (MB only)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor=BG)
for ax, param in zip(axes, ['v','a','Ter']):
    ax.set_facecolor(BG)
    x = ddm_mb[param].dropna().values
    ax.hist(x, bins=18, alpha=.85, color='#2ECC71',
             edgecolor='white', linewidth=0.5)
    ax.axvline(np.median(x), color='black', ls='--',
                label=f'median = {np.median(x):+.3f}')
    ax.set_xlabel(f'{param}'); ax.set_ylabel('# MB subjects')
    ax.set_title({'v':'drift rate (v)','a':'boundary (a)','Ter':'non-decision time (Ter)'}[param],
                  fontweight='bold', fontsize=11)
    ax.legend(fontsize=8); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.suptitle(f"Section 3 — EZ-DDM parameter distributions, MB only (n={len(ddm_mb)})",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section3_ddm_MB_distributions.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section3_ddm_MB_distributions.png")

# Plot 2: scatter of each param × {worry, somatic} in MB
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), facecolor=BG)
for col_i, param in enumerate(['v','a','Ter']):
    for row_i, pred in enumerate(['worry','somatic_anxiety']):
        ax = axes[row_i, col_i]; ax.set_facecolor(BG)
        g = ddm_mb.dropna(subset=[pred, param])
        ax.scatter(g[pred], g[param], alpha=.7, s=46, color='#2ECC71',
                    edgecolors='white', linewidths=0.5)
        m_, b_ = np.polyfit(g[pred], g[param], 1)
        xr = np.linspace(g[pred].min(), g[pred].max(), 30)
        ax.plot(xr, m_*xr + b_, color='#27ae60', lw=2)
        rs, pp = spearmanr(g[pred], g[param])
        ax.text(0.04, 0.94, transform=ax.transAxes,
                 s=f"ρ = {rs:+.3f}\np = {pp:.3f}",
                 fontsize=10, fontweight='bold' if pp<.05 else 'normal',
                 color='black' if pp<.05 else '#555555',
                 verticalalignment='top')
        ax.set_xlabel(pred); ax.set_ylabel(param)
        ax.set_title(f"{param} × {pred}", fontweight='bold', fontsize=10)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.suptitle(f"Section 3 — EZ-DDM × anxiety, MB only (n={len(ddm_mb)})",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section3_ddm_MB_scatter.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section3_ddm_MB_scatter.png")

# Plot 3: Bayesian regression posteriors (MB only)
fig, axes = plt.subplots(3, 1, figsize=(10, 5), facecolor=BG, sharex=True)
COEFFS = ['intercept','b_worry','b_somatic']
COEFF_COL = ['#7f7f7f','#fb6a4a','#08519c']
for ax, param in zip(axes, ['v','a','Ter']):
    ax.set_facecolor(BG)
    summ = results_3b[param]['summary']
    ys = np.arange(len(COEFFS))
    for y, c in zip(ys, COEFFS):
        m_ = summ.loc[c,'mean']
        lo = summ.loc[c,'hdi_3%']; hi = summ.loc[c,'hdi_97%']
        col = COEFF_COL[ys.tolist().index(y)]
        ax.plot([lo, hi], [y, y], color=col, lw=4, alpha=.9)
        ax.plot(m_, y, 'o', color=col, markersize=10,
                markeredgecolor='black', markeredgewidth=0.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_yticks(ys); ax.set_yticklabels(COEFFS)
    ax.invert_yaxis()
    ax.set_xlabel(f"94% HDI  (z-scored {param})")
    ax.set_title(f"{param}", fontweight='bold', fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.suptitle(f"Section 3 — Bayesian regression in MB:  z(param) ~ z(worry) + z(somatic)",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section3_ddm_MB_bayes.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section3_ddm_MB_bayes.png")

### 3b — Parameter recovery: are these EZ-DDM estimates trustworthy?

A reasonable concern: with only **16 query trials per subject**, can
EZ-DDM reliably recover individual differences in v, a, and Ter?
Below we run a parameter-recovery simulation:

1. Simulate 300 synthetic subjects with known (true) v, a, Ter values
   sampled from ranges similar to our observed MB participants.
2. For each, simulate 16 trials by Euler-Maruyama integration of a
   true Wiener diffusion process (the data-generating model EZ-DDM
   is supposed to invert).
3. Apply EZ-DDM to the simulated data and compare recovered vs true
   parameters.

Quality metrics:

* **Spearman ρ** between true and recovered values across simulated
  subjects (rank-recovery; what matters for our individual-difference
  correlations with anxiety).
* **RMSE** (absolute recovery accuracy).
* **Bias** (systematic over/under-estimation).

If recovery is poor, the §3 effects might be artefacts of estimator
noise rather than real anxiety-DDM relationships. Repeated at
n_trials = 32 and 64 to show how recovery improves with more trials.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3b. PARAMETER RECOVERY: simulate DDM trials, fit EZ, compare to truth
# ══════════════════════════════════════════════════════════════════════
import warnings; warnings.filterwarnings('ignore', category=RuntimeWarning)

# DDM trial simulator (Euler-Maruyama of the Wiener process)
def _sim_trial(v, a, Ter, s=0.1, dt=0.001, max_t=30.0, rng=None):
    if rng is None: rng = np.random
    x = a / 2.0   # unbiased start
    t = 0.0
    sqrt_dt = np.sqrt(dt)
    while 0 < x < a and t < max_t:
        x += v*dt + s*sqrt_dt*rng.standard_normal()
        t += dt
    return t + Ter, (1 if x >= a else 0)

def _run_recovery(N_SIM, N_TRIALS, rng):
    v_t = rng.uniform(0.005, 0.10, N_SIM)
    a_t = rng.uniform(0.15,  0.90, N_SIM)
    T_t = rng.uniform(0.20,  1.50, N_SIM)
    rows = []
    for i in range(N_SIM):
        rts = np.empty(N_TRIALS); chs = np.empty(N_TRIALS, dtype=int)
        for k in range(N_TRIALS):
            rt_, c_ = _sim_trial(v_t[i], a_t[i], T_t[i], rng=rng)
            rts[k] = rt_; chs[k] = c_
        rt_correct = rts[chs == 1]
        p = ez_ddm(rt_correct, N_TRIALS, int(chs.sum()))
        rows.append({'v_t':v_t[i],'a_t':a_t[i],'Ter_t':T_t[i],
                      'v_r':p['v'],'a_r':p['a'],'Ter_r':p['Ter']})
    return pd.DataFrame(rows)

# Run recovery at our N (16) and at 2x and 4x for context
print("Parameter-recovery simulation (this takes ~1-2 min)...")
rng_recov = np.random.default_rng(42)
recov_16 = _run_recovery(N_SIM=300, N_TRIALS=16, rng=rng_recov)
recov_32 = _run_recovery(N_SIM=200, N_TRIALS=32, rng=rng_recov)
recov_64 = _run_recovery(N_SIM=150, N_TRIALS=64, rng=rng_recov)

def _recovery_table(rt_df, n_trials):
    finite = rt_df.dropna(subset=['v_r','a_r','Ter_r'])
    print(f"\n  --- n_trials = {n_trials}  ({len(finite)}/{len(rt_df)} finite) ---")
    print(f"  {'param':>6s}  {'true M':>+8s}  {'recov M':>+9s}  {'bias':>+7s}  "
          f"{'RMSE':>7s}  {'Spear ρ':>+9s}  {'Pearson r':>+10s}")
    for p in ['v','a','Ter']:
        g = rt_df.dropna(subset=[f'{p}_r'])
        bias = (g[f'{p}_r'] - g[f'{p}_t']).mean()
        rmse = np.sqrt(((g[f'{p}_r'] - g[f'{p}_t'])**2).mean())
        rho  = spearmanr(g[f'{p}_t'], g[f'{p}_r'])[0]
        rp   = pearsonr (g[f'{p}_t'], g[f'{p}_r'])[0]
        print(f"  {p:>6s}  {g[f'{p}_t'].mean():>+8.3f}  {g[f'{p}_r'].mean():>+9.3f}  "
              f"{bias:>+7.3f}  {rmse:>7.3f}  {rho:>+9.3f}  {rp:>+10.3f}")

print("\n=== Recovery results (true vs recovered EZ-DDM) ===")
_recovery_table(recov_16, 16)
_recovery_table(recov_32, 32)
_recovery_table(recov_64, 64)

recov_16.to_csv("rt_analyses_ez_ddm_recovery.csv", index=False)
print("\nSaved rt_analyses_ez_ddm_recovery.csv  (n=16 detailed; n=32, 64 only printed)")

# ─── Plot: true vs recovered, with rho/RMSE annotations, for each n_trials ───
BG = "#FAFBFC"
fig, axes = plt.subplots(3, 3, figsize=(13, 12), facecolor=BG)
for col_i, (rt_df, n_t, color) in enumerate([(recov_16, 16, '#7570b3'),
                                            (recov_32, 32, '#1f77b4'),
                                            (recov_64, 64, '#08519c')]):
    for row_i, p in enumerate(['v','a','Ter']):
        ax = axes[row_i, col_i]; ax.set_facecolor(BG)
        g = rt_df.dropna(subset=[f'{p}_r'])
        ax.scatter(g[f'{p}_t'], g[f'{p}_r'], alpha=.55, s=22, color=color,
                    edgecolors='white', linewidths=0.3)
        # Identity line
        lo = min(g[f'{p}_t'].min(), g[f'{p}_r'].min())
        hi = max(g[f'{p}_t'].max(), g[f'{p}_r'].max())
        ax.plot([lo, hi], [lo, hi], color='black', lw=1, ls='--', alpha=.6)
        rho = spearmanr(g[f'{p}_t'], g[f'{p}_r'])[0]
        rmse = np.sqrt(((g[f'{p}_r'] - g[f'{p}_t'])**2).mean())
        bias = (g[f'{p}_r'] - g[f'{p}_t']).mean()
        ax.text(0.04, 0.96, transform=ax.transAxes,
                 s=f"ρ = {rho:+.2f}\nRMSE = {rmse:.3f}\nbias = {bias:+.3f}",
                 fontsize=10, fontweight='bold', verticalalignment='top',
                 color='black')
        ax.set_xlabel(f'{p}_true');  ax.set_ylabel(f'{p}_recovered')
        if row_i == 0:
            ax.set_title(f'n_trials = {n_t}', fontweight='bold', fontsize=11)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Section 3b — EZ-DDM parameter recovery  (300/200/150 simulated subjects per panel)",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section3b_ddm_recovery.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section3b_ddm_recovery.png")

### 5.5 — Generative drift-diffusion model: poor model vs wasteful over-deliberation

A purely descriptive "slower but equally accurate" pattern (5.3) cannot, by itself, tell
**compensation** (rational extra caution for a weaker signal) from **waste** (caution beyond
what pays). This generative DDM does.

**Model** (noise s=1; per the worked spec). For probe *t* of subject *i* with gap `d=|ΔEV|`:

* drift `v = k_i · d`, where `k_i` = evidence quality / model precision
* `P(correct | d) = sigmoid(a_i · k_i · d)`
* `E[delib | d] = (a_i / (2 k_i d)) · tanh(a_i k_i d / 2)`

Accuracy depends on the **product** `a·k`; RT depends on `a` and `k` **separately**, so varying
`d` identifies both (recovery confirmed: a within ~5%, k within ~10% at this trial count).

**Non-decision time is NOT fit.** Deliberation is `rt_s − motor_baseline` (= `delib_s`), i.e. `t0_i`
is **fixed** to the per-subject motor baseline. Any anxiety-related slowing must therefore load
on `a` (caution) or `k` (precision), not on a free peripheral term. Fit on **raw (non-log)** `delib_s`.

**Links to anxiety** (both scales, hierarchical / partial-pooled):

* `log k_i = a_k + bk·z(anx) + ε`   →  `bk < 0` = anxiety **degrades the model** (poorer tree)
* `log a_i = a_a + ba·z(anx) + ε`   →  `ba > 0` = anxiety **adds caution** (more deliberation)

**The clean test — Surplus.** A higher `a` can be rational compensation for low `k`. Compare each
subject's caution to the reward-rate-optimal caution for **their own** `k_i`:
`a*_i = argmax_a E_d[ P(correct) / (t0 + E[RT] + ITI) ]`, then `Surplus_i = log a_i − log a*_i`.

* `gamma_W > 0` (Surplus rises with anxiety) → **over-deliberation / waste**
* `gamma_W ≈ 0` with `bk < 0` → **poor model** (cautious exactly as much as a bad signal warrants)

**Fitting:** closed-form **two-stage penalized MLE** (per-subject fit with weak shrinkage
priors, then second-level regression) — needs only scipy + statsmodels, **no PyMC/pytensor**,
runs in seconds. This is the EZ/closed-form fallback noted in the spec. (A hierarchical-Bayes
PyMC version is available on request if the environment supports it.)

In [ ]:
# ═══ 5.5  GENERATIVE DDM (drift∝gap; t0 FIXED to motor baseline; raw delib_s) ═══
# No-PyMC implementation: closed-form two-stage penalized MLE (the EZ/closed-form
# fallback from the spec). Stage 1: per-subject (a_i,k_i) by maximum likelihood of
# accuracy + RT under the closed forms, with weak shrinkage priors centred on the
# group mean (partial-pooling-lite, re-estimated EM-style). Stage 2: regress
# log k_i and log a_i on z(somatic)+z(worry). t0 is NOT fit (deliberation = delib_s
# = rt_s - motor_baseline). Needs only scipy + statsmodels.
import numpy as np, pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit
import statsmodels.api as _sm

_dd = rt_trial.dropna(subset=['worry','somatic_anxiety','delib_s','abs_ev_diff','correct_canon']).copy()
_dd = _dd[(_dd['abs_ev_diff']>0) & (_dd['correct_canon']!=0.5)].copy()   # drop value-neutral ties
_subs=_dd['subject_id'].unique(); _si={s:i for i,s in enumerate(_subs)}
_dd['si']=_dd['subject_id'].map(_si); S=len(_subs)
_meta=(_dd.groupby('subject_id')
         .agg(worry=('worry','first'), som=('somatic_anxiety','first'),
              motor=('motor_s','first')).reindex(_subs))
zS=((_meta['som']  -_meta['som'].mean())  /_meta['som'].std()).values
zW=((_meta['worry']-_meta['worry'].mean())/_meta['worry'].std()).values
gd=[_dd.loc[_dd['si']==i,'abs_ev_diff'].values.astype(float)  for i in range(S)]
gc=[_dd.loc[_dd['si']==i,'correct_canon'].values.astype(float) for i in range(S)]
gr=[_dd.loc[_dd['si']==i,'delib_s'].values.astype(float)      for i in range(S)]
print(f"DDM input: S={S} subjects, {len(_dd)} trials, "
      f"d in [{_dd['abs_ev_diff'].min():.0f},{_dd['abs_ev_diff'].max():.0f}]")

def _Emu(a,k,d): x=a*k*d; return (a/(2*k*d))*np.tanh(x/2)
def _fit_subj(d,corr,delib,sigma,mu_la,mu_lk,s_la=1.0,s_lk=1.0):
    def nll(par):
        la,lk=par; a,k=np.exp(la),np.exp(lk)
        p=np.clip(expit(a*k*d),1e-6,1-1e-6)
        ll=(corr*np.log(p)+(1-corr)*np.log(1-p)).sum()
        mu=np.clip(_Emu(a,k,d),1e-3,None)
        ll+=(-0.5*((np.log(delib)-np.log(mu))/sigma)**2).sum()
        ll+=-0.5*((la-mu_la)/s_la)**2 - 0.5*((lk-mu_lk)/s_lk)**2   # shrinkage
        return -ll
    r=minimize(nll,[mu_la,mu_lk],method='Nelder-Mead',
               options=dict(xatol=1e-4,fatol=1e-4,maxiter=2000))
    return np.exp(r.x)   # (a_i, k_i)

# EM-style: refit all subjects, then update group means + residual sigma
mu_la,mu_lk,sigma = np.log(2.0), np.log(0.15), 0.6
for _it in range(3):
    A=np.zeros(S); K=np.zeros(S); res=[]
    for i in range(S):
        a,k=_fit_subj(gd[i],gc[i],gr[i],sigma,mu_la,mu_lk); A[i],K[i]=a,k
        mu=np.clip(_Emu(a,k,gd[i]),1e-3,None); res+=list(np.log(gr[i])-np.log(mu))
    mu_la,mu_lk=np.log(A).mean(),np.log(K).mean(); sigma=max(0.2,np.std(res))
a_hat,k_hat=A,K
print(f"  fitted: log a M={np.log(a_hat).mean():+.2f}, log k M={np.log(k_hat).mean():+.2f}, "
      f"RT-residual sigma={sigma:.2f}")

def _reg(y): return _sm.OLS(y,_sm.add_constant(np.c_[zS,zW])).fit()
def _p1(b,p): return p/2 if b>0 else 1-p/2
mk=_reg(np.log(k_hat)); ma=_reg(np.log(a_hat))
print("\nlog k_i ~ somatic + worry   (bk<0 => anxiety degrades the MODEL / precision):")
print(f"  somatic bk={mk.params[1]:+.3f} p={mk.pvalues[1]:.3f}   "
      f"worry bk={mk.params[2]:+.3f} p={mk.pvalues[2]:.3f}")
print("log a_i ~ somatic + worry   (ba>0 => anxiety adds CAUTION / deliberation):")
print(f"  somatic ba={ma.params[1]:+.3f} p={ma.pvalues[1]:.3f}   "
      f"worry ba={ma.params[2]:+.3f} p={ma.pvalues[2]:.3f}")

# Surplus = caution above the reward-rate optimum for each subject's own k
ITI=1.0
def _astar(ki,t0i,dd_):
    def negrr(aa):
        x=aa*ki*dd_
        return -np.mean(expit(x)/(t0i+(aa/(2*ki*dd_))*np.tanh(x/2)+ITI))
    return minimize_scalar(negrr,bounds=(0.05,25),method='bounded').x
_surp=np.array([np.log(a_hat[i]) - np.log(_astar(
                    k_hat[i],
                    float(_meta['motor'].iloc[i]) if np.isfinite(_meta['motor'].iloc[i]) else 0.4,
                    gd[i])) for i in range(S)])
ms=_reg(_surp)
print("\nSurplus (log a - log a*) ~ somatic + worry   [gamma>0 = wasteful over-deliberation]:")
print(f"  somatic gamma={ms.params[1]:+.3f}  1s p={_p1(ms.params[1],ms.pvalues[1]):.3f}")
print(f"  worry   gamma={ms.params[2]:+.3f}  1s p={_p1(ms.params[2],ms.pvalues[2]):.3f}")
pd.DataFrame(dict(subject_id=_subs,k=k_hat,a=a_hat,surplus=_surp,zSom=zS,zWorry=zW)
            ).to_csv("rt_analyses_ddm_generative.csv",index=False)
print("\nInterpretation: ba>0 with gamma>0 => over-deliberation (caution beyond what pays);")
print("                bk<0 with gamma~0 => poor model (cautious as a weak signal warrants).")
print("NOTE: a large RT-residual sigma means the RT channel constrains a-vs-k weakly")
print("      (separation leans on accuracy); review motor-correction floor if sigma is high.")
print("Saved rt_analyses_ddm_generative.csv")

---

# Section 4 · Validations & Sensitivity (heavy)

Sample descriptives, paradigm-validation LMMs, questionnaire structure, assumption checks, power, partials, robustness.

---
## 1.5 · V4-specific descriptives (threat induction)

These columns don't exist in V3 data. They capture the silent earned-speaking-time accumulation driven by sub-optimal MB choices.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# V4: earned-speaking-time + threat-induction descriptives
# ══════════════════════════════════════════════════════════════════════
import ast

# ── 1) Final earned speaking time distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(id_vars['earned_time'].dropna(), bins=12, color='#dc2626', edgecolor='white')
ax.axvline(30,  ls='--', color='gray', label='base (30 s)')
ax.axvline(300, ls='--', color='red',  label='ceiling (300 s)')
ax.set_xlabel('Final earned speaking time (s)')
ax.set_ylabel('Subjects')
ax.set_title(f'Distribution of earned time (N={id_vars.earned_time.notna().sum()})')
ax.legend()

# ── 2) earned_time vs anxiety ──
ax = axes[1]
plot_df = id_vars.dropna(subset=['earned_time','somatic_anxiety'])
ax.scatter(plot_df['somatic_anxiety'], plot_df['earned_time'], alpha=0.7, color='#1e3c72')
if len(plot_df) > 2:
    r, p = stats.spearmanr(plot_df['somatic_anxiety'], plot_df['earned_time'])
    ax.set_title(f'Earned time × Somatic Anxiety   ρ={r:.2f}  p={p:.3f}')
ax.set_xlabel('STAI-S (post)')
ax.set_ylabel('Final earned speaking time (s)')

plt.tight_layout()
plt.savefig(f'fig_{OUT_PREFIX}_earned_time.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3) Parse SpeakingTime_Log → penalty breakdown by phase ──
def parse_log(s):
    if pd.isna(s) or not isinstance(s, str): return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

log_records = []
for _, r in df.iterrows():
    sid = r['subject_id']
    for entry in parse_log(r.get('SpeakingTime_Log', '[]')):
        log_records.append({'subject_id': sid, **entry})
log_df = pd.DataFrame(log_records)

if len(log_df):
    print('Penalty events breakdown:')
    print(log_df.groupby('phase')['added'].agg(['count','sum']).to_string())
    print()
    print('Per-subject penalty events (mean ± sd):')
    per_subj = log_df.groupby('subject_id')['added'].agg(['count','sum'])
    print(per_subj.describe().to_string())
else:
    print('No penalty events in SpeakingTime_Log.')

# ── 4) Suboptimal flags from the per-trial Query_*_Suboptimal columns ──
subopt_cols = [c for c in df.columns if 'Suboptimal' in c]
df['n_suboptimal'] = df[subopt_cols].sum(axis=1)
df['n_suboptimal_q1'] = df[[c for c in subopt_cols if c.startswith('Query_1')]].sum(axis=1)
df['n_suboptimal_q2'] = df[[c for c in subopt_cols if c.startswith('Query_2')]].sum(axis=1)
print()
print(f"Suboptimal choices per subject (out of {N_QUERY_TRIALS*2}): "
      f"mean={df.n_suboptimal.mean():.2f}  median={df.n_suboptimal.median()}  range={df.n_suboptimal.min()}-{df.n_suboptimal.max()}")

# Pass the per-subject counts back into id_vars
id_vars = id_vars.merge(df[['subject_id','n_suboptimal','n_suboptimal_q1','n_suboptimal_q2']],
                        on='subject_id', how='left')


---
## 1 · Descriptives

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A  RATINGS ACROSS TIME
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

for i, stim in enumerate(['Feather', 'Triangle']):
    ax = axes[i]
    vals = df[[f'{stim}_R1', f'{stim}_R2', f'{stim}_R3']].values
    means = vals.mean(axis=0)
    sems  = vals.std(axis=0) / np.sqrt(len(df))
    timepoints = ['Post-MF\n(before reval)', 'Post-Reval 1', 'Post-Reval 2']
    
    # Spaghetti plot (faint individual lines)
    for row in vals:
        ax.plot([0,1,2], row, color='grey', alpha=0.08, lw=0.5)
    
    ax.errorbar([0,1,2], means, yerr=1.96*sems, marker='o', color='C0' if stim=='Feather' else 'C3',
                lw=2.5, capsize=5, markersize=8, zorder=5)
    ax.axhline(MF_MAP[stim], color='grey', ls='--', alpha=0.5, label=f'MF value ({MF_MAP[stim]})')
    ax.axhline(0, color='black', ls='-', alpha=0.2, lw=0.5)
    ax.set_xticks([0,1,2])
    ax.set_xticklabels(timepoints)
    ax.set_title(f'{stim} Ratings', fontsize=13, fontweight='bold')
    ax.set_ylabel('Rating') if i==0 else None
    ax.legend(fontsize=9)

fig.suptitle('Rating Trajectories Over Time', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print summary
for stim in ['Feather', 'Triangle']:
    print(f'\n{stim}:')
    for t, label in zip([1,2,3], ['Post-MF','Post-Reval1','Post-Reval2']):
        col = f'{stim}_R{t}'
        print(f'  {label}: M = {df[col].mean():.2f}, SD = {df[col].std():.2f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1B  QUERY PERFORMANCE: P(GAMBLE) BY EV DIFFERENCE
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for qi, q in enumerate([1, 2]):
    ax = axes[qi]
    sub = trial_df[trial_df['query_num'] == q]
    
    for stim, color in [('Feather','C0'), ('Triangle','C3')]:
        s = sub[sub.state == stim]
        grouped = s.groupby('ev_diff')['choice'].agg(['mean','sem','count']).reset_index()
        ax.errorbar(grouped['ev_diff'], grouped['mean'], yerr=1.96*grouped['sem'],
                    marker='o', label=stim, color=color, capsize=3)
    
    ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
    ax.axvline(0, color='grey', ls='--', alpha=0.4)
    ax.set_xlabel('EV difference (gamble EV − fixed offer)')
    ax.set_ylabel('P(choose gamble)') if qi==0 else None
    ax.set_title(f'Query {q} (Revaluation {q})', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylim(-0.05, 1.05)

fig.suptitle('Choice Curves by EV Difference', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Overall accuracy
print('\nOverall accuracy (chose optimally given MB EV):')
for q in [1, 2]:
    sub = trial_df[trial_df['query_num'] == q]
    for stim in ['Feather', 'Triangle']:
        s = sub[sub.state == stim]
        print(f'  Query {q} / {stim}: {s.correct.mean():.3f} (n_trials={len(s)})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1C  ACCURACY BY QUERY × STATE (BAR PLOT)
# ══════════════════════════════════════════════════════════════════════

acc_summary = trial_df.groupby(['query_num','state'])['correct'].agg(['mean','sem']).reset_index()

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(2)
w = 0.35
for i, stim in enumerate(['Feather','Triangle']):
    sub = acc_summary[acc_summary.state == stim]
    ax.bar(x + i*w, sub['mean'], w, yerr=1.96*sub['sem'],
           label=stim, color=['C0','C3'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1\n(Reval 1)', 'Query 2\n(Reval 2)'])
ax.set_ylabel('Accuracy (MB-optimal choice)')
ax.set_title('MB-Optimal Accuracy by Query × Stimulus', fontweight='bold')
ax.legend()
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 2 · Linear Mixed Models

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LMM UTILITIES (statsmodels implementation)
# ══════════════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


def _parse_lme4_formula(formula_str):
    """
    Parse an lme4-style formula string into fixed and random parts.

    Supports syntax like:
        y ~ x1 + x2 + (1 + x1 | group)
        y ~ 1 + x1 + (x1 | group)
        y ~ (1 + x1 + x2 | group)
        y ~ x1 + (0 + x1 | group)

    Rules
    -----
    - Terms inside (...|group) define RANDOM effects for that group.
      '1' inside means random intercept; '0' means suppress random intercept.
    - Terms OUTSIDE (...|...) define FIXED effects.
      '1' outside means fixed intercept (default); '0' means suppress it.
    - A term present ONLY inside the random part gets no fixed-effect column.
    - A term present in BOTH gets both a fixed slope and a random slope.

    Returns
    -------
    outcome        : str   — left-hand side variable name
    fixed_intercept: bool  — whether to include a fixed intercept
    fixed_terms    : list  — non-intercept fixed-effect term names
    random_spec    : dict  — {group_col: {'intercept': bool, 'slopes': [str]}}
    """
    formula_str = formula_str.strip()

    # ── Split LHS and RHS ─────────────────────────────────────────────
    lhs, rhs = formula_str.split('~', 1)
    outcome = lhs.strip()
    rhs = rhs.strip()

    # ── Extract all (terms | group) blocks ────────────────────────────
    random_blocks = re.findall(r'\(([^)]+)\)', rhs)

    # Remove random blocks from RHS to get fixed-only part
    fixed_rhs = re.sub(r'\([^)]+\)', '', rhs)
    # Clean up leftover operators
    fixed_rhs = re.sub(r'\+\s*\+', '+', fixed_rhs)
    fixed_rhs = fixed_rhs.strip().strip('+').strip()

    # ── Parse fixed terms ─────────────────────────────────────────────
    fixed_intercept = True   # default: include intercept
    fixed_terms = []

    if fixed_rhs:
        for tok in re.split(r'\+', fixed_rhs):
            tok = tok.strip()
            if tok in ('', '1'):
                fixed_intercept = True
            elif tok == '0':
                fixed_intercept = False
            else:
                fixed_terms.append(tok)

    # ── Parse random blocks ────────────────────────────────────────────
    # Each block: "1 + x1 + x2 | group"  or  "0 + x1 | group"
    random_spec = {}

    for block in random_blocks:
        if '|' not in block:
            raise ValueError(f"Random block '{block}' missing '|' separator.")
        re_terms_str, group = block.rsplit('|', 1)
        group = group.strip()

        re_intercept = True
        re_slopes = []

        for tok in re.split(r'\+', re_terms_str):
            tok = tok.strip()
            if tok in ('', '1'):
                re_intercept = True
            elif tok == '0':
                re_intercept = False
            else:
                re_slopes.append(tok)

        random_spec[group] = {
            'intercept': re_intercept,
            'slopes':    re_slopes,
        }

    return outcome, fixed_intercept, fixed_terms, random_spec


def _build_statsmodels_inputs(outcome, fixed_intercept, fixed_terms, random_spec):
    """
    Translate parsed formula components into the strings/dicts that
    BinomialBayesMixedGLM.from_formula() expects.

    BinomialBayesMixedGLM takes:
      - fixed_formula  : standard patsy formula for fixed effects
      - random_formula : dict {label: patsy_formula_string}
                         Each entry is one variance component (its own SD).
                         Intercept and each slope get separate entries.

    Returns
    -------
    fixed_formula  : str
    random_formula : dict
    """
    # Fixed formula
    fixed_rhs_parts = []
    if not fixed_intercept:
        fixed_rhs_parts.append('0')
    fixed_rhs_parts += fixed_terms
    fixed_formula = f"{outcome} ~ {' + '.join(fixed_rhs_parts) or '1'}"

    # Random formula — one entry per variance component
    # Intercept and each slope get their own entry (independent SDs)
    random_formula = {}
    for group, spec in random_spec.items():
        if spec['intercept']:
            random_formula[f'{group}_intercept'] = f'0 + C({group})'
        for slope in spec['slopes']:
            random_formula[f'{group}_{slope}'] = f'0 + C({group}):{slope}'

    return fixed_formula, random_formula


def logistic_lmm(df_long, formula, group_col=None, verbose=True):
    """
    Fit a logistic mixed-effects model using an lme4-style formula.

    The formula fully specifies what is fixed and what is random.
    No separate fixed_cols / group_col arguments needed.

    Supported syntax examples
    -------------------------
    'y ~ x1 + x2 + (1 + x1 | subject)'
        fixed intercept, fixed x1+x2, random intercept + random slope x1

    'y ~ x1 + (x2 | subject)'
        fixed intercept (default), fixed x1, random slope x2 only

    'y ~ 0 + x1 + (1 | subject)'
        NO fixed intercept, fixed x1, random intercept only

    'y ~ (1 + x1 + x2 | subject)'
        fixed intercept (default from patsy), random intercept + slopes x1 x2
        no extra fixed slopes (x1, x2 only modelled as random)

    Parameters
    ----------
    df_long   : pd.DataFrame — trial-level long-format data
    formula   : str          — lme4-style formula (see above)
    group_col : str or None  — ignored; kept for backwards compatibility
    verbose   : bool         — print results table

    Returns
    -------
    dict with keys:
        'results'        : pd.DataFrame — Parameter, Estimate, SE, z, p
        'beta'           : np.ndarray   — posterior means (fixed effects)
        'se'             : np.ndarray   — posterior SDs  (fixed effects)
        'sigma_u'        : dict         — {component_label: SD} per random term
        'random_effects' : np.ndarray   — all random coefficient posteriors
        'converged'      : bool
        'formula'        : str
        'model'          : fitted BayesMixedGLMResults object
    """
    if verbose:
        print(f'\n{"="*65}')
        print(f'Logistic GLMM: {formula}')
        print(f'{"="*65}')

    # ── Parse formula ──────────────────────────────────────────────────
    outcome, fixed_intercept, fixed_terms, random_spec = _parse_lme4_formula(formula)

    # ── Build statsmodels inputs ───────────────────────────────────────
    fixed_formula, random_formula = _build_statsmodels_inputs(
        outcome, fixed_intercept, fixed_terms, random_spec
    )

    if verbose:
        print(f'  Fixed formula : {fixed_formula}')
        print(f'  Random terms  : {list(random_formula.keys())}')

    n_obs  = len(df_long)
    n_subj = {g: df_long[g].nunique() for g in random_spec}

    # ── Fit ────────────────────────────────────────────────────────────
    model  = BinomialBayesMixedGLM.from_formula(
        fixed_formula,
        random_formula,
        data=df_long
    )
    result = model.fit_vb()

    # ── Fixed effects ──────────────────────────────────────────────────
    beta  = np.asarray(result.fe_mean)
    se    = np.asarray(result.fe_sd)
    names = list(model.exog_names)

    z      = beta / se
    p_vals = 2 * (1 - stats.norm.cdf(np.abs(z)))

    results_df = pd.DataFrame({
        'Parameter': names,
        'Estimate':  beta,
        'SE':        se,
        'z':         z,
        'p':         p_vals
    })

    # ── Variance components ────────────────────────────────────────────
    # vcp_mean is a vector of log(SD), one per entry in random_formula,
    # in the same order as random_formula keys
    vc_labels = list(random_formula.keys())
    sigma_u = {
        label: float(np.exp(vcp))
        for label, vcp in zip(vc_labels, result.vcp_mean)
    }

    re_values = np.asarray(result.vc_mean)

    # ── Print ──────────────────────────────────────────────────────────
    if verbose:
        print(f'\nN obs = {n_obs}')
        for g, n in n_subj.items():
            print(f'N {g} = {n}')
        print(f'\nRandom effect SDs:')
        for label, sd in sigma_u.items():
            print(f'  {label:35s}  σ={sd:.4f}')
        print(f'\nFixed Effects:')
        for _, row in results_df.iterrows():
            sig = ('***' if row['p'] < .001 else
                   '**'  if row['p'] < .01  else
                   '*'   if row['p'] < .05  else '')
            print(f"  {row['Parameter']:25s}  β={row['Estimate']:+.4f}"
                  f"  SE={row['SE']:.4f}  z={row['z']:+.3f}"
                  f"  p={row['p']:.4f} {sig}")
        print()
        print(result.summary())

    return {
        'results':        results_df,
        'beta':           beta,
        'se':             se,
        'sigma_u':        sigma_u,
        'random_effects': re_values,
        'converged':      True,
        'formula':        formula,
        'model':          result
    }

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2B  LMM: Query 2 only — Accuracy difference Feather vs Triangle
# ══════════════════════════════════════════════════════════════════════

q2_df = trial_df[trial_df['query_num'] == 2].copy()
q2_df['is_triangle'] = (q2_df['state'] == 'Triangle').astype(float)
q2_df['trial_total'] = q2_df['trial_total'] - 1
lmm_q2 = logistic_lmm(
    q2_df,
    'correct ~ 1+abs_ev_diff+is_triangle+trial_total+(1+abs_ev_diff+is_triangle+trial_total|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2C  LMM: correct ~ ev_diff + query + (1|subject)
# ══════════════════════════════════════════════════════════════════════

lmm_ev_df = trial_df.copy()
lmm_ev_df['query_c'] = lmm_ev_df['query_num'] - 1.5

lmm_ev = logistic_lmm(
    lmm_ev_df,
    'correct ~ 1+abs_ev_diff + query_c + (1+abs_ev_diff + query_c|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2D  LMM: Anti-MF trials — both queries
#     chose_mb ~ query + abs_ev_diff + is_triangle + (1|subject)
# ══════════════════════════════════════════════════════════════════════

anti_mf_lmm = trial_df[trial_df['is_anti_mf'].fillna(0) == 1].copy()
anti_mf_lmm = anti_mf_lmm.dropna(subset=['chose_mb'])
anti_mf_lmm['query_c'] = anti_mf_lmm['query_num'] - 1.5
anti_mf_lmm['is_triangle'] = (anti_mf_lmm['state'] == 'Triangle').astype(float)

print(f'Anti-MF trials: {len(anti_mf_lmm)} from {anti_mf_lmm.subject_id.nunique()} subjects')
print(f'  Query 1: {len(anti_mf_lmm[anti_mf_lmm.query_num==1])}')
print(f'  Query 2: {len(anti_mf_lmm[anti_mf_lmm.query_num==2])}')
print(f'  Mean chose_mb Q1: {anti_mf_lmm[anti_mf_lmm.query_num==1].chose_mb.mean():.3f}')
print(f'  Mean chose_mb Q2: {anti_mf_lmm[anti_mf_lmm.query_num==2].chose_mb.mean():.3f}')

lmm_anti = logistic_lmm(
    anti_mf_lmm,
    'chose_mb ~ 1+query_c+abs_ev_diff+is_triangle+(1+query_c+abs_ev_diff+is_triangle|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2E  LMM: Anti-MF trials — Query 1 only
# ══════════════════════════════════════════════════════════════════════

anti_mf_q1 = trial_df[trial_df['is_anti_mf'].fillna(0) == 1].copy()
anti_mf_q1 = anti_mf_q1.dropna(subset=['chose_mb'])
anti_mf_q1 = anti_mf_q1[anti_mf_q1['query_num'] == 1]
anti_mf_q1['ev_diff_z'] = (anti_mf_q1['ev_diff'] - anti_mf_q1['ev_diff'].mean()) / anti_mf_q1['ev_diff'].std()
anti_mf_q1['is_triangle'] = (anti_mf_q1['state'] == 'Triangle').astype(float)

print(f'Anti-MF Q1 trials: {len(anti_mf_q1)} from {anti_mf_q1.subject_id.nunique()} subjects')
print(f'  Mean chose_mb: {anti_mf_q1.chose_mb.mean():.3f}')

lmm_anti_q1 = logistic_lmm(
    anti_mf_q1,
    'chose_mb ~ 1+abs_ev_diff+is_triangle+(1+abs_ev_diff+is_triangle|subject_id)',
)

#  ANTI-MF PERFORMANCE ACROSS QUERIES — DESCRIPTIVE

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Overall anti-MF performance by query
ax = axes[0]
anti_summary = anti_mf_df.groupby(['query_num','state']).agg(
    mean_mb=('chose_mb','mean'),
    sem_mb=('chose_mb','sem')
).reset_index()

x = np.arange(2)
w = 0.35
for i, stim in enumerate(['Feather','Triangle']):
    s = anti_summary[anti_summary.state == stim]
    ax.bar(x + i*w, s['mean_mb'], w, yerr=1.96*s['sem_mb'],
           label=stim, color=['C0','C3'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1 (Reval 1)', 'Query 2 (Reval 2)'])
ax.set_ylabel('P(chose MB over MF)')
ax.set_title('Anti-MF Trial Performance', fontweight='bold')
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.legend()
ax.set_ylim(0, 1)

# Panel 2: By anxiety median split
ax = axes[1]
med_anx = trial_df['somatic_anxiety'].median()
anti_mf_df['anx_group'] = np.where(anti_mf_df['somatic_anxiety'] >= med_anx, 'High Somatic Anxiety', 'Low Somatic Anxiety')

anx_summary = anti_mf_df.groupby(['query_num','anx_group']).agg(
    mean_mb=('chose_mb','mean'),
    sem_mb=('chose_mb','sem')
).reset_index()

for i, ag in enumerate(['Low Somatic Anxiety', 'High Somatic Anxiety']):
    s = anx_summary[anx_summary.anx_group == ag]
    ax.bar(x + i*w, s['mean_mb'], w, yerr=1.96*s['sem_mb'],
           label=ag, color=['C4','C1'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1 (Reval 1)', 'Query 2 (Reval 2)'])
ax.set_ylabel('P(chose MB over MF)')
ax.set_title('Anti-MF by Somatic Anxiety (Median Split)', fontweight='bold')
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## 5 · Reaction Time Analyses

All RT analyses use **log(RT)**. N = 139 (no exclusions).

### Theoretical motivation
Larger |EV difference| between the gamble EV and the fixed offer means the correct choice is more obvious — there is less to gain from deliberation. Therefore:
1. **RT should be *faster* when |EV diff| is large** — obvious choices need less thought
2. **RT should be *slower* when |EV diff| is small** — close calls demand more deliberation
3. **RT should improve accuracy, but primarily when |EV diff| is large** — deliberation is productive when stakes are high and there's a discernible correct answer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5A  |EV diff| → log(RT) and RT × |EV diff| → accuracy
# ══════════════════════════════════════════════════════════════════════

# Work with anti-MF trials that have valid RT
rt_df = anti_mf_df[anti_mf_df['log_rt'].notna()].copy()
print(f'Anti-MF trials with valid log(RT): {len(rt_df)} from {rt_df.subject_id.nunique()} subjects')

# ── Standardize predictors ──
rt_df['abs_ev_diff_z'] = (rt_df['abs_ev_diff'] - rt_df['abs_ev_diff'].mean()) / rt_df['abs_ev_diff'].std()
rt_df['log_rt_z']      = (rt_df['log_rt'] - rt_df['log_rt'].mean()) / rt_df['log_rt'].std()
rt_df['anxiety_z']     = (rt_df['somatic_anxiety'] - rt_df['somatic_anxiety'].mean()) / rt_df['somatic_anxiety'].std()
rt_df['erq_r_z']       = (rt_df['erq_reappraisal'] - rt_df['erq_reappraisal'].mean()) / rt_df['erq_reappraisal'].std()
rt_df['query_c']       = rt_df['query_num'] - 1.5

# ══════════════════════════════════════════════════════════════════════
# MODEL 1: log(RT) ~ |EV diff|
# ══════════════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('MODEL 1: |EV diff| → log(RT)')
print('═' * 70)

# Hard vs Easy median split
median_abs = rt_df['abs_ev_diff'].median()
rt_df['difficulty'] = np.where(rt_df['abs_ev_diff'] <= median_abs, 'Hard', 'Easy')

cell_rt = rt_df.groupby('difficulty').agg(
    log_rt_mean=('log_rt', 'mean'), log_rt_sem=('log_rt', 'sem'),
    n=('log_rt', 'count')
).reset_index()
for _, row in cell_rt.iterrows():
    print(f'  {row["difficulty"]:>6s}: log_rt = {row["log_rt_mean"]:.3f} ± {row["log_rt_sem"]:.3f} (n={int(row["n"])})')

hard_rt = rt_df[rt_df['difficulty'] == 'Hard']['log_rt']
easy_rt = rt_df[rt_df['difficulty'] == 'Easy']['log_rt']
t_rt, p_rt = stats.ttest_ind(hard_rt, easy_rt, equal_var=False)
d_rt = (hard_rt.mean() - easy_rt.mean()) / np.sqrt((hard_rt.var() + easy_rt.var()) / 2)
print(f'\n  Hard vs Easy: t={t_rt:.3f}, p={p_rt:.4f}, d={d_rt:.3f}')

# Trial-level Spearman
r_ev_rt, p_ev_rt = stats.spearmanr(rt_df['abs_ev_diff'], rt_df['log_rt'])
print(f'  |EV diff| × log_rt (Spearman): r={r_ev_rt:.3f}, p={p_ev_rt:.4f}')

# OLS with clustered SEs
print('\n--- OLS: log_rt ~ abs_ev_diff_z + is_triangle + query_c (clustered SEs) ---')
ols_data = rt_df[['log_rt', 'abs_ev_diff_z', 'is_triangle', 'query_c', 'subject_id']].dropna()
X_ols = sm.add_constant(ols_data[['abs_ev_diff_z', 'is_triangle', 'query_c']])
ols_model = sm.OLS(ols_data['log_rt'], X_ols).fit(cov_type='cluster',
                                                      cov_kwds={'groups': ols_data['subject_id']})
print(ols_model.summary2().tables[1])

# ══════════════════════════════════════════════════════════════════════
# MODEL 2: accuracy ~ log_rt × |EV diff|
# RT helps decisions BUT only when |EV diff| is high
# ══════════════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('MODEL 2: accuracy ~ log_rt × |EV diff|')
print('═' * 70)

rt_df['log_rt_x_abs_ev'] = rt_df['log_rt_z'] * rt_df['abs_ev_diff_z']

lmm_rt_choice = logistic_lmm(
    rt_df,
    'chose_mb ~ 1 + log_rt_z + abs_ev_diff_z + log_rt_x_abs_ev '
    '+ is_triangle + query_c + (1 | subject_id)'
)

### 0A — Worry vs somatic_anxiety sanity check

We have two anxiety scales in the data:
* `Survey_ANXIETY_1_Total_Score` — 8-item sum, range 8–40 → **`worry`** (PSWQ-A).
* `Survey_ANXIETY_2_Total_Score` — 10-item sum, range 10–47 → **`somatic_anxiety`** (MASQ-AA).

Plot the two scales against each other to see whether they agree
throughout the range, or diverge at the high end. Every downstream
analysis is run on **both** `somatic_anxiety` AND `worry` separately —
we never collapse them into a single "anxiety" predictor.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 0A.  worry vs somatic_anxiety  (diagnostic, top-of-notebook)
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

g = rt_df[['worry','somatic_anxiety']].dropna()
rs, ps = spearmanr(g['worry'], g['somatic_anxiety'])
rp, pp = pearsonr (g['worry'], g['somatic_anxiety'])
print(f"N with both scales: {len(g)}")
print(f"Pearson  r = {rp:+.3f}  (p={pp:.3g})")
print(f"Spearman rho = {rs:+.3f}  (p={ps:.3g})")

# Identify "high-end" subjects on each scale = top quartile
hi_w = g['worry']           >= g['worry'].quantile(0.75)
hi_s = g['somatic_anxiety'] >= g['somatic_anxiety'].quantile(0.75)
both  = hi_w & hi_s
onlyW = hi_w & ~hi_s
onlyS = ~hi_w & hi_s
neith = ~hi_w & ~hi_s
print(f"\nTop-quartile classification (Q75 split on each scale):")
print(f"  Both high (worry+somatic_anxiety): {int(both.sum()):>3d}")
print(f"  High worry only:                   {int(onlyW.sum()):>3d}")
print(f"  High somatic_anxiety only:         {int(onlyS.sum()):>3d}")
print(f"  Neither high:                      {int(neith.sum()):>3d}")

# Plot: scatter, with rank-rank also as inset, plus highlight of high end
BG = "#FAFBFC"
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=BG)

# --- raw scale scatter ---
ax = axes[0]; ax.set_facecolor(BG)
ax.scatter(g.loc[neith,'worry'], g.loc[neith,'somatic_anxiety'],
            color='#bcbcbc', s=30, alpha=.6, label='neither top-Q', edgecolors='white', linewidths=0.4)
ax.scatter(g.loc[onlyW,'worry'], g.loc[onlyW,'somatic_anxiety'],
            color='#fb9a99', s=45, alpha=.85, label='high worry only',          edgecolors='white', linewidths=0.6)
ax.scatter(g.loc[onlyS,'worry'], g.loc[onlyS,'somatic_anxiety'],
            color='#a6cee3', s=45, alpha=.85, label='high somatic_anxiety only', edgecolors='white', linewidths=0.6)
ax.scatter(g.loc[both,'worry'],  g.loc[both,'somatic_anxiety'],
            color='#33a02c', s=55, alpha=.9,  label='both high',     edgecolors='white', linewidths=0.6)
# regression line
m_, b_ = np.polyfit(g['worry'], g['somatic_anxiety'], 1)
xr = np.linspace(g['worry'].min(), g['worry'].max(), 50)
ax.plot(xr, m_*xr + b_, color='black', lw=1.5, alpha=.7,
         label=f'OLS: y={m_:.2f}x+{b_:.2f}')
ax.axvline(g['worry'].quantile(0.75), color='red', ls=':', alpha=.5, label='Q75 cutoffs')
ax.axhline(g['somatic_anxiety'].quantile(0.75), color='red', ls=':', alpha=.5)
ax.set_xlabel("worry (PSWQ-A, 8 items)")
ax.set_ylabel("somatic_anxiety (MASQ-AA, 10 items)")
ax.set_title(f"Raw totals\nPearson r={rp:+.3f}, Spearman rho={rs:+.3f} (n={len(g)})",
              fontweight='bold', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# --- rank scatter (shows monotonic agreement, removes scaling) ---
ax = axes[1]; ax.set_facecolor(BG)
g2 = g.copy()
g2['rank_w'] = g2['worry'].rank()
g2['rank_s'] = g2['somatic_anxiety'].rank()
ax.scatter(g2['rank_w'], g2['rank_s'], s=28, alpha=.6, color='#7570b3',
            edgecolors='white', linewidths=0.4)
m_, b_ = np.polyfit(g2['rank_w'], g2['rank_s'], 1)
xr = np.linspace(g2['rank_w'].min(), g2['rank_w'].max(), 50)
ax.plot(xr, m_*xr + b_, color='black', lw=1.5, alpha=.7)
# mark identity line
ax.plot([0, len(g)], [0, len(g)], color='gray', ls='--', alpha=.5, label='identity (rank worry = rank somatic_anxiety)')
ax.axvline(0.75*len(g), color='red', ls=':', alpha=.5, label='Q75 cutoffs')
ax.axhline(0.75*len(g), color='red', ls=':', alpha=.5)
ax.set_xlabel("rank(worry)")
ax.set_ylabel("rank(somatic_anxiety)")
ax.set_title(f"Rank-rank (monotonic agreement)\nSpearman rho={rs:+.3f}",
              fontweight='bold', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# --- divergence: |z(worry) - z(somatic_anxiety)| as a function of mean(z)  ---
# This makes "high end divergence" visible directly.
ax = axes[2]; ax.set_facecolor(BG)
g3 = g.copy()
g3['z_w'] = (g3['worry']           - g3['worry'].mean())           / g3['worry'].std()
g3['z_s'] = (g3['somatic_anxiety'] - g3['somatic_anxiety'].mean()) / g3['somatic_anxiety'].std()
g3['mean_z']  = (g3['z_w'] + g3['z_s']) / 2
g3['delta_z'] = g3['z_w'] - g3['z_s']
ax.scatter(g3['mean_z'], g3['delta_z'], s=30, alpha=.6, color='#e7298a',
            edgecolors='white', linewidths=0.4)
ax.axhline(0, color='black', lw=1)
ax.axhline(g3['delta_z'].std(),  color='gray', ls=':', alpha=.6, label='+/- 1 SD of disagreement')
ax.axhline(-g3['delta_z'].std(), color='gray', ls=':', alpha=.6)
ax.axvline(g3['mean_z'].quantile(0.75), color='red', ls=':', alpha=.5,
            label='Q75 of mean(z)')
# Trendline of |delta_z| vs mean_z (Loess-like via low-order polyfit)
xr = np.linspace(g3['mean_z'].min(), g3['mean_z'].max(), 50)
abs_d = g3['delta_z'].abs()
m_, b_ = np.polyfit(g3['mean_z'], abs_d, 1)
ax.plot(xr, m_*xr + b_, color='black', lw=1.5, alpha=.6,
         label=f'|delta_z| ~ slope={m_:+.2f}*mean_z')
# Spearman rho between mean_z and |delta_z|: does spread grow at the high end?
rs_d, ps_d = spearmanr(g3['mean_z'], g3['delta_z'].abs())
ax.set_xlabel("Mean of (z(worry) + z(somatic_anxiety))/2")
ax.set_ylabel("Disagreement  (z(worry) - z(somatic_anxiety))")
ax.set_title(f"Divergence vs anxiety level\n"
             f"Spearman(|disagreement|, level) rho={rs_d:+.3f}, p={ps_d:.3g}",
              fontweight='bold', fontsize=11)
ax.legend(fontsize=8, loc='upper right')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("worry vs somatic_anxiety — sanity check (downstream analyses run on BOTH)",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_anxiety_scales_check.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print(f"\nSaved fig_anxiety_scales_check.png")
print(f"\nSpearman(level, |disagreement|) = {rs_d:+.3f}, p={ps_d:.3g}")
print(f"  Positive rho ↔ scales diverge MORE at high anxiety end.")
print(f"  Negative rho ↔ scales agree MORE at high anxiety end.")

### 0B — Factor structure of all three questionnaires

We have three self-report scales in the data:

| Code | Instrument | Items | Scale | Theoretically… |
|---|---|---|---|---|
| **ERQ** | Emotion Regulation Questionnaire (Gross & John 2003) | 10 | 1–7 | 2 subscales: **Reappraisal** {Q1, Q3, Q5, Q7, Q8, Q10} and **Suppression** {Q2, Q4, Q6, Q9} |
| **ANXIETY_1** = PSWQ-A | Penn State Worry Questionnaire — Abbreviated | 8 | 1–5 | Designed unidimensional (worry); Q8 = "I notice that I have been worrying" probes meta-awareness |
| **ANXIETY_2** = MASQ-AA | Mood and Anxiety Symptom Questionnaire — Anxious Arousal | 10 | 1–5 | Single somatic-arousal factor; content sub-clusters: respiratory/throat (Q1, Q5, Q7), trembling/motor (Q4, Q8, Q9), autonomic (Q3, Q6, Q10), vestibular (Q2) |

For each questionnaire we run:

1. Item distributions (Likert response patterns).
2. Inter-item Spearman correlation matrix + **Cronbach's α**.
3. **PCA**: scree, % variance per component, loadings on PC1–PC2.
4. **Hierarchical clustering** of items (avg. linkage on 1−ρ).
5. Compare data-driven 2-cluster solution to the theoretical
   subscales; compute Cronbach α on each theoretical subscale.
6. Save subscale scores to `df` for downstream use.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 0B.  FACTOR STRUCTURE OF ALL THREE QUESTIONNAIRES
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

# ─── Questionnaire definitions: items + content labels + theoretical subscales ───
QUESTIONNAIRES = {
    'ERQ': {
        'prefix': 'Survey_ERQ_Q',
        'scale_max': 7,
        'item_text': {
            1:  'change thinking → +emotion',     # Reappraisal
            2:  'keep emotions to myself',         # Suppression
            3:  'change thinking → -emotion',     # Reappraisal
            4:  'don\'t express +emotion',         # Suppression
            5:  'reframe stress to stay calm',    # Reappraisal
            6:  'control by not expressing',       # Suppression
            7:  'change way thinking → +emotion', # Reappraisal
            8:  'control by changing thinking',   # Reappraisal
            9:  'don\'t express -emotion',         # Suppression
            10: 'change way thinking → -emotion', # Reappraisal
        },
        'theoretical_subscales': {
            'Reappraisal': [1, 3, 5, 7, 8, 10],
            'Suppression': [2, 4, 6, 9],
        }
    },
    'PSWQ-A': {
        'prefix': 'Survey_ANXIETY_1_Q',
        'scale_max': 5,
        'item_text': {
            1: 'worries overwhelm me',
            2: 'many things make me worry',
            3: 'know I shouldn\'t but can\'t help',
            4: 'pressure → I worry',
            5: 'always worrying',
            6: 'finish task → worry about next',
            7: 'lifelong worrier',
            8: 'I NOTICE I have been worrying',  # meta-awareness
        },
        'theoretical_subscales': {
            'WorryContent': [1, 2, 3, 4, 5, 6, 7],
            'MetaWorry':    [8],
        }
    },
    'MASQ-AA': {
        'prefix': 'Survey_ANXIETY_2_Q',
        'scale_max': 5,
        'item_text': {
            1:  'short of breath',           # Respiratory
            2:  'dizzy / light-headed',       # Vestibular
            3:  'cold/sweaty hands',          # Autonomic
            4:  'shaky hands',                # Trembling
            5:  'trouble swallowing',         # Throat
            6:  'hot/cold spells',            # Autonomic
            7:  'felt like choking',          # Throat
            8:  'muscles twitched/trembled',  # Trembling
            9:  'trembling / shaking',        # Trembling
            10: 'dry mouth',                  # Autonomic
        },
        'theoretical_subscales': {
            'Respiratory_Throat': [1, 5, 7],
            'Trembling_Motor':    [4, 8, 9],
            'Autonomic':          [3, 6, 10],
            'Vestibular':         [2],
        }
    },
}

def cronbach_alpha(X):
    X = np.asarray(X, dtype=float)
    k = X.shape[1]
    if k < 2: return np.nan
    return float(k/(k-1) * (1 - X.var(axis=0, ddof=1).sum() / X.sum(axis=1).var(ddof=1)))

BG = "#FAFBFC"
all_results = {}

for q_name, q_spec in QUESTIONNAIRES.items():
    print("\n" + "="*78)
    print(f"  {q_name}")
    print("="*78)
    items_cols = sorted(
        [c for c in rt_df.columns if c.startswith(q_spec['prefix']) and c.endswith('_Raw')],
        key=lambda c: int(c.split('_Q')[1].split('_')[0]))
    item_nums = [int(c.split('_Q')[1].split('_')[0]) for c in items_cols]
    labels = [f"Q{n}" for n in item_nums]
    text_lookup = q_spec['item_text']
    full_labels = [f"Q{n}: {text_lookup.get(n, '?')}" for n in item_nums]

    W = rt_df[items_cols].dropna().copy()
    W.columns = labels
    n_subj = len(W)
    print(f"  N items: {len(items_cols)}, N subjects with full responses: {n_subj}")

    # (1) Item content + means
    print(f"\n  Item content (mean / SD):")
    for n, lab, col in zip(item_nums, labels, items_cols):
        m = W[lab].mean(); sd = W[lab].std()
        print(f"    {lab}  μ={m:.2f} σ={sd:.2f}  '{text_lookup.get(n,'?')}'")

    # (2) Inter-item correlation + Cronbach's α
    corr_S = W.corr(method='spearman').values
    alpha_full = cronbach_alpha(W.values)
    print(f"\n  Cronbach's α (all {len(items_cols)} items): {alpha_full:.3f}")

    # (3) PCA
    Z = (W - W.mean()) / W.std()
    U, sv, Vt = np.linalg.svd(Z.values, full_matrices=False)
    eigvals = (sv**2) / (n_subj - 1)
    pct_var = eigvals / eigvals.sum()
    loadings = (Vt.T * sv) / np.sqrt(n_subj - 1)
    print(f"  PCA eigenvalues: {[f'{v:.2f}' for v in eigvals[:5]]}{'...' if len(eigvals)>5 else ''}")
    print(f"  %% variance:     {[f'{p*100:.1f}%' for p in pct_var[:5]]}{'...' if len(pct_var)>5 else ''}")
    print(f"  PC1 = {pct_var[0]*100:.1f}%, PC1+PC2 = {(pct_var[0]+pct_var[1])*100:.1f}%")
    print(f"  Loadings on PC1, PC2:")
    for i, lab in enumerate(labels):
        print(f"    {lab}: PC1={loadings[i,0]:+.3f}  PC2={loadings[i,1]:+.3f}  "
              f"text='{text_lookup.get(item_nums[i],'?')}'")

    # (4) Hierarchical clustering
    dist_mat = 1 - corr_S
    np.fill_diagonal(dist_mat, 0); dist_mat = (dist_mat + dist_mat.T)/2
    condensed = squareform(dist_mat, checks=False)
    Z_link = linkage(condensed, method='average')

    # (5) Compare to theoretical subscales
    print(f"\n  Theoretical subscales (per Gross & John 2003 / standard MASQ-AA usage):")
    for sub_name, sub_items in q_spec['theoretical_subscales'].items():
        sub_cols = [f"Q{n}" for n in sub_items if n in item_nums]
        if len(sub_cols) >= 2:
            a = cronbach_alpha(W[sub_cols].values)
            print(f"    {sub_name}: items {sub_items} → Cronbach α = {a:.3f}, "
                  f"score range [{W[sub_cols].sum(axis=1).min()}, {W[sub_cols].sum(axis=1).max()}]")
        else:
            print(f"    {sub_name}: items {sub_items}  (single-item or n<2; α n/a)")

    # Save subscale scores to rt_df
    df_score_cols = []
    for sub_name, sub_items in q_spec['theoretical_subscales'].items():
        sub_cols_orig = [c for c in items_cols
                         if int(c.split('_Q')[1].split('_')[0]) in sub_items]
        if sub_cols_orig:
            new_col = f"{q_name.replace('-','')}_{sub_name}"
            rt_df[new_col] = rt_df[sub_cols_orig].sum(axis=1)
            df_score_cols.append(new_col)
    print(f"  → Saved subscale scores: {df_score_cols}")

    # (6) Compare data-driven 2-cluster partition vs theoretical primary split
    cid2 = fcluster(Z_link, t=2, criterion='maxclust')
    sub_a = [lab for lab, c in zip(labels, cid2) if c==1]
    sub_b = [lab for lab, c in zip(labels, cid2) if c==2]
    print(f"\n  Data-driven 2-cluster partition (avg-linkage on 1-ρ):")
    print(f"    Cluster 1: {sub_a}")
    print(f"    Cluster 2: {sub_b}")
    if q_name == 'ERQ':
        # Compare to Reappraisal vs Suppression
        reapp = q_spec['theoretical_subscales']['Reappraisal']
        cluster_of = {n: cid2[i] for i, n in enumerate(item_nums)}
        in_reapp_cluster = [cluster_of[n] for n in reapp]
        majority = max(set(in_reapp_cluster), key=in_reapp_cluster.count)
        match = sum(1 for n in item_nums
                    if (n in reapp and cluster_of[n]==majority) or
                       (n not in reapp and cluster_of[n]!=majority))
        print(f"    Match with theoretical Reappraisal/Suppression split: "
              f"{match}/{len(item_nums)} items "
              f"({'PERFECT' if match==len(item_nums) else f'{match*100/len(item_nums):.0f}%'})")

    all_results[q_name] = dict(W=W, labels=labels, item_nums=item_nums,
                                full_labels=full_labels, corr_S=corr_S,
                                alpha_full=alpha_full, eigvals=eigvals,
                                pct_var=pct_var, loadings=loadings, Z_link=Z_link,
                                text_lookup=text_lookup, q_spec=q_spec)

# ──────── PLOT: one figure per questionnaire (4 panels each) ────────
LIKERT_COL_5 = ['#2c7fb8','#7fcdbb','#edf8b1','#fdae61','#d7191c']
LIKERT_COL_7 = ['#2166ac','#67a9cf','#d1e5f0','#f7f7f7','#fddbc7','#ef8a62','#b2182b']
for q_name, R in all_results.items():
    W = R['W']; labels = R['labels']; corr_S = R['corr_S']
    item_nums = R['item_nums']; text_lookup = R['text_lookup']
    pct_var = R['pct_var']; eigvals = R['eigvals']; Z_link = R['Z_link']
    scale_max = R['q_spec']['scale_max']
    likert_levels = list(range(1, scale_max+1))
    likert_col = LIKERT_COL_5 if scale_max == 5 else LIKERT_COL_7

    fig, axes = plt.subplots(2, 2, figsize=(15, 11), facecolor=BG)
    # (a) item distributions
    ax = axes[0,0]; ax.set_facecolor(BG)
    counts = pd.DataFrame({lab: W[lab].value_counts().reindex(likert_levels, fill_value=0)
                            for lab in labels}).T
    pct_resp = counts.div(counts.sum(axis=1), axis=0) * 100
    bottoms = np.zeros(len(labels))
    for i, lev in enumerate(likert_levels):
        vals = pct_resp[lev].values
        ax.barh(labels, vals, left=bottoms, color=likert_col[i],
                 label=f'{lev}', edgecolor='white', linewidth=0.4)
        for j, v in enumerate(vals):
            if v >= 6:
                ax.text(bottoms[j] + v/2, j, f"{v:.0f}%",
                         ha='center', va='center', fontsize=7,
                         color='white' if i in (0, len(likert_levels)-1) else 'black')
        bottoms += vals
    ax.set_xlim(0, 100); ax.set_xlabel(f"% (Likert {likert_levels[0]}–{likert_levels[-1]})")
    ax.set_title("Item response distributions", fontweight='bold', fontsize=11)
    ax.legend(title='Lik', fontsize=7, loc='center left', bbox_to_anchor=(1.02, 0.5))
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    # (b) correlation heatmap
    ax = axes[0,1]; ax.set_facecolor(BG)
    vmax_ = max(0.3, np.nanmax(np.abs(corr_S - np.eye(len(corr_S)))))
    im = ax.imshow(corr_S, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{corr_S[i,j]:.2f}",
                     ha='center', va='center', fontsize=8,
                     color='white' if abs(corr_S[i,j]) > 0.6 else 'black',
                     fontweight='bold')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(f"Spearman inter-item ρ  (Cronbach α={R['alpha_full']:.3f})",
                  fontweight='bold', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # (c) PCA scree
    ax = axes[1,0]; ax.set_facecolor(BG)
    xs = np.arange(1, len(eigvals)+1)
    ax.bar(xs, eigvals, color='#fdae6b', edgecolor='black', alpha=.85)
    ax.axhline(1, color='red', ls='--', alpha=.7, label='Kaiser (eig=1)')
    ax2 = ax.twinx()
    ax2.plot(xs, np.cumsum(pct_var)*100, 'o-', color='#08519c', lw=2,
              markersize=7, label='cum % var')
    ax.set_xlabel("Component"); ax.set_ylabel("Eigenvalue", color='#fdae6b')
    ax2.set_ylabel("Cum % variance", color='#08519c'); ax2.set_ylim(0, 105)
    ax.set_title(f"PCA scree  (PC1={pct_var[0]*100:.1f}% · PC2={pct_var[1]*100:.1f}%)",
                  fontweight='bold', fontsize=11)
    ax.legend(loc='upper right', fontsize=8)
    ax.spines["top"].set_visible(False)

    # (d) dendrogram
    ax = axes[1,1]; ax.set_facecolor(BG)
    dendrogram(Z_link, labels=labels, ax=ax, leaf_font_size=10,
                color_threshold=0.5*max(Z_link[:,2]))
    ax.set_xlabel("Item"); ax.set_ylabel("1 − Spearman ρ")
    ax.set_title("Hierarchical clustering (avg linkage)",
                  fontweight='bold', fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle(f"Section 0B — {q_name} structure  (n={len(W)})",
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    fname = f"fig_questionnaire_{q_name.replace('-','')}.png"
    plt.savefig(fname, dpi=180, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f"\nSaved {fname}")

# ──────── Subscales × downstream + each other ────────
print("\n" + "="*78); print("Subscale inter-correlations  (Spearman)"); print("="*78)
sub_cols = [c for c in rt_df.columns if c.startswith(('ERQ_','PSWQA_','MASQAA_'))
             and c not in ('ERQ_Total_Score',)]
if len(sub_cols) >= 2:
    print(rt_df[sub_cols].corr(method='spearman').round(3).to_string())

### 0C — Pooled factor analysis across all questionnaires

The Section 0B analyses ran factor structure **within** each
questionnaire. Here we instead **pool all 28 items** (10 ERQ + 8
PSWQ-A + 10 MASQ-AA) into a single PCA / hierarchical-clustering
analysis. The question is: ignoring the questionnaire labels, what
*empirical* clusters emerge? Do items cluster purely by
questionnaire (i.e., the three scales measure three distinct
constructs), or do items mix across questionnaires (e.g., reappraisal
items separating from worry+somatic items, suggesting a "negative
affect" pole and a "regulation" pole)?

Items have different response scales (ERQ = 1–7, PSWQ and MASQ-AA
= 1–5), so we **z-score each item before pooling** so the PCA isn't
biased by scale range.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 0C.  POOLED FACTOR ANALYSIS — all 28 questionnaire items together
# ══════════════════════════════════════════════════════════════════════

# Build long item table with questionnaire-prefixed labels
all_items = []
for q_name, R in all_results.items():
    for n, lab in zip(R['item_nums'], R['labels']):
        all_items.append({
            'q': q_name, 'item_num': n, 'col': f"{R['q_spec']['prefix']}{n}_Raw",
            'label': f"{q_name[:3]}_{lab}",
            'text': R['text_lookup'].get(n, '?'),
        })
all_items_df = pd.DataFrame(all_items)
print(f"Pooled item set: {len(all_items_df)} items "
      f"(ERQ={sum(all_items_df['q']=='ERQ')}, "
      f"PSWQ={sum(all_items_df['q']=='PSWQ-A')}, "
      f"MASQ={sum(all_items_df['q']=='MASQ-AA')})")

# Build subject × item matrix; standardise each item (handles 1-5 vs 1-7 scale)
M = rt_df[all_items_df['col'].tolist()].dropna().copy()
M.columns = all_items_df['label'].tolist()
n_subj = len(M)
print(f"N subjects with full data on all {len(all_items_df)} items: {n_subj}")
Mz = (M - M.mean()) / M.std()

# Joint Spearman correlation matrix
corr_J = M.corr(method='spearman').values
labels_J = list(M.columns)

# Cronbach alpha (treating all items as one scale — sanity check)
alpha_pool = cronbach_alpha(Mz.values)
print(f"\nCronbach alpha if treating all 28 items as one scale: {alpha_pool:.3f}")
print("(High α here would mean a single underlying dimension; lower α "
      "means multiple distinct constructs.)")

# PCA on standardised data
U_J, sv_J, Vt_J = np.linalg.svd(Mz.values, full_matrices=False)
ev_J = (sv_J ** 2) / (n_subj - 1)
pct_J = ev_J / ev_J.sum()
loadings_J = (Vt_J.T * sv_J) / np.sqrt(n_subj - 1)
print(f"\nPCA eigenvalues (top 8): {[f'{v:.2f}' for v in ev_J[:8]]}")
print(f"%% variance (top 8):     {[f'{p*100:.1f}%' for p in pct_J[:8]]}")
print(f"Cumulative %% (top 8):   {[f'{p*100:.1f}%' for p in np.cumsum(pct_J)[:8]]}")
n_kaiser = int((ev_J >= 1.0).sum())
print(f"Components with eigenvalue >= 1 (Kaiser): {n_kaiser}")

# Show loadings on top 4 components, sorted by abs loading per component
print("\n" + "-"*78)
print("Top loadings on PC1, PC2, PC3, PC4 (sorted by abs value):")
print("-"*78)
for k in range(min(4, loadings_J.shape[1])):
    print(f"\nPC{k+1}  (eigenvalue {ev_J[k]:.2f}, {pct_J[k]*100:.1f}% variance)")
    sort_idx = np.argsort(-np.abs(loadings_J[:, k]))
    for rank, i in enumerate(sort_idx[:12]):
        ld = loadings_J[i, k]
        item_row = all_items_df.iloc[i]
        print(f"  {labels_J[i]:>10s}  load={ld:+.3f}  "
              f"({item_row['q']:>7s})  '{item_row['text']}'")

# ─── Hierarchical clustering on joint correlation matrix ───
dist_J = 1 - corr_J
np.fill_diagonal(dist_J, 0); dist_J = (dist_J + dist_J.T) / 2
condensed_J = squareform(dist_J, checks=False)
Z_link_J = linkage(condensed_J, method='average')

# Dump a few cluster solutions
print("\n" + "-"*78)
print("Hierarchical clustering on pooled items (avg-linkage on 1-rho)")
print("-"*78)
for k in [2, 3, 4, 5]:
    cid = fcluster(Z_link_J, t=k, criterion='maxclust')
    cluster_summary = {}
    for lab, c, q in zip(labels_J, cid, all_items_df['q']):
        cluster_summary.setdefault(int(c), []).append((lab, q))
    print(f"\n{k}-cluster solution:")
    for cnum in sorted(cluster_summary):
        items = cluster_summary[cnum]
        # Tally questionnaire composition
        q_tally = {}
        for _, q in items:
            q_tally[q] = q_tally.get(q, 0) + 1
        compo = ", ".join([f"{q}={n}" for q,n in sorted(q_tally.items(), key=lambda x:-x[1])])
        item_list = ", ".join([lab for lab,_ in items])
        print(f"  Cluster {cnum} (n={len(items)}, {compo}): {item_list}")

# ─── Plot 1: pooled corr heatmap, ordered by hierarchical clustering ───
BG = "#FAFBFC"
from scipy.cluster.hierarchy import leaves_list
order = leaves_list(Z_link_J)
labels_ord = [labels_J[i] for i in order]
corr_ord = corr_J[np.ix_(order, order)]
# Color labels by questionnaire
Q_COLOR = {'ERQ':'#fdae6b','PSWQ-A':'#fb6a4a','MASQ-AA':'#08519c'}

fig, ax = plt.subplots(figsize=(13, 11), facecolor=BG)
ax.set_facecolor(BG)
im = ax.imshow(corr_ord, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(labels_ord)))
ax.set_xticklabels(labels_ord, rotation=90, fontsize=7)
ax.set_yticks(range(len(labels_ord)))
ax.set_yticklabels(labels_ord, fontsize=7)
# Color tick labels by questionnaire
q_lookup = dict(zip(labels_J, all_items_df['q']))
for tick, lab in zip(ax.get_xticklabels(), labels_ord):
    tick.set_color(Q_COLOR[q_lookup[lab]])
for tick, lab in zip(ax.get_yticklabels(), labels_ord):
    tick.set_color(Q_COLOR[q_lookup[lab]])
plt.colorbar(im, ax=ax, label='Spearman ρ', shrink=0.7)
ax.set_title("Pooled inter-item correlation matrix  (ordered by hierarchical clustering)\n"
              "Tick colors: orange=ERQ, red=PSWQ-A, blue=MASQ-AA",
              fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig("fig_pooled_questionnaire_corr.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_pooled_questionnaire_corr.png")

# ─── Plot 2: PC1×PC2 biplot of item loadings, colored by questionnaire ───
fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor=BG)
# Panel A: scree
ax = axes[0]; ax.set_facecolor(BG)
xs = np.arange(1, len(ev_J)+1)
ax.bar(xs, ev_J, color='#fdae6b', edgecolor='black', alpha=.85)
ax.axhline(1, color='red', ls='--', alpha=.7, label='Kaiser (eig=1)')
ax2 = ax.twinx()
ax2.plot(xs, np.cumsum(pct_J)*100, 'o-', color='#08519c', lw=2,
          markersize=6, label='cum % var')
ax.set_xlabel("Component"); ax.set_ylabel("Eigenvalue", color='#fdae6b')
ax2.set_ylabel("Cumulative % variance", color='#08519c'); ax2.set_ylim(0, 105)
ax.set_title(f"Scree, pooled 28 items  ({n_kaiser} components ≥ Kaiser cutoff)",
              fontweight='bold', fontsize=11)
ax.legend(loc='upper right', fontsize=8)
ax.spines["top"].set_visible(False)

# Panel B: PC1 vs PC2 loadings biplot
ax = axes[1]; ax.set_facecolor(BG)
for q_name, color in Q_COLOR.items():
    mask = all_items_df['q'] == q_name
    pts = loadings_J[mask.values][:, :2]
    labs = all_items_df.loc[mask, 'label'].tolist()
    ax.scatter(pts[:,0], pts[:,1], s=80, color=color, alpha=.85,
                edgecolors='white', linewidths=0.6, label=q_name)
    for (x,y), lab in zip(pts, labs):
        # Strip questionnaire prefix to keep label short
        short = lab.split('_', 1)[1] if '_' in lab else lab
        ax.annotate(short, (x,y), fontsize=7,
                     xytext=(3, 3), textcoords='offset points')
ax.axhline(0, color='black', lw=0.4); ax.axvline(0, color='black', lw=0.4)
ax.set_xlabel(f"PC1  ({pct_J[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2  ({pct_J[1]*100:.1f}% var)")
ax.set_title("PC1 × PC2 loadings biplot  (items by questionnaire)",
              fontweight='bold', fontsize=11)
ax.legend(fontsize=10, loc='best')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle(f"Section 0C — Pooled factor analysis of all 28 items  (n={n_subj}, α={alpha_pool:.3f})",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_pooled_questionnaire_pca.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_pooled_questionnaire_pca.png")

# ─── Plot 3: dendrogram with items color-coded by questionnaire ───
fig, ax = plt.subplots(figsize=(16, 6), facecolor=BG)
ax.set_facecolor(BG)
dendrogram(Z_link_J, labels=labels_J, ax=ax, leaf_font_size=8,
            color_threshold=0.5*max(Z_link_J[:,2]))
ax.set_xlabel("Item"); ax.set_ylabel("1 − Spearman ρ  (avg-linkage)")
# Color the x-tick labels by questionnaire
for tick in ax.get_xticklabels():
    lab = tick.get_text()
    if lab in q_lookup:
        tick.set_color(Q_COLOR[q_lookup[lab]])
        tick.set_fontweight('bold')
ax.set_title("Section 0C — Pooled hierarchical clustering of all 28 items\n"
             "Item color: orange=ERQ, red=PSWQ-A, blue=MASQ-AA",
              fontweight='bold', fontsize=11)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("fig_pooled_questionnaire_dendrogram.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_pooled_questionnaire_dendrogram.png")

# ─── Save pooled PC scores per subject ───
for k in range(min(4, loadings_J.shape[1])):
    rt_df[f'Pooled_PC{k+1}'] = np.nan
rt_df.loc[M.index, [f'Pooled_PC{k+1}' for k in range(min(4, loadings_J.shape[1]))]] = \
    Mz.values @ Vt_J[:min(4, loadings_J.shape[1])].T
print(f"\nSaved per-subject Pooled_PC1..PC4 scores to rt_df.")

### 1A-anti — RT for anti-MF vs non-anti-MF queries

Anti-MF queries are the diagnostic trials where MB and MF predict
*different* choices. Non-anti-MF queries are the agreement trials
(the easy ones — both strategies vote the same way). If subjects
deliberate more on the diagnostic trials than the agreement trials,
we'd expect:

* `median_rt_antimf > median_rt_notantimf` within subject.
* The gap may be larger in MB subjects (who actually have an MB
  representation to consult on these trials).
* The gap may correlate with anxiety in MB subjects (anxious MB ⇒
  more deliberation on diagnostic trials).

Per subject we compute median RT (and delib) on each subset, the
within-subject difference (anti-MF − not), then test:

1. Wilcoxon signed-rank: anti-MF > non-anti-MF, paired across subjects,
   in MB and non-MB separately.
2. Spearman ρ between the **gap** (anti − not) and anxiety, by group.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A-anti.  RT FOR ANTI-MF vs NON-ANTI-MF QUERIES (per subject)
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import wilcoxon, spearmanr, mannwhitneyu

# Per-subject scalars on each subset.
# Per user (2026-04-28) all delib measures are on the log scale:
#   delib_log = mean over trials of log( max(rt_t - motor_baseline, 0.01) )
def _agg_subset(d):
    if len(d) == 0:
        return (np.nan, np.nan)
    return (float(d['rt_s'].median()), float(d['log_delib'].mean()))

anti_rows = []
for sid, g in rt_trial.groupby('subject_id'):
    mb = rt_motor_map.get(sid, np.nan)
    if not np.isfinite(mb): continue
    g_anti = g[g['is_anti_mf'] == 1]
    g_not  = g[g['is_anti_mf'] == 0]
    rt_anti, delib_log_anti = _agg_subset(g_anti)
    rt_not,  delib_log_not  = _agg_subset(g_not)
    anti_rows.append({
        'subject_id'                     : sid,
        'n_anti_mf'                      : len(g_anti),
        'n_not_anti_mf'                  : len(g_not),
        'median_rt_s_antimf'             : rt_anti,
        'median_rt_s_notantimf'          : rt_not,
        'delib_log_antimf'               : delib_log_anti,
        'delib_log_notantimf'            : delib_log_not,
        'rt_diff_antimf_minus_not'       : (rt_anti        - rt_not)        if pd.notna(rt_anti)        else np.nan,
        'delib_log_diff_antimf_minus_not': (delib_log_anti - delib_log_not) if pd.notna(delib_log_anti) else np.nan,
    })
anti_df = pd.DataFrame(anti_rows)
print(f"N subjects with anti-MF/non-anti-MF data: {len(anti_df)}")
print(f"  Anti-MF trials per subject: median={anti_df['n_anti_mf'].median():.0f}, "
      f"range [{anti_df['n_anti_mf'].min()}, {anti_df['n_anti_mf'].max()}]")
print(f"  Non-anti-MF trials per subject: median={anti_df['n_not_anti_mf'].median():.0f}")

# Merge into rt_per_subj for downstream use
rt_per_subj = rt_per_subj.merge(anti_df, on='subject_id', how='left')

# ── Wilcoxon paired test: anti-MF RT > not-anti-MF RT, by group ──
print("\nWilcoxon signed-rank (paired): anti-MF RT > non-anti-MF RT")
for grp_label, gdf in [('All', rt_per_subj),
                        ('non-MB', rt_per_subj[rt_per_subj['mb_group']=='non-MB']),
                        ('MB',     rt_per_subj[rt_per_subj['mb_group']=='MB'])]:
    g = gdf.dropna(subset=['median_rt_s_antimf','median_rt_s_notantimf'])
    g = g[g['median_rt_s_antimf'] != g['median_rt_s_notantimf']]   # drop ties
    if len(g) < 5: continue
    w, p = wilcoxon(g['median_rt_s_antimf'], g['median_rt_s_notantimf'],
                     alternative='greater')
    med_diff = float(g['rt_diff_antimf_minus_not'].median())
    print(f"  {grp_label:>7s} (n={len(g):3d}): "
          f"median diff = {med_diff:+.3f}s,  Wilcoxon W={w:.0f},  one-sided p={p:.4g}")

# ── Spearman: anti-MF gap (RT, delib_log) ↔ {somatic_anxiety, worry}, by group ──
for anx_col in ['somatic_anxiety','worry']:
    print(f"\n=== Spearman: gap(anti-MF − non-anti-MF) ↔ {anx_col}, by MB group ===")
    for diff_col in ['rt_diff_antimf_minus_not','delib_log_diff_antimf_minus_not']:
        print(f"\n  {diff_col}:")
        for grp_label, gdf in [('All', rt_per_subj),
                                ('non-MB', rt_per_subj[rt_per_subj['mb_group']=='non-MB']),
                                ('MB',     rt_per_subj[rt_per_subj['mb_group']=='MB'])]:
            g = gdf.dropna(subset=[anx_col, diff_col])
            if len(g) < 5: continue
            rs, ps = spearmanr(g[anx_col], g[diff_col])
            ps_1 = ps/2 if rs > 0 else 1 - ps/2
            print(f"    {grp_label:>7s} (n={len(g):3d}): "
                  f"rho={rs:+.3f}  one-sided p={ps_1:.3g}  (two-sided {ps:.3g})")

rt_per_subj.to_csv("rt_analyses_per_subject.csv", index=False)
print("\nSaved rt_analyses_per_subject.csv (now includes anti-MF columns)")

# ── Plot: paired RT (anti vs not) by subject, per MB group ──
BG = "#FAFBFC"
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=BG, sharey=True)
for ax, (grp_label, gdf) in zip(axes,
        [('All', rt_per_subj),
         ('non-MB (MF)', rt_per_subj[rt_per_subj['mb_group']=='non-MB']),
         ('MB',     rt_per_subj[rt_per_subj['mb_group']=='MB'])]):
    ax.set_facecolor(BG)
    g = gdf.dropna(subset=['median_rt_s_antimf','median_rt_s_notantimf'])
    n = len(g)
    if n < 3:
        ax.text(0.5, 0.5, 'too few', transform=ax.transAxes); continue
    # Paired-points plot
    for _, row in g.iterrows():
        c = '#e6550d' if row['median_rt_s_antimf'] > row['median_rt_s_notantimf'] else '#3182bd'
        ax.plot([0, 1], [row['median_rt_s_notantimf'], row['median_rt_s_antimf']],
                 color=c, lw=0.6, alpha=.5, marker='o', markersize=3)
    med_not  = g['median_rt_s_notantimf'].median()
    med_anti = g['median_rt_s_antimf'].median()
    ax.plot([0, 1], [med_not, med_anti], 'k-', lw=2.5,
             label=f"medians: {med_not:.2f} → {med_anti:.2f}")
    w, p = wilcoxon(g['median_rt_s_antimf'], g['median_rt_s_notantimf'],
                     alternative='greater') if n >= 5 else (np.nan, np.nan)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anti-MF','anti-MF'])
    ax.set_ylabel("Per-subject median RT (s)")
    ax.set_title(f"{grp_label}  (n={n})\n"
                 f"Wilcoxon one-sided p = {p:.3g}",
                  fontweight='bold', fontsize=10)
    ax.legend(fontsize=8, loc='upper left')
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Section 1A-anti — per-subject median RT on anti-MF vs non-anti-MF queries",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_section1A_antimf_vs_notantimf_rt.png", dpi=180,
              bbox_inches='tight', facecolor=BG)
plt.show()
print("\nSaved fig_section1A_antimf_vs_notantimf_rt.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A-bis.  ASSUMPTION CHECK: Pearson vs Spearman, per (metric × subgroup)
# ══════════════════════════════════════════════════════════════════════
# For every (anxiety vs metric) pair we'll later correlate, run:
#   - Shapiro-Wilk normality test on each variable
#   - Outlier count via |z-score| > 3
#   - Linearity check: |Pearson r| vs |Spearman rho|
#       Big gap (>0.05) suggests monotonic-but-non-linear relationship
# Recommend Spearman if any check fails; Pearson otherwise.
# Output is one terse line per failing test only.
from scipy.stats import shapiro

def assumption_check(x, y):
    g = pd.DataFrame({'x': x, 'y': y}).dropna()
    n = len(g)
    if n < 10:
        return ('Spearman', f'n={n} too small')
    issues = []
    try:
        _, p_x = shapiro(g['x'])
    except Exception: p_x = np.nan
    try:
        _, p_y = shapiro(g['y'])
    except Exception: p_y = np.nan
    if not np.isnan(p_x) and p_x < 0.05:
        issues.append(f"x non-normal Shapiro p={p_x:.3f}")
    if not np.isnan(p_y) and p_y < 0.05:
        issues.append(f"y non-normal Shapiro p={p_y:.3f}")
    if g['x'].std() > 0:
        zx = np.abs((g['x'] - g['x'].mean()) / g['x'].std())
    else: zx = pd.Series(0, index=g.index)
    if g['y'].std() > 0:
        zy = np.abs((g['y'] - g['y'].mean()) / g['y'].std())
    else: zy = pd.Series(0, index=g.index)
    n_out = int(((zx > 3) | (zy > 3)).sum())
    if n_out > 0:
        issues.append(f"{n_out} outlier(s) |z|>3")
    rp, _ = pearsonr (g['x'], g['y'])
    rs, _ = spearmanr(g['x'], g['y'])
    nl_gap = abs(abs(rs) - abs(rp))
    if nl_gap > 0.05:
        issues.append(f"|rho|={abs(rs):.3f} >> |r|={abs(rp):.3f} (non-linear monotonic)")
    if not issues:
        return ('Pearson', 'PASS')
    return ('Spearman', '; '.join(issues))

# Column lists — single OT definition; delib_s and acc_canon only.
RT_COLS  = [f'delib_log_{c}' for c in ['all','feather','triangle']]
ACC_COLS = [f'acc_canon_{c}' for c in ['all','feather','triangle']]
OT_COLS  = [f'OT_{c}'        for c in ['all','feather','triangle']]
ALL_METRICS = RT_COLS + ACC_COLS + OT_COLS

rec_rows = []
for anx_col in ['somatic_anxiety','worry']:
    print(f"\n=== Assumption checks: {anx_col} × metric, per subgroup ===\n")
    n_pearson = n_spearman = 0
    fail_lines = []
    for metric in ALL_METRICS:
        for grp in [None, 'non-MB', 'MB']:
            g = rt_per_subj if grp is None else rt_per_subj[rt_per_subj['mb_group'] == grp]
            rec, why = assumption_check(g[anx_col], g[metric])
            grp_label = 'All' if grp is None else grp
            if rec == 'Pearson':
                n_pearson += 1
            else:
                n_spearman += 1
                fail_lines.append(f"  {metric:32s}  ({grp_label:7s})  → Spearman:  {why}")
            rec_rows.append({'anx': anx_col, 'metric': metric,
                             'group':  grp_label,
                             'recommended': rec, 'reason': why})
    print(f"  Pearson PASS:        {n_pearson:3d}/{n_pearson + n_spearman} tests")
    print(f"  Spearman recommended:{n_spearman:3d}/{n_pearson + n_spearman} tests")
    print(f"  Failures (Spearman recommended) — first 20:")
    for line in fail_lines[:20]:
        print(line)
    if len(fail_lines) > 20:
        print(f"    ... ({len(fail_lines) - 20} more, see saved CSV)")

pd.DataFrame(rec_rows).to_csv("rt_analyses_assumption_checks.csv", index=False)
print(f"\nSaved rt_analyses_assumption_checks.csv (full table, both anxieties)")

---
## Section 2.5 — Power analysis (one-sided, 95 % power)

All hypotheses below are **directional** (preregistration §3 / §6).
Power is therefore computed using a **one-sided** test at $\alpha=.05$.
The headline target is **95 % power**.

H1 is a *moderation* claim: anxiety → overthinking manifests in MB
subjects but not in non-MB subjects. The confirmatory test is
therefore the **interaction** in the rank-based moderation regression
(Section 2.7), not just a simple correlation in MB. Statistical power
for the interaction is approximately the power to detect a
**difference between two independent correlations** ($\rho_{MB}$ vs
$\rho_{\text{non-MB}}$).

### Power formula (primary — moderation interaction)

For two independent groups with sample correlations $\rho_{MB}$ and
$\rho_{\text{non-MB}}$, Fisher-$z$ transform each:
$z_g = \mathrm{atanh}(\rho_g)$, with $\mathrm{SE}(z_g) = 1/\sqrt{n_g-3}$.
The difference $\Delta z = z_{MB} - z_{\text{non-MB}}$ has
$\mathrm{SE} = \sqrt{1/(n_{MB}-3) + 1/(n_{\text{non-MB}}-3)}$, so
the one-sided power to detect $\Delta z > 0$ is
$$\text{power} = \Phi\!\left(\Delta z / \mathrm{SE} - z_{1-\alpha}\right).$$

### Power formula (reference — MB-only simple correlation)

Reported alongside for context: power to detect Pearson/Spearman
$\rho_{MB}$ within the MB subgroup using the standard Fisher-$z$
single-correlation formula
$\Phi(z_\rho\sqrt{n-3} - z_{1-\alpha})$.

### What we compute

Given the observed proportion of MB subjects $p_{MB}$ (above-chance
criterion in cell 1):

* Power achieved at the current sample for the **interaction**
  (primary) and the MB-only correlation (reference).
* Required total $N$ for 95 % power on the **interaction**, sweeping
  $\rho_{MB}$ and assuming $\rho_{\text{non-MB}} = 0$ (the H1 null
  prediction in the non-MB subgroup).
* Required total $N$ at the **empirically-observed** $\rho_{MB}$
  and $\rho_{\text{non-MB}}$.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2.5  POWER ANALYSIS — MODERATION INTERACTION (primary)
#       + MB-only simple correlation (reference)
#       ONE-SIDED tests, primary target = 95 % power
# ══════════════════════════════════════════════════════════════════════
from scipy.stats import norm

ALPHA      = 0.05
TARGET_PWR = 0.95     # primary target — directional hypotheses
Z_CRIT     = norm.ppf(1 - ALPHA)

def power_simple(rho, n, alpha=ALPHA):
    '''One-sided power to detect a single correlation rho.'''
    if n <= 3 or rho == 0: return alpha
    z_rho = np.arctanh(abs(rho))
    se    = 1.0 / np.sqrt(n - 3)
    return float(norm.cdf(z_rho/se - norm.ppf(1 - alpha)))

def n_for_power_simple(rho, target=TARGET_PWR, alpha=ALPHA):
    if rho == 0: return np.inf
    z_rho  = np.arctanh(abs(rho))
    return int(np.ceil(((norm.ppf(1-alpha) + norm.ppf(target)) / z_rho)**2 + 3))

def power_interaction(rho_mb, rho_non, n_mb, n_non, alpha=ALPHA):
    '''One-sided power to detect difference between two independent
       Fisher-z transformed correlations (rho_MB - rho_nonMB > 0).'''
    if n_mb <= 3 or n_non <= 3: return alpha
    z_diff = np.arctanh(rho_mb) - np.arctanh(rho_non)
    se     = np.sqrt(1.0/(n_mb-3) + 1.0/(n_non-3))
    if se == 0 or z_diff == 0: return alpha
    return float(norm.cdf(z_diff/se - norm.ppf(1 - alpha)))

def n_total_for_interaction(rho_mb, rho_non, p_mb,
                             target=TARGET_PWR, alpha=ALPHA, max_n=20000):
    '''Smallest TOTAL N giving one-sided power >= target for the
       interaction, given p_MB.'''
    z_diff = np.arctanh(rho_mb) - np.arctanh(rho_non)
    if z_diff <= 0: return np.inf
    for N in range(20, max_n+1):
        n_mb  = int(round(p_mb * N))
        n_non = N - n_mb
        if n_mb < 4 or n_non < 4: continue
        if power_interaction(rho_mb, rho_non, n_mb, n_non, alpha) >= target:
            return N
    return np.inf

# ── Observed sample composition ──
n_total = len(subj_meta)
n_mb    = int((subj_meta['mb_group'] == 'MB').sum())
n_nonmb = int((subj_meta['mb_group'] == 'non-MB').sum())
p_mb    = n_mb / n_total
print(f"Observed sample: N={n_total}  ({n_mb} MB, {n_nonmb} non-MB; p_MB={p_mb:.3f})")

# ── Observed effect: Spearman ρ in each subgroup (PRIMARY), headline OT ──
# Single OT definition: OT_all = z(delib_all) − z(logit acc_canon_all).
# Power computed for BOTH somatic_anxiety and worry.
HEADLINE_COL = 'OT_all'
observed_rhos = {}
for anx_col in ['somatic_anxiety','worry']:
    g_mb  = rt_per_subj[rt_per_subj['mb_group']=='MB'   ].dropna(subset=[anx_col, HEADLINE_COL])
    g_non = rt_per_subj[rt_per_subj['mb_group']=='non-MB'].dropna(subset=[anx_col, HEADLINE_COL])
    g_all = rt_per_subj.dropna(subset=[anx_col, HEADLINE_COL])
    rho_mb,  _ = spearmanr(g_mb[anx_col],  g_mb[HEADLINE_COL])
    rho_non, _ = spearmanr(g_non[anx_col], g_non[HEADLINE_COL])
    rho_all, _ = spearmanr(g_all[anx_col], g_all[HEADLINE_COL])
    observed_rhos[anx_col] = dict(rho_mb=rho_mb, rho_non=rho_non, rho_all=rho_all,
                                  n_mb=len(g_mb), n_non=len(g_non), n_all=len(g_all))
    print(f"\nObserved Spearman ρ for {HEADLINE_COL} × {anx_col}:")
    print(f"  All subjects:     ρ={rho_all:+.3f}  (n={len(g_all)})")
    print(f"  MB subgroup:      ρ={rho_mb:+.3f}   (n={len(g_mb)})")
    print(f"  non-MB subgroup:  ρ={rho_non:+.3f}  (n={len(g_non)})")

# Use somatic_anxiety as the headline for the rest of power analysis
rho_mb  = observed_rhos['somatic_anxiety']['rho_mb']
rho_non = observed_rhos['somatic_anxiety']['rho_non']
rho_all = observed_rhos['somatic_anxiety']['rho_all']
print("\n(Power curves below use SOMATIC_ANXIETY × OT_all as the headline effect; "
      "worry-based power is shown alongside in the figure.)")

# ───────── PRIMARY: power for the moderation INTERACTION ─────────
print("\n" + "="*70)
print("PRIMARY — moderation interaction (rho_MB vs rho_nonMB, Fisher-z diff)")
print("="*70)
pwr_now = power_interaction(rho_mb, rho_non, n_mb, n_nonmb)
print(f"\nObserved interaction effect:")
print(f"  Delta z = atanh({rho_mb:+.3f}) - atanh({rho_non:+.3f}) "
      f"= {np.arctanh(rho_mb)-np.arctanh(rho_non):+.4f}")
print(f"  At current sample (n_MB={n_mb}, n_non={n_nonmb}): "
      f"power = {pwr_now:.3f}")

# Required N at observed effect, with rho_non taken as observed AND as 0
print(f"\nRequired TOTAL N at one-sided alpha={ALPHA}, target={int(TARGET_PWR*100)}% "
      f"(p_MB={p_mb:.2f}):")
print(f"{'rho_MB':>8s}  {'rho_non':>8s}  {'N_total':>8s}  {'n_MB':>6s}  {'n_non':>6s}  "
      f"{'(N - 139)':>10s}")
scenarios = [
    (rho_mb, rho_non, "OBSERVED: rho_MB & rho_non as measured"),
    (rho_mb, 0.0,     "rho_non = 0 (H1 null in non-MB)"),
    (0.30,   0.0,     "benchmark: rho_MB=.30, rho_non=0"),
    (0.35,   0.0,     "benchmark: rho_MB=.35, rho_non=0"),
    (0.40,   0.0,     "benchmark: rho_MB=.40, rho_non=0"),
    (0.30,   -0.05,   "robust:    rho_MB=.30, rho_non=-.05"),
]
target_Ns = []
for rmb, rnon, lab in scenarios:
    N = n_total_for_interaction(rmb, rnon, p_mb, TARGET_PWR)
    n_mb_proj = int(round(p_mb * N)) if np.isfinite(N) else np.nan
    n_non_proj = int(N - n_mb_proj) if np.isfinite(N) else np.nan
    delta = (N - n_total) if np.isfinite(N) else np.nan
    target_Ns.append((rmb, rnon, N, n_mb_proj, n_non_proj, delta, lab))
    print(f"{rmb:>+8.3f}  {rnon:>+8.3f}  {N if np.isfinite(N) else 'inf':>8}  "
          f"{n_mb_proj if np.isfinite(N) else '-':>6}  "
          f"{n_non_proj if np.isfinite(N) else '-':>6}  "
          f"{('+'+str(delta)) if np.isfinite(N) else '-':>10}  ({lab})")

# Floor: max of observed-effect N and benchmark rho_MB=.25 against rho_non=0
N_floor = n_total_for_interaction(0.25, 0.0, p_mb, TARGET_PWR)
N_obs   = n_total_for_interaction(rho_mb, 0.0, p_mb, TARGET_PWR)
N_target = int(max(N_floor, N_obs))
print(f"\n>>> RECOMMENDED TARGET N_total = max(obs, |rho|=.25 floor) = {N_target} <<<")
print(f"    (current N = {n_total};  need to add ~{N_target - n_total} more subjects)")

# ───────── REFERENCE: power for MB-only simple correlation ─────────
print("\n" + "="*70)
print("REFERENCE — MB-only simple Spearman correlation (single-rho power)")
print("="*70)
print(f"\nPower at current n_MB={n_mb} (one-sided, alpha={ALPHA}):")
for rho in [0.20, 0.25, 0.30, 0.35, 0.40]:
    print(f"  |rho|={rho:.2f}: power = {power_simple(rho, n_mb):.3f}")
abs_r_mb = abs(rho_mb)
print(f"\nObserved |rho_MB| = {abs_r_mb:.3f}; "
      f"required n_MB for {int(TARGET_PWR*100)}% power: "
      f"{n_for_power_simple(abs_r_mb)}")

# ── Plot: power curves ──
BG_COLOR = "#FAFBFC"
fig, axes = plt.subplots(1, 2, figsize=(13, 4.7), facecolor=BG_COLOR)

# Panel 1: power for INTERACTION vs total N, for several rho_MB (rho_non=0)
ax = axes[0]; ax.set_facecolor(BG_COLOR)
Ns = np.arange(40, 801, 10)
for rmb, color in [(0.20,'#c6dbef'),(0.25,'#9ecae1'),(0.30,'#6baed6'),
                   (0.35,'#4292c6'),(0.40,'#2171b5'),(0.50,'#08519c')]:
    pw = []
    for N in Ns:
        nmb_ = int(round(p_mb * N)); nnon_ = N - nmb_
        pw.append(power_interaction(rmb, 0.0, nmb_, nnon_) if min(nmb_,nnon_)>3 else np.nan)
    ax.plot(Ns, pw, color=color, lw=2, label=f'rho_MB={rmb:.2f}, rho_non=0')
ax.axhline(0.80, color='gray', ls=':', alpha=.5)
ax.axhline(0.90, color='orange', ls=':', alpha=.6)
ax.axhline(0.95, color='red', ls='--', alpha=.7, label='95 % (PRIMARY)')
ax.axvline(n_total, color='black', ls=':', alpha=.6,
            label=f'current N={n_total}')
ax.set_xlabel("Total sample size N")
ax.set_ylabel("Power (one-sided) for INTERACTION")
ax.set_title(f"Power for moderation interaction\n(p_MB={p_mb:.2f}, one-sided alpha=0.05)",
              fontweight='bold', fontsize=10)
ax.legend(fontsize=7, loc='lower right')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Panel 2: required total N for 95% power across rho_MB
ax = axes[1]; ax.set_facecolor(BG_COLOR)
rhos = np.linspace(0.10, 0.60, 40)
N_for_int_rnon0 = [n_total_for_interaction(r, 0.0, p_mb, TARGET_PWR) for r in rhos]
N_for_int_rnonObs = [n_total_for_interaction(r, rho_non, p_mb, TARGET_PWR) for r in rhos]
N_for_simple_mb = [int(np.ceil(n_for_power_simple(r) / p_mb)) for r in rhos]
ax.plot(rhos, N_for_int_rnon0,  color='#a50f15', lw=2,
         label='INTERACTION (rho_non=0)  ← PRIMARY')
ax.plot(rhos, N_for_int_rnonObs, color='#fb6a4a', lw=2, ls='--',
         label=f'INTERACTION (rho_non={rho_non:+.2f}, observed)')
ax.plot(rhos, N_for_simple_mb,   color='#08519c', lw=2,
         label='MB-only simple rho (reference)')
ax.axhline(n_total, color='black', ls=':', alpha=.6,
            label=f'current N={n_total}')
ax.axvline(rho_mb, color='gray', ls='--', alpha=.6,
            label=f'observed rho_MB={rho_mb:+.2f}')
ax.set_xlabel("True rho_MB")
ax.set_ylabel("Required TOTAL N")
ax.set_title(f"Required N for {int(TARGET_PWR*100)} % power\n(one-sided alpha=0.05)",
              fontweight='bold', fontsize=10)
ax.set_ylim(0, min(2000, max([n for n in N_for_int_rnon0+N_for_int_rnonObs if np.isfinite(n)] + [800])*1.1))
ax.legend(fontsize=7, loc='upper right')
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Power analysis — moderation interaction primary, MB-only correlation reference  "
              "(one-sided, target 95 %)",
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig("fig_power_analysis_mb_subgroup.png", dpi=180,
              bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("Saved fig_power_analysis_mb_subgroup.png")

---
## Section 2.6 — Linear-OLS moderation (secondary / comparison)

> **Note.** The **primary** specification is now the rank-based
> regression in Section 2.7 (preregistration §6.1, v2.1). This linear
> OLS is reported as a comparison/secondary analysis, for
> transparency and continuity with the prior exploratory work. Per
> cell 1A-bis, Pearson assumptions fail in 72/72 anxiety × outcome
> tests — anxiety is non-normal in every subgroup — so the linear
> form here is interpretive only.

Linear OLS on the raw (unranked) variables:

```
OT  ~  worry + mb_group + worry:mb_group
```

with one-sided p-values for the directional H1 prediction
(interaction *positive*: slope of OT on worry **more positive in MB
than non-MB**, given the new sign convention OT = z(log RT) − z(acc),
so higher OT = more overthinking). Also runs the preregistered
robustness analyses (subjective accuracy / deliberation RT versions).

This is post-hoc on the current data — the same code is what would run
on the new replication sample.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2.6  LINEAR-OLS MODERATION (secondary / comparison)
#       Primary specification is rank-based — see Section 2.7
# ══════════════════════════════════════════════════════════════════════
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist

# Per user (2026-04-28): consolidated to a single primary OT measure
# (deliberation RT × canonical accuracy). All-trials, Feather-only,
# Triangle-only versions retained for the per-stim sensitivity checks.
OT_FLAVOURS = [
    ('OT_all',       'PRIMARY: canonical OT, delib RT, all trials'),
    ('OT_feather',   'per-stim:  Feather only'),
    ('OT_triangle',  'per-stim:  Triangle only'),
]

def _onesided_t(t_stat, rt_df, predicted_pos=True):
    '''One-sided p-value for the predicted direction.
       OT is signed so that positive slope = more overthinking.'''
    if predicted_pos:
        return float(t_dist.sf(t_stat, rt_df))    # P(T > t)
    return float(t_dist.cdf(t_stat, rt_df))       # P(T < t)

def _star(p):
    return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''

def fit_moderation(df_in, ot_col, label, anx_col):
    g = df_in.dropna(subset=[anx_col, ot_col, 'mb_group']).copy()
    g['mb_dummy'] = (g['mb_group'] == 'MB').astype(int)
    g['worry_c'] = g[anx_col] - g[anx_col].mean()
    m = smf.ols(f'{ot_col} ~ worry_c * mb_dummy', data=g).fit()
    df_resid = int(m.df_resid)

    # Coefficients
    b_int = m.params['worry_c:mb_dummy']
    t_int = m.tvalues['worry_c:mb_dummy']
    se_int = m.bse['worry_c:mb_dummy']
    p_int_2 = m.pvalues['worry_c:mb_dummy']
    p_int_1 = _onesided_t(t_int, df_resid, predicted_pos=True)

    # Simple slope: non-MB (mb_dummy = 0)  →  beta_worry
    b_non = m.params['worry_c']; se_non = m.bse['worry_c']; t_non = m.tvalues['worry_c']
    p_non_2 = m.pvalues['worry_c']
    p_non_1 = _onesided_t(t_non, df_resid, predicted_pos=True)

    # Simple slope: MB (mb_dummy = 1)  →  beta_worry + beta_interaction
    b_mb = b_non + b_int
    cov  = m.cov_params()
    se_mb = float(np.sqrt(cov.loc['worry_c','worry_c']
                           + cov.loc['worry_c:mb_dummy','worry_c:mb_dummy']
                           + 2*cov.loc['worry_c','worry_c:mb_dummy']))
    t_mb  = b_mb / se_mb
    p_mb_2 = 2 * (1 - t_dist.cdf(abs(t_mb), df_resid))
    p_mb_1 = _onesided_t(t_mb, df_resid, predicted_pos=True)

    print(f"\n--- {label}  ({ot_col}) ---")
    print(f"  N = {len(g)}  (MB={int((g['mb_dummy']==1).sum())}, non-MB={int((g['mb_dummy']==0).sum())})")
    print(f"  Interaction (worry × MB):  beta={b_int:+.4f} SE={se_int:.4f}  "
          f"t({df_resid})={t_int:+.2f}  one-sided p={p_int_1:.4g} {_star(p_int_1)}  "
          f"(two-sided p={p_int_2:.4g})")
    print(f"  Simple slope, non-MB:      beta={b_non:+.4f} SE={se_non:.4f}  "
          f"one-sided p={p_non_1:.4g} {_star(p_non_1)}  (two-sided p={p_non_2:.4g})")
    print(f"  Simple slope, MB:          beta={b_mb:+.4f} SE={se_mb:.4f}  "
          f"one-sided p={p_mb_1:.4g} {_star(p_mb_1)}  (two-sided p={p_mb_2:.4g})")

    # Apply decision rule from PREREGISTRATION §6.5
    # OT signed so that POSITIVE slope = more overthinking.
    int_ok = (p_int_1 < 0.05) and (b_int > 0)
    mb_ok  = (p_mb_1  < 0.05) and (b_mb  > 0)
    wrong_dir = (b_int < 0) and (p_int_2 < 0.05)
    if int_ok and mb_ok:
        verdict = "H1 SUPPORTED"
    elif int_ok and not mb_ok:
        verdict = "MARGINAL (interaction OK, MB slope p>.05)"
    elif wrong_dir:
        verdict = "H1 DISCONFIRMED (significant effect in WRONG direction)"
    else:
        verdict = "H1 NOT SUPPORTED (interaction p>.05)"
    print(f"  → DECISION: {verdict}")

    return dict(label=label, ot=ot_col, n=len(g),
                b_int=b_int, se_int=se_int, p_int_1=p_int_1, p_int_2=p_int_2,
                b_non=b_non, p_non_1=p_non_1,
                b_mb=b_mb,  se_mb=se_mb, p_mb_1=p_mb_1, p_mb_2=p_mb_2,
                verdict=verdict, model=m)

print("=" * 78)
print("LINEAR-OLS MODERATION (secondary / comparison) — primary is rank in §2.7")
print("=" * 78)

# Single OT, but run separately for somatic_anxiety AND worry
all_mod_results = {}
for anx_col in ['somatic_anxiety','worry']:
    print(f"\n\n========================  {anx_col.upper()}  ========================")
    res_list = [fit_moderation(rt_per_subj, ot, lab, anx_col) for ot, lab in OT_FLAVOURS]
    all_mod_results[anx_col] = res_list

mod_rows = []
for anx_col, res_list in all_mod_results.items():
    for r in res_list:
        d = {k: v for k, v in r.items() if k != 'model'}
        d['anx'] = anx_col
        mod_rows.append(d)
pd.DataFrame(mod_rows).to_csv("rt_analyses_moderation_results.csv", index=False)
print(f"\nSaved rt_analyses_moderation_results.csv ({len(mod_rows)} rows; both anxieties)")

# Per-anxiety verdict summary
for anx_col, res_list in all_mod_results.items():
    n_supported = sum(1 for r in res_list if r['verdict'].startswith('H1 SUPPORTED'))
    n_marginal  = sum(1 for r in res_list if r['verdict'].startswith('MARGINAL'))
    n_disconf   = sum(1 for r in res_list if r['verdict'].startswith('H1 DISCONFIRMED'))
    print(f"\n=== {anx_col} robustness summary across {len(res_list)} OT flavours ===")
    print(f"  H1 SUPPORTED:                {n_supported}")
    print(f"  MARGINAL:                    {n_marginal}")
    print(f"  H1 DISCONFIRMED:             {n_disconf}")
    print(f"  H1 NOT SUPPORTED:            {len(res_list) - n_supported - n_marginal - n_disconf}")

# ── Plot: OT vs anx_col, separate regression per MB group ──
BG_COLOR = "#FAFBFC"
MB_COLORS = {"non-MB": "#3498DB", "MB": "#2ECC71"}

for anx_col in ['somatic_anxiety','worry']:
    res_list = all_mod_results[anx_col]
    fig, axes = plt.subplots(1, len(OT_FLAVOURS), figsize=(6*len(OT_FLAVOURS), 5),
                              facecolor=BG_COLOR, squeeze=False)
    for ax, (ot_col, label), res in zip(axes.flat, OT_FLAVOURS, res_list):
        ax.set_facecolor(BG_COLOR)
        g = rt_per_subj.dropna(subset=[anx_col, ot_col, 'mb_group'])
        for grp, color in MB_COLORS.items():
            gg = g[g['mb_group'] == grp]
            if len(gg) < 3: continue
            ax.scatter(gg[anx_col], gg[ot_col], alpha=.6, s=32,
                       color=color, edgecolors='white', linewidths=0.4,
                       label=f"{grp} (n={len(gg)})")
            m_g, b_g = np.polyfit(gg[anx_col], gg[ot_col], 1)
            xr = np.linspace(gg[anx_col].min(), gg[anx_col].max(), 50)
            ax.plot(xr, m_g*xr + b_g, color=color, lw=2)
        ax.axhline(0, color='k', lw=.4)
        title = (f"{label}\n"
                 f"interaction beta={res['b_int']:+.3f}, "
                 f"one-sided p={res['p_int_1']:.3g}\n"
                 f"MB slope={res['b_mb']:+.3f} (p={res['p_mb_1']:.3g}), "
                 f"non-MB={res['b_non']:+.3f}\n"
                 f"→ {res['verdict']}")
        ax.set_xlabel(anx_col); ax.set_ylabel(ot_col)
        ax.set_title(title, fontweight='bold', fontsize=9)
        ax.legend(fontsize=8); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(f"Linear-OLS moderation (secondary) — OT ~ {anx_col} × MB-group",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = f"fig_section2_moderation_{anx_col}.png"
    plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print(f"\nSaved {out}")

---
## Section 2.7 — Rank-based moderation regression (primary)

The assumption-check cell (1A-bis) showed that anxiety is non-normal
in every subgroup, with several outliers and several |ρ−r| > .05
non-linear-monotonic patterns. Pearson's assumptions are not met for
*any* of the 72 anxiety × metric tests. The OLS moderation in
Section 2.6 is therefore re-run on ranks.

**Approach** (rank regression, the parametric analog of Spearman):

1. Rank-transform `worry` and `OT` (separately, with average ties)
   across all subjects in the relevant analysis.
2. Centre ranked worry.
3. Fit OLS: `rank_OT ~ rank_worry * mb_dummy`.
4. Read off the same interaction coefficient + simple-slope estimates;
   apply the preregistered one-sided decision rule.

The interaction coefficient now tests whether the *rank-rank slope*
of OT on worry differs by MB group — directly the rank-based analog
of the preregistered moderation test.

We also report the **Aligned-Rank Transform (ART)** F-test on the
interaction as a sensitivity check, since rank-then-OLS can mis-handle
interactions in the presence of strong main effects (Wobbrock et al.
2011). ART aligns the data by removing main effects before ranking,
giving a properly nonparametric F on the interaction.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2.7  RANK-BASED MODERATION REGRESSION (primary)
#       + Aligned-Rank Transform (ART) sensitivity check on interaction
# ══════════════════════════════════════════════════════════════════════
import statsmodels.formula.api as smf
from scipy.stats import t as t_dist, f as f_dist
from scipy.stats import rankdata

def _rank(s):
    return pd.Series(rankdata(s, method='average', nan_policy='omit'), index=s.index)

def _onesided_t(t_stat, rt_df, predicted_pos=True):
    '''One-sided p-value for the predicted direction.
       OT signed so positive slope = more overthinking.'''
    if predicted_pos:
        return float(t_dist.sf(t_stat, rt_df))    # P(T > t)
    return float(t_dist.cdf(t_stat, rt_df))       # P(T < t)

def _star(p):
    return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''

def fit_rank_moderation(df_in, ot_col, label, anx_col):
    g = df_in.dropna(subset=[anx_col, ot_col, 'mb_group']).copy()
    g['mb_dummy']      = (g['mb_group'] == 'MB').astype(int)
    g['rank_worry']    = _rank(g[anx_col]).values
    g['rank_OT']       = _rank(g[ot_col]).values
    g['rank_worry_c']  = g['rank_worry'] - g['rank_worry'].mean()

    m = smf.ols('rank_OT ~ rank_worry_c * mb_dummy', data=g).fit()
    df_resid = int(m.df_resid)

    b_int  = m.params['rank_worry_c:mb_dummy']
    se_int = m.bse['rank_worry_c:mb_dummy']
    t_int  = m.tvalues['rank_worry_c:mb_dummy']
    p_int_2= m.pvalues['rank_worry_c:mb_dummy']
    p_int_1= _onesided_t(t_int, df_resid, predicted_pos=True)

    b_non  = m.params['rank_worry_c']; se_non = m.bse['rank_worry_c']
    t_non  = m.tvalues['rank_worry_c']
    p_non_2= m.pvalues['rank_worry_c']
    p_non_1= _onesided_t(t_non, df_resid, predicted_pos=True)

    b_mb = b_non + b_int
    cov  = m.cov_params()
    se_mb = float(np.sqrt(cov.loc['rank_worry_c','rank_worry_c']
                           + cov.loc['rank_worry_c:mb_dummy','rank_worry_c:mb_dummy']
                           + 2*cov.loc['rank_worry_c','rank_worry_c:mb_dummy']))
    t_mb  = b_mb / se_mb
    p_mb_2= 2 * (1 - t_dist.cdf(abs(t_mb), df_resid))
    p_mb_1= _onesided_t(t_mb, df_resid, predicted_pos=True)

    # ─── Aligned-Rank Transform on the interaction ───
    # Steps: (1) remove main effects → align;  (2) rank aligned;
    #        (3) F-test on interaction in OLS on aligned ranks.
    g['worry_c'] = g[anx_col] - g[anx_col].mean()
    main_only = smf.ols('rank_OT ~ worry_c + mb_dummy', data=g).fit()
    g['aligned_int'] = g['rank_OT'] - main_only.fittedvalues
    g['rank_aligned'] = _rank(g['aligned_int']).values
    art = smf.ols('rank_aligned ~ worry_c * mb_dummy', data=g).fit()
    art_F  = float(art.f_test('worry_c:mb_dummy = 0').fvalue)
    art_df = (1, int(art.df_resid))
    art_p  = float(1 - f_dist.cdf(art_F, *art_df))   # two-sided F-test
    # One-sided ART p in the predicted (positive) direction
    art_t  = art.tvalues['worry_c:mb_dummy']
    art_p_1 = _onesided_t(art_t, int(art.df_resid), predicted_pos=True)

    print(f"\n--- {label}  ({ot_col}) ---")
    print(f"  N = {len(g)}  (MB={int((g['mb_dummy']==1).sum())}, non-MB={int((g['mb_dummy']==0).sum())})")
    print(f"  RANK-OLS interaction:  beta={b_int:+.4f} SE={se_int:.4f}  "
          f"t({df_resid})={t_int:+.2f}  one-sided p={p_int_1:.4g} {_star(p_int_1)}  "
          f"(two-sided p={p_int_2:.4g})")
    print(f"  Simple slope, non-MB:  beta={b_non:+.4f}  one-sided p={p_non_1:.4g} {_star(p_non_1)}")
    print(f"  Simple slope, MB:      beta={b_mb:+.4f} SE={se_mb:.4f}  "
          f"one-sided p={p_mb_1:.4g} {_star(p_mb_1)}  (two-sided p={p_mb_2:.4g})")
    print(f"  ART sensitivity (interaction): F({art_df[0]},{art_df[1]})={art_F:.2f},  "
          f"two-sided p={art_p:.4g},  one-sided p={art_p_1:.4g} {_star(art_p_1)}")

    # OT signed so POSITIVE slope = more overthinking
    int_ok = (p_int_1 < 0.05) and (b_int > 0)
    mb_ok  = (p_mb_1  < 0.05) and (b_mb  > 0)
    wrong_dir = (b_int < 0) and (p_int_2 < 0.05)
    if int_ok and mb_ok:
        verdict = "H1 SUPPORTED"
    elif int_ok and not mb_ok:
        verdict = "MARGINAL (interaction OK, MB slope p>.05)"
    elif wrong_dir:
        verdict = "H1 DISCONFIRMED (significant in WRONG direction)"
    else:
        verdict = "H1 NOT SUPPORTED (interaction p>.05)"
    print(f"  → DECISION: {verdict}")

    return dict(label=label, ot=ot_col, n=len(g),
                b_int=b_int, se_int=se_int, p_int_1=p_int_1, p_int_2=p_int_2,
                b_non=b_non, p_non_1=p_non_1,
                b_mb=b_mb, se_mb=se_mb, p_mb_1=p_mb_1, p_mb_2=p_mb_2,
                art_F=art_F, art_p_2=art_p, art_p_1=art_p_1,
                verdict=verdict, model=m)

print("=" * 78)
print("RANK-BASED MODERATION (primary, given assumption violations)")
print("=" * 78)

all_rank_results = {}
for anx_col in ['somatic_anxiety','worry']:
    print(f"\n\n========================  {anx_col.upper()}  ========================")
    res_list = [fit_rank_moderation(rt_per_subj, ot, lab, anx_col) for ot, lab in OT_FLAVOURS]
    all_rank_results[anx_col] = res_list

rank_rows = []
for anx_col, res_list in all_rank_results.items():
    for r in res_list:
        d = {k: v for k, v in r.items() if k != 'model'}
        d['anx'] = anx_col
        rank_rows.append(d)
pd.DataFrame(rank_rows).to_csv("rt_analyses_rank_moderation_results.csv", index=False)
print(f"\nSaved rt_analyses_rank_moderation_results.csv ({len(rank_rows)} rows; both anxieties)")

for anx_col, res_list in all_rank_results.items():
    n_supported = sum(1 for r in res_list if r['verdict'].startswith('H1 SUPPORTED'))
    n_marginal  = sum(1 for r in res_list if r['verdict'].startswith('MARGINAL'))
    n_disconf   = sum(1 for r in res_list if r['verdict'].startswith('H1 DISCONFIRMED'))
    print(f"\n=== {anx_col} rank-based robustness summary across {len(res_list)} OT flavours ===")
    print(f"  H1 SUPPORTED:       {n_supported}")
    print(f"  MARGINAL:           {n_marginal}")
    print(f"  H1 DISCONFIRMED:    {n_disconf}")
    print(f"  H1 NOT SUPPORTED:   {len(res_list) - n_supported - n_marginal - n_disconf}")

# ── Plot: rank(OT) vs rank(anx_col), per MB group, one panel per OT flavour ──
BG_COLOR = "#FAFBFC"
MB_COLORS = {"non-MB": "#3498DB", "MB": "#2ECC71"}

for anx_col in ['somatic_anxiety','worry']:
    res_list = all_rank_results[anx_col]
    fig, axes = plt.subplots(1, len(OT_FLAVOURS), figsize=(6*len(OT_FLAVOURS), 5),
                              facecolor=BG_COLOR, squeeze=False)
    for ax, (ot_col, label), res in zip(axes.flat, OT_FLAVOURS, res_list):
        ax.set_facecolor(BG_COLOR)
        g = rt_per_subj.dropna(subset=[anx_col, ot_col, 'mb_group']).copy()
        g['rank_anx'] = _rank(g[anx_col]).values
        g['rank_OT']    = _rank(g[ot_col]).values
        for grp, color in MB_COLORS.items():
            gg = g[g['mb_group'] == grp]
            if len(gg) < 3: continue
            ax.scatter(gg['rank_anx'], gg['rank_OT'], alpha=.6, s=32,
                       color=color, edgecolors='white', linewidths=0.4,
                       label=f"{grp} (n={len(gg)})")
            m_g, b_g = np.polyfit(gg['rank_anx'], gg['rank_OT'], 1)
            xr = np.linspace(gg['rank_anx'].min(), gg['rank_anx'].max(), 50)
            ax.plot(xr, m_g*xr + b_g, color=color, lw=2)
        title = (f"{label}\n"
                 f"rank-OLS interaction beta={res['b_int']:+.3f}, "
                 f"one-sided p={res['p_int_1']:.3g}\n"
                 f"MB rank-slope={res['b_mb']:+.3f} (p={res['p_mb_1']:.3g}), "
                 f"non-MB={res['b_non']:+.3f}\n"
                 f"ART one-sided p={res['art_p_1']:.3g}  ->  {res['verdict']}")
        ax.set_xlabel(f"rank({anx_col})")
        ax.set_ylabel(f"rank({ot_col})")
        ax.set_title(title, fontweight='bold', fontsize=9)
        ax.legend(fontsize=8); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(f"Rank-based moderation regression — rank(OT) ~ rank({anx_col}) × MB-group",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    out = f"fig_section2_rank_moderation_{anx_col}.png"
    plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print(f"\nSaved {out}")

In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS: Robustness of OT ~ Anxiety to HBI assignment uncertainty
# ══════════════════════════════════════════════════════════════════════
# HBI responsibility assignments can shift when the candidate model set changes,
# and borderline subjects (low max responsibility) are especially unstable.
# This cell tests whether the OT~anxiety finding (especially for MB) survives
# reassignment of uncertain subjects. Three scenarios:
#   A. Decisive-only   — drop subjects with max(resp) < TAU
#   B. Second-best flip — reassign borderline subjects to 2nd-best model
#   C. Monte Carlo     — sample assignments from responsibility distribution

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

TAU    = 0.80   # confidence threshold for "decisive"
N_BOOT = 1000   # Monte Carlo draws
SEED   = 42

GROUPS_ALL = MODEL_GROUPS + ["__pool__"]

def _label(mk):
    return "pooled (MF+MB)" if mk == "__pool__" else mk

def run_ot_from_map(model_map, subj_base=subj):
    """Given {subject_id_str: model_name}, rerun OT = z(acc) - z(RT) vs anxiety per group."""
    s = subj_base.copy()
    s["model"] = s["subject_id"].astype(str).map(model_map)
    out = {}
    for mk in GROUPS_ALL:
        g = (s[s["model"].isin(MODEL_GROUPS)] if mk == "__pool__"
             else s[s["model"] == mk]).dropna(subset=["somatic_anxiety"]).copy()
        if len(g) < 10:
            out[mk] = (np.nan, np.nan, len(g))
            continue
        z_rt  = (g["mean_log_delib"] - g["mean_log_delib"].mean()) / g["mean_log_delib"].std()
        z_acc = (g["accuracy"]       - g["accuracy"].mean())       / g["accuracy"].std()
        ot = z_acc - z_rt
        rho, p = spearmanr(g["somatic_anxiety"], ot)
        out[mk] = (rho, p, len(g))
    return out

# ── Subject order must match responsibility row order ──
sid_keys = list(winning_params.keys())
sid_str  = [str(s) for s in sid_keys]
max_resp = responsibility.max(axis=1)
argmax_k = responsibility.argmax(axis=1)
second_k = np.argsort(responsibility, axis=1)[:, -2]

# ── Baseline (argmax) ──
baseline_map = {sid_str[i]: GROUP_MAP.get(model_names_choice[argmax_k[i]], "Other")
                for i in range(len(sid_str))}
base = run_ot_from_map(baseline_map)

def _print_block(title, res):
    print("\n" + "="*72)
    print(title)
    print("="*72)
    for mk in GROUPS_ALL:
        r, p, n = res[mk]
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        print(f"  {_label(mk):25s}  rho={r:+.3f}  p={p:.4f} {star:<3s}  n={n}")

_print_block("BASELINE  (argmax responsibility)", base)

# ── A. Decisive-only ──
decisive_mask = max_resp >= TAU
n_drop = int((~decisive_mask).sum())
decisive_map = {sid_str[i]: baseline_map[sid_str[i]]
                for i in range(len(sid_str)) if decisive_mask[i]}
dec = run_ot_from_map(decisive_map)
_print_block(f"A. DECISIVE-ONLY  (max resp >= {TAU}; drop {n_drop}/{len(sid_str)})", dec)

# ── B. Second-best flip ──
flip_map = dict(baseline_map)
for i in range(len(sid_str)):
    if not decisive_mask[i]:
        flip_map[sid_str[i]] = GROUP_MAP.get(model_names_choice[second_k[i]], "Other")
flip = run_ot_from_map(flip_map)
_print_block(f"B. SECOND-BEST FLIP  (borderline subjects swapped to 2nd-best; n_flipped={n_drop})", flip)

# ── C. Monte Carlo ──
rng = np.random.default_rng(SEED)
K = responsibility.shape[1]
mc_rho = {mk: np.full(N_BOOT, np.nan) for mk in GROUPS_ALL}
mc_p   = {mk: np.full(N_BOOT, np.nan) for mk in GROUPS_ALL}
mc_n   = {mk: np.zeros(N_BOOT, dtype=int) for mk in GROUPS_ALL}

for b in range(N_BOOT):
    draws = np.array([rng.choice(K, p=responsibility[i]) for i in range(len(sid_str))])
    mc_map = {sid_str[i]: GROUP_MAP.get(model_names_choice[draws[i]], "Other")
              for i in range(len(sid_str))}
    for mk, (r, p, n) in run_ot_from_map(mc_map).items():
        mc_rho[mk][b] = r
        mc_p[mk][b]   = p
        mc_n[mk][b]   = n

print("\n" + "="*72)
print(f"C. MONTE CARLO  ({N_BOOT} draws from responsibility distribution)")
print("="*72)
print(f"  {'group':25s}  {'rho median':>11s}  {'95% CI':>22s}  {'Pr(p<.05)':>10s}  {'median n':>9s}")
for mk in GROUPS_ALL:
    r = mc_rho[mk][~np.isnan(mc_rho[mk])]
    p = mc_p[mk][~np.isnan(mc_p[mk])]
    n_med = int(np.median(mc_n[mk])) if len(mc_n[mk]) else 0
    if len(r) < 10:
        print(f"  {_label(mk):25s}  (insufficient data, median n={n_med})")
        continue
    ci_lo, ci_hi = np.percentile(r, [2.5, 97.5])
    print(f"  {_label(mk):25s}  {np.median(r):+11.3f}  [{ci_lo:+.3f}, {ci_hi:+.3f}]  {(p < .05).mean():>10.3f}  {n_med:>9d}")

# ── Compact per-scenario summary table ──
print("\n" + "="*72)
print("SUMMARY — rho (p) across scenarios")
print("="*72)
hdr = f"  {'group':25s}  {'baseline':>14s}  {'decisive':>14s}  {'2nd-best':>14s}  {'MC median [CI]':>28s}"
print(hdr)
for mk in GROUPS_ALL:
    def fmt(x):
        r, p, _ = x
        if np.isnan(r): return f"{'--':>14s}"
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        return f"{r:+.3f} ({p:.3f}){star}"
    r_mc = mc_rho[mk][~np.isnan(mc_rho[mk])]
    mc_str = "--" if len(r_mc) < 10 else f"{np.median(r_mc):+.3f} [{np.percentile(r_mc,2.5):+.2f},{np.percentile(r_mc,97.5):+.2f}]"
    print(f"  {_label(mk):25s}  {fmt(base[mk]):>14s}  {fmt(dec[mk]):>14s}  {fmt(flip[mk]):>14s}  {mc_str:>28s}")

# ── Visualise MC distribution of rho ──
BG_COLOR = "#FAFBFC"
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# Left: boxplot of MC rho per group, with baseline overlaid
ax = axL; ax.set_facecolor(BG_COLOR)
vals, positions, colors = [], [], []
MIN_MC = 10
for i, mk in enumerate(GROUPS_ALL):
    r = mc_rho[mk][~np.isnan(mc_rho[mk])]
    if len(r) < MIN_MC: continue
    vals.append(r); positions.append(i)
    colors.append(MODEL_COLORS.get(mk, "#34495E") if mk != "__pool__" else "#34495E")

if len(vals) == 0:
    # No group has enough non-NaN MC samples to plot. Common at very small N — skip silently.
    ax.text(0.5, 0.5,
            f"insufficient MC samples\n(need >= {MIN_MC} non-NaN per group)",
            transform=ax.transAxes, ha="center", va="center",
            color="#888", fontsize=11)
else:
    bp = ax.boxplot(vals, positions=positions, widths=0.55,
                    patch_artist=True, showfliers=False)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col); patch.set_alpha(0.55)
for i, mk in enumerate(GROUPS_ALL):
    r = base[mk][0]
    if not np.isnan(r):
        ax.plot(i, r, "kD", markersize=9, zorder=5,
                label="argmax baseline" if i == 0 else None)

ax.axhline(0, color="gray", ls="--", alpha=0.5)
ax.set_xticks(range(len(GROUPS_ALL)))
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_GROUPS] + ["Pool"], fontsize=11)
ax.set_ylabel("Spearman rho  (OT ~ anxiety)", fontsize=11)
ax.set_title(f"MC distribution of rho\n({N_BOOT} draws from HBI responsibility)",
             fontweight="bold", fontsize=11)
ax.legend(loc="best", fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Right: Pr(p<.05) across scenarios
ax = axR; ax.set_facecolor(BG_COLOR)
xs = np.arange(len(GROUPS_ALL))
w = 0.22
scen_labels = ["baseline", "decisive", "2nd-best", "MC mean"]
for j, (lbl, src) in enumerate(zip(scen_labels, [base, dec, flip, None])):
    ys = []
    for mk in GROUPS_ALL:
        if lbl == "MC mean":
            p = mc_p[mk][~np.isnan(mc_p[mk])]
            ys.append((p < .05).mean() if len(p) >= 10 else np.nan)
        else:
            _, pv, _ = src[mk]
            ys.append(1.0 if pv < .05 else 0.0)
    ax.bar(xs + (j - 1.5) * w, ys, w, label=lbl, alpha=0.85, edgecolor="white")

ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.set_xticks(xs)
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_GROUPS] + ["Pool"], fontsize=11)
ax.set_ylabel("Pr(p < .05)  /  indicator", fontsize=11)
ax.set_title("Significance robustness across scenarios", fontweight="bold", fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc="upper right")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Sensitivity of OT ~ Anxiety finding to HBI assignment uncertainty",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_sensitivity_ot_anxiety_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1C.  CATEGORICAL: median split on each anxiety scale → Mann-Whitney + Welch t
# ══════════════════════════════════════════════════════════════════════
def _split_table(rt_df, metric_cols, anx_bin_col, group_col='mb_group'):
    rows = []
    for col in metric_cols:
        for grp in [None, 'non-MB', 'MB']:
            g = rt_df if grp is None else rt_df[rt_df[group_col] == grp]
            g = g.dropna(subset=[col, anx_bin_col])
            grp_label = 'All' if grp is None else grp
            lo = g.loc[g[anx_bin_col]=='Low',  col]
            hi = g.loc[g[anx_bin_col]=='High', col]
            if len(lo) < 3 or len(hi) < 3: continue
            t, pt = ttest_ind(hi, lo, equal_var=False)
            u, pu = mannwhitneyu(hi, lo, alternative='two-sided')
            rows.append({'metric': col, 'group': grp_label,
                         'n_low': len(lo), 'n_high': len(hi),
                         'M_low': lo.mean(), 'M_high': hi.mean(),
                         't': t, 'p_t': pt, 'U': u, 'p_U': pu})
    return pd.DataFrame(rows)

for anx_col in ['somatic_anxiety','worry']:
    print(f"\n\n=========== Median-split on {anx_col} ===========")
    med_anx = rt_per_subj[anx_col].median()
    bin_col = f'{anx_col}_bin'
    rt_per_subj[bin_col] = np.where(rt_per_subj[anx_col] >= med_anx, 'High', 'Low')
    print(f"  median = {med_anx:.2f};  "
          f"Low n={(rt_per_subj[bin_col]=='Low').sum()}, "
          f"High n={(rt_per_subj[bin_col]=='High').sum()}")
    print(f"\n  Delib-RT median-split tests:")
    print(_split_table(rt_per_subj, RT_COLS, bin_col).round(3).to_string(index=False))
    print(f"\n  Canonical-accuracy median-split tests:")
    print(_split_table(rt_per_subj, ACC_COLS, bin_col).round(3).to_string(index=False))
    print(f"\n  OT median-split tests:")
    print(_split_table(rt_per_subj, OT_COLS, bin_col).round(3).to_string(index=False))

### 5.2 — Each anxiety controlling for the other (items 1.1 & 2)

Section 1B reported plain correlations; here both scales are entered **together** so each
coefficient is the *unique* (partial) effect on overthinking once the shared anxiety variance
is removed. Rank regression `rank(OT) ~ rank(somatic) + rank(worry)`, run for all / MB / non-MB.
One-sided p for the predicted positive (more-overthinking) direction. This directly answers
"does somatic still predict OT controlling for worry?" and the reverse.

In [ ]:
# ═══ 5.2  Partial (rank) effects on OT, each anxiety controlling for the other ═══
from scipy.stats import rankdata, t as _tdist
def _rank(s): return pd.Series(rankdata(s), index=s.index)
def _os(tv, dfr, pos=True): return float(_tdist.sf(tv,dfr)) if pos else float(_tdist.cdf(tv,dfr))
print("Partial rank effects on OT_all  (one-sided p, predicted +)\n")
print(f"{'group':8s}{'n':>5s}    somatic | worry              worry | somatic")
_rows52=[]
for grp in ['all','MB','non-MB']:
    sub = rt_per_subj if grp=='all' else rt_per_subj[rt_per_subj['mb_group']==grp]
    g = sub.dropna(subset=['somatic_anxiety','worry','OT_all']).copy()
    g['rOT']=_rank(g['OT_all']).values
    g['rS'] =_rank(g['somatic_anxiety']).values
    g['rW'] =_rank(g['worry']).values
    m = smf.ols('rOT ~ rS + rW', g).fit(); dfr=int(m.df_resid)
    bS,pS = m.params['rS'], _os(m.tvalues['rS'],dfr)
    bW,pW = m.params['rW'], _os(m.tvalues['rW'],dfr)
    print(f"{grp:8s}{len(g):>5d}    beta={bS:+.3f}  1s p={pS:.3f}        beta={bW:+.3f}  1s p={pW:.3f}")
    _rows52.append(dict(group=grp,n=len(g),beta_somatic=bS,p_somatic_1s=pS,
                        beta_worry=bW,p_worry_1s=pW))
pd.DataFrame(_rows52).to_csv("rt_analyses_partial_OT_control.csv",index=False)
print("\nReading: within each group, the predictor that stays significant is the one")
print("carrying overthinking once the shared anxiety variance is partialled out.")
print("Saved rt_analyses_partial_OT_control.csv")

### 5.4 — Robustness: non-log median deliberation RT (item 1.2)

The primary deliberation measure is the per-subject **mean of per-trial log** RT. Rebuild OT
using each subject's **median raw (non-log) motor-corrected** deliberation time instead, and
rerun the primary rank moderation + MB simple slope. If the MB simple slope stays significant,
the overthinking effect is not an artifact of the log transform.

In [ ]:
# ═══ 5.4  Non-log MEDIAN RT robustness ═════════════════════════════════════
from scipy.stats import rankdata, t as _tdist
def _rank(s): return pd.Series(rankdata(s), index=s.index)
def _os(tv,dfr,pos=True): return float(_tdist.sf(tv,dfr)) if pos else float(_tdist.cdf(tv,dfr))
def _zc(s): s=s.astype(float); return (s-s.mean())/s.std()
def _logit(p,eps=.025): p=np.clip(p,eps,1-eps); return np.log(p/(1-p))

# per-subject median motor-corrected deliberation (seconds), non-log
_med = rt_trial.groupby('subject_id')['delib_s'].median().rename('delib_med_s').reset_index()
ps5  = rt_per_subj.merge(_med, on='subject_id', how='left')
ps5['OT_med'] = _zc(ps5['delib_med_s']) - _zc(_logit(ps5['acc_canon_all']))

def _moder(col, anx, frame):
    g = frame.dropna(subset=[anx,col,'mb_group']).copy()
    g['mb']=(g['mb_group']=='MB').astype(int)
    g['rO']=_rank(g[col]).values; g['rA']=_rank(g[anx]).values; g['rAc']=g['rA']-g['rA'].mean()
    m=smf.ols('rO ~ rAc*mb', g).fit(); dfr=int(m.df_resid)
    pi = _os(m.tvalues['rAc:mb'], dfr)
    cov=m.cov_params(); bmb=m.params['rAc']+m.params['rAc:mb']
    se=np.sqrt(cov.loc['rAc','rAc']+cov.loc['rAc:mb','rAc:mb']+2*cov.loc['rAc','rAc:mb'])
    return pi, _os(bmb/se, dfr)

print("Primary rank moderation + MB simple slope:  LOG-OT  vs  MEDIAN-RT-OT\n")
for anx in ['somatic_anxiety','worry']:
    pil,pml = _moder('OT_all', anx, ps5)
    pim,pmm = _moder('OT_med', anx, ps5)
    print(f"  {anx:16s} LOG   : interaction 1s p={pil:.3f},  MB slope 1s p={pml:.3f}")
    print(f"  {'':16s} MEDIAN: interaction 1s p={pim:.3f},  MB slope 1s p={pmm:.3f}")
ps5[['subject_id','delib_med_s','OT_med']].to_csv("rt_analyses_OT_median_robustness.csv",index=False)
print("\nSaved rt_analyses_OT_median_robustness.csv")

### 5.6 — Simple: worry × {choose-MB, log deliberation time} across three samples

For **worry**, two correlations:
* **choose-MB** = `mb_score` = P(chose MB option | anti-MF trial)  × worry
* **log deliberation time** = `delib_log_all`  × worry

computed in three samples: (1) the **full sample**, (2) **model-agnostic MB learners**
(`mb_group == 'MB'`), and (3) **CBM model-fit MB learners** (`cbm_model == 'MB'`, the
HBI/computational separation from `Modelling_AllChoices.ipynb`). Spearman ρ throughout.

*Note: within an MB subgroup the `choose-MB` range is restricted (everyone is above chance),
so its correlation is on a truncated range — interpret accordingly.*

In [ ]:
# ═══ 5.6  worry × {choose-MB, log-delib-time}  in 3 samples ════════════════
from scipy.stats import spearmanr
_ps6 = rt_per_subj.copy()
# Attach the CBM (model-fit) separation if it was loaded in the CBM-overlap cell
if 'cbm_labels' in globals() and cbm_labels is not None:
    _ps6 = _ps6.merge(cbm_labels[['subject_id','cbm_model']], on='subject_id', how='left')
    _has_cbm = True
else:
    _ps6['cbm_model'] = np.nan; _has_cbm = False
    print("NOTE: cbm_labels not available in this kernel — CBM row skipped.\n")

_samples = [('full sample',          _ps6),
            ('model-agnostic MB',     _ps6[_ps6['mb_group']=='MB'])]
if _has_cbm:
    _samples.append(('CBM model-fit MB', _ps6[_ps6['cbm_model']=='MB']))

print(f"{'sample':20s}{'n':>5s}    choose-MB x worry        log-delibRT x worry")
_rows6=[]
for lab, d in _samples:
    g1 = d.dropna(subset=['mb_score','worry'])
    g2 = d.dropna(subset=['delib_log_all','worry'])
    r1,p1 = spearmanr(g1['mb_score'],      g1['worry']) if len(g1)>=4 else (np.nan,np.nan)
    r2,p2 = spearmanr(g2['delib_log_all'], g2['worry']) if len(g2)>=4 else (np.nan,np.nan)
    print(f"{lab:20s}{len(d):>5d}    rho={r1:+.3f} p={p1:.3f}       rho={r2:+.3f} p={p2:.3f}")
    _rows6.append(dict(sample=lab, n=len(d),
                       rho_chooseMB_worry=r1, p_chooseMB_worry=p1,
                       rho_logdelib_worry=r2, p_logdelib_worry=p2))
pd.DataFrame(_rows6).to_csv("rt_analyses_worry_chooseMB_delib.csv", index=False)
print("\nSaved rt_analyses_worry_chooseMB_delib.csv")

---
### 5.7 — Worry / somatic / reappraisal vs the two OT halves, across separations

`OT_i = z(delib_log_i) − z(logit(acc_canon_i))`. Here we correlate each predictor with the
**two halves separately**:
* **accuracy half** = `z_acc_canon_all` = `z(logit(acc_canon))`
* **deliberation half** = `z_delib_log_all` = `z(delib_log)`

in five samples: **full sample**, model-agnostic **MB** and **non-MB** (`mb_group`), and CBM
model-fit **MB** and **MF** (`cbm_model`). Spearman ρ, Pearson r, and p-values are reported,
with a 2×5 scatter grid per predictor (top row = accuracy half, bottom row = deliberation half).

The setup cell defines a reusable helper; the three cells after it run worry, somatic anxiety,
and reappraisal. **Reappraisal** = ERQ cognitive-reappraisal subscale (items 1,3,5,7,8,10;
Gross & John 2003) — edit `_REAPP_ITEMS` if this survey's item order differs.

In [ ]:
# ═══ 5.7  SETUP — build frame (+ reappraisal, + CBM split) and helper ══════════
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# Reappraisal = ERQ cognitive-reappraisal subscale
_REAPP_ITEMS = [1,3,5,7,8,10]
_rcols = [f'Survey_ERQ_Q{q}_Raw' for q in _REAPP_ITEMS]
_reapp = rt_df[['subject_id']].copy()
_reapp['reappraisal'] = rt_df[_rcols].sum(axis=1, skipna=False)

ps_corr = rt_per_subj.merge(_reapp, on='subject_id', how='left')
if 'cbm_labels' in globals() and cbm_labels is not None:
    ps_corr = ps_corr.merge(cbm_labels[['subject_id','cbm_model']], on='subject_id', how='left')
else:
    ps_corr['cbm_model'] = np.nan
    print("NOTE: cbm_labels unavailable in this kernel — CBM panels will be empty.")
print(f"ps_corr: {len(ps_corr)} subjects; reappraisal n={ps_corr['reappraisal'].notna().sum()}; "
      f"cbm labelled n={ps_corr['cbm_model'].notna().sum()}")

# OT halves (pool-z-scored in §1A): accuracy half and deliberation-time half
_TARGETS = [('z_acc_canon_all', 'accuracy half  z(logit acc_canon)'),
            ('z_delib_log_all', 'deliberation half  z(delib_log)')]

def _groups(frame):
    return [('full sample', frame),
            ('MA: MB',      frame[frame['mb_group']=='MB']),
            ('MA: non-MB',  frame[frame['mb_group']=='non-MB']),
            ('CBM: MB',     frame[frame['cbm_model']=='MB']),
            ('CBM: MF',     frame[frame['cbm_model']=='MF'])]

def _pack(x, y):
    g = pd.DataFrame({'x':x,'y':y}).dropna()
    if len(g) < 4: return dict(n=len(g), sr=np.nan, sp=np.nan, pr=np.nan, pp=np.nan,
                               x=g['x'].values, y=g['y'].values)
    sr,sp = spearmanr(g['x'],g['y']); pr,pp = pearsonr(g['x'],g['y'])
    return dict(n=len(g), sr=sr, sp=sp, pr=pr, pp=pp, x=g['x'].values, y=g['y'].values)

def corr_OT_halves(pred, pretty, frame=None):
    """Print Spearman+Pearson+p for `pred` vs each OT half across the 5 groups,
    save a tidy CSV, and draw a 2x5 scatter grid."""
    frame = ps_corr if frame is None else frame
    grps = _groups(frame); rows=[]
    print(f"\n================  {pretty}  vs OT halves  ================")
    for tcol,tlab in _TARGETS:
        print(f"  -- vs {tlab} --")
        for glab,gd in grps:
            d=_pack(gd[pred], gd[tcol])
            print(f"     {glab:12s} n={d['n']:3d}   Spearman rho={d['sr']:+.3f} (p={d['sp']:.3f})"
                  f"   Pearson r={d['pr']:+.3f} (p={d['pp']:.3f})")
            rows.append(dict(predictor=pred, target=tcol, group=glab, n=d['n'],
                             spearman_rho=d['sr'], spearman_p=d['sp'],
                             pearson_r=d['pr'], pearson_p=d['pp']))
    pd.DataFrame(rows).to_csv(f"rt_analyses_corr_{pred}_OThalves.csv", index=False)
    # visual: 2 rows (halves) x 5 cols (groups)
    BG="#FAFBFC"; ncol=len(grps)
    fig,axes=plt.subplots(2,ncol,figsize=(3.4*ncol,7.4),facecolor=BG)
    for ri,(tcol,tlab) in enumerate(_TARGETS):
        for ci,(glab,gd) in enumerate(grps):
            ax=axes[ri,ci]; ax.set_facecolor(BG)
            d=_pack(gd[pred], gd[tcol])
            if d['n']>=4 and len(np.unique(d['x']))>1:
                ax.scatter(d['x'],d['y'],s=22,alpha=.55,color='#2C7FB8',
                           edgecolors='white',linewidths=.4)
                m,b=np.polyfit(d['x'],d['y'],1)
                xr=np.linspace(np.min(d['x']),np.max(d['x']),30)
                ax.plot(xr,m*xr+b,color='#08519c',lw=1.8)
                ax.set_title(f"{glab}\nS rho={d['sr']:+.2f} p={d['sp']:.3f}\n"
                             f"P r={d['pr']:+.2f} p={d['pp']:.3f}  n={d['n']}",
                             fontsize=8,fontweight='bold')
            else:
                ax.set_title(f"{glab}\n(n={d['n']}, n/a)",fontsize=8)
            if ci==0: ax.set_ylabel(tlab,fontsize=8)
            ax.set_xlabel(pretty,fontsize=8)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            ax.tick_params(labelsize=7)
    fig.suptitle(f"{pretty}  x  OT halves   (top: accuracy half,  bottom: deliberation half)",
                 fontsize=12,fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig(f"fig_section5_corr_{pred}_OThalves.png",dpi=170,bbox_inches='tight',facecolor=BG)
    plt.show()
    return pd.DataFrame(rows)

print("helper ready: corr_OT_halves(pred, pretty)")

In [ ]:
# ═══ 5.7a  WORRY  vs OT halves ═════════════════════════════════════════════
_ = corr_OT_halves('worry', 'Worry (PSWQ-A)')

In [ ]:
# ═══ 5.7b  SOMATIC ANXIETY  vs OT halves ═══════════════════════════════════
_ = corr_OT_halves('somatic_anxiety', 'Somatic anxiety (MASQ-AA)')

In [ ]:
# ═══ 5.7c  REAPPRAISAL  vs OT halves ═══════════════════════════════════════
_ = corr_OT_halves('reappraisal', 'Reappraisal (ERQ)')